# 4.6.2 Analyze Forecast Errors and Feature Drivers

Notebook 4.6.1 has already selected and frozen one forecasting champion for each of the **15 Target × Horizon jobs** and evaluated that system on the unseen January–March 2026 holdout. This notebook does **not** choose models again.

Its job is to explain the frozen forecasting system: **where forecast error concentrates, which conditions distinguish reliable from unreliable forecasts, what the selected models rely on, and what those findings imply for later counterfactual use.**

Two questions remain separate throughout:

- **Forecast drivers:** what information the fitted champion models rely on when producing forecasts.
- **Error drivers:** the temporal, geographic, historical, and contextual conditions associated with larger or smaller forecast errors.

This first phase focuses on error drivers. It builds one auditable champion-holdout diagnostic surface and examines error across targets, horizons, time, geography, historical activity, historical volatility, and genuine zero-trip outcomes.

For **Taxi and FHVHV trip counts, zero is a real observed outcome**: no qualifying transaction occurred in that Taxi Zone × Date × Temporal Bucket. A zero-heavy history is therefore different from missing or incomplete history.


## Notebook Setup

All notebook-wide infrastructure is defined before the numbered analysis begins: imports, authoritative paths, forecasting scope, execution controls, schema contracts, shared helpers, and visualization utilities.

The large forecasting matrices remain on disk until they are needed. Pandas reads only the final-holdout rows and columns required for this analysis, after which the diagnostic work stays in familiar in-memory DataFrames.

The assembled diagnostic surface is saved to Parquet. On a clean rerun, the notebook can load that completed surface directly instead of rebuilding the large upstream joins.


In [1]:
!pip install lightgbm==4.7.0


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# ---------------------------------------------------------------------
# 4.6.2.0A — Imports
# ---------------------------------------------------------------------

import gc
import os
import uuid
from pathlib import Path

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

from project_branding import (
    BRAND_COLOR_SEQUENCE,
    BRAND_DIVERGING_SEQUENCE,
    BRAND_COLORS,
    BRAND_CLUSTER_PALETTE,
    BRAND_SEQUENTIAL_HEATMAP_SCALE,
    apply_branding,
)

# Reader-facing output stays compact; saved analytical values keep full precision.
pd.options.display.float_format = "{:,.3f}".format

import pyarrow.parquet as pq

import shutil
from time import perf_counter
from lightgbm import LGBMRegressor
from tqdm.auto import tqdm


/root/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [50]:
# ---------------------------------------------------------------------
# 4.6.2.0B — Paths, scope, toggles, and diagnostic contracts
# ---------------------------------------------------------------------

PROJECT_ROOT = Path(".")
PIPELINE_DIR = PROJECT_ROOT / "pipeline_data"

INPUT_131_FINAL_DIR = PIPELINE_DIR / "1.3.1.final_tables"
CANONICAL_CLUSTER_ASSIGNMENTS_PATH = (
    PIPELINE_DIR
    / "3.2.2.final_tables"
    / "canonical_cluster_assignments.parquet"
)
INPUT_423_FINAL_DIR = PIPELINE_DIR / "4.2.3.final_tables"
INPUT_441_FINAL_DIR = PIPELINE_DIR / "4.4.1.final_tables"
INPUT_451_FINAL_DIR = PIPELINE_DIR / "4.5.1.final_tables"
INPUT_452_FINAL_DIR = PIPELINE_DIR / "4.5.2.final_tables"
INPUT_461_FINAL_DIR = PIPELINE_DIR / "4.6.1.final_tables"

INTERMEDIATE_DIR = PIPELINE_DIR / "4.6.2.intermediate_tables"
FINAL_DIR = PIPELINE_DIR / "4.6.2.final_tables"

INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# Frozen forecasting scope
# ---------------------------------------------------------------------

FINAL_HOLDOUT_START_DATE = pd.Timestamp("2026-01-05")
FINAL_HOLDOUT_END_DATE = pd.Timestamp("2026-03-31")

FORECAST_TARGETS = [
    "taxi_trip_count",
    "taxi_avg_trip_speed",
    "fhvhv_trip_count",
    "fhvhv_avg_trip_speed",
    "subway_ridership",
]
FORECAST_HORIZONS = [1, 2, 5]
EXPECTED_JOB_COUNT = len(FORECAST_TARGETS) * len(FORECAST_HORIZONS)

TARGET_LABELS = {
    "taxi_trip_count": "Taxi trips",
    "taxi_avg_trip_speed": "Taxi average speed",
    "fhvhv_trip_count": "FHVHV trips",
    "fhvhv_avg_trip_speed": "FHVHV average speed",
    "subway_ridership": "Subway ridership",
}
TARGET_SHORT_LABELS = {
    "taxi_trip_count": "Taxi trips",
    "taxi_avg_trip_speed": "Taxi speed",
    "fhvhv_trip_count": "FHVHV trips",
    "fhvhv_avg_trip_speed": "FHVHV speed",
    "subway_ridership": "Subway",
}
TARGET_TYPES = {
    "taxi_trip_count": "Demand",
    "taxi_avg_trip_speed": "Speed",
    "fhvhv_trip_count": "Demand",
    "fhvhv_avg_trip_speed": "Speed",
    "subway_ridership": "Demand",
}
FAMILY_LABELS = {
    "classical": "Classical",
    "tree": "Tree",
    "neural": "Neural",
    "transformer": "Transformer",
}

# Taxi and FHVHV counts come directly from transactions. A zero therefore
# represents zero qualifying trips, not a missing observation.
TRANSACTION_EVENT_TARGETS = {"taxi_trip_count", "fhvhv_trip_count"}

TEMPORAL_BUCKET_ORDER = [
    "weekday_overnight",
    "weekday_am_peak",
    "weekday_midday",
    "weekday_pm_peak",
    "weekday_evening",
    "weekend_overnight",
    "weekend_am_peak",
    "weekend_midday",
    "weekend_pm_peak",
    "weekend_evening",
]
TEMPORAL_BUCKET_LABELS = {
    "weekday_overnight": "Weekday · Overnight",
    "weekday_am_peak": "Weekday · AM peak",
    "weekday_midday": "Weekday · Midday",
    "weekday_pm_peak": "Weekday · PM peak",
    "weekday_evening": "Weekday · Evening",
    "weekend_overnight": "Weekend · Overnight",
    "weekend_am_peak": "Weekend · AM peak",
    "weekend_midday": "Weekend · Midday",
    "weekend_pm_peak": "Weekend · PM peak",
    "weekend_evening": "Weekend · Evening",
}
POLICY_GEOGRAPHY_LABELS = {
    "cbd": "CBD",
    "gateway_to_cbd": "Gateway + adjacent",
    "adjacent_to_cbd": "Gateway + adjacent",
    "non_cbd": "Outside",
}

FRESHKILLS_ZONE_NAME = "Freshkills Park"

JOB_ORDER = [
    (metric, horizon)
    for metric in FORECAST_TARGETS
    for horizon in FORECAST_HORIZONS
]
JOB_LABEL_ORDER = [
    f"{TARGET_SHORT_LABELS[metric]} · h={horizon}"
    for metric, horizon in JOB_ORDER
]


# ---------------------------------------------------------------------
# Authoritative 4.6.1 evaluation handoff
# ---------------------------------------------------------------------
# 4.6.1 owns the exact storage locations of its downstream evidence. Some
# frozen artifacts legitimately remain in intermediate_tables, so 4.6.2
# resolves individual paths from the producer's manifest in Section 4.6.2.1A.

FORECAST_EVALUATION_HANDOFF_PATH = (
    INPUT_461_FINAL_DIR / "forecast_evaluation_handoff_manifest.parquet"
)


# ---------------------------------------------------------------------
# Selective context sources
# ---------------------------------------------------------------------

FORECASTING_MATRIX_PATHS = {
    "post_cp_primary":
        INPUT_423_FINAL_DIR / "forecasting_matrix__post_cp_primary.parquet",
    "full_history_regime_aware":
        INPUT_423_FINAL_DIR / "forecasting_matrix__full_history_regime_aware.parquet",
}

FEATURE_CATALOG_PATH = (
    INPUT_423_FINAL_DIR / "forecast_feature_catalog_complete.parquet"
)
TAXI_ZONE_SPATIAL_CONTEXT_PATH = (
    INPUT_131_FINAL_DIR / "nyc_taxi_zones_harmonized.parquet"
)

# ---------------------------------------------------------------------
# Existing interpretation evidence used later in 4.6.2
# ---------------------------------------------------------------------


TREE_FEATURE_IMPORTANCE_PATH = (
    INPUT_441_FINAL_DIR / "lgbm_selected_feature_importance.parquet"
)
TREE_SELECTED_MANIFEST_PATH = (
    INPUT_441_FINAL_DIR / "lgbm_selected_manifest.parquet"
)
TREE_SELECTED_MODEL_REGISTRY_PATH = (
    INPUT_441_FINAL_DIR / "lgbm_selected_model_registry.parquet"
)
TREE_PREDICTOR_REGISTRY_PATH = (
    INPUT_441_FINAL_DIR / "lgbm_predictor_registry.parquet"
)
TREE_SELECTED_VALIDATION_PREDICTIONS_PATH = (
    INPUT_441_FINAL_DIR
    / "lgbm_selected_row_level_validation_predictions.parquet"
)
TREE_FULL_AUDIT_MODEL_REGISTRY_PATH = (
    INPUT_441_FINAL_DIR / "lgbm_final_model_registry.parquet"
)
NEURAL_GROUPED_IMPORTANCE_PATH = (
    INPUT_451_FINAL_DIR / "neural_grouped_input_importance.parquet"
)
NEURAL_SENSITIVITY_SUMMARY_PATH = (
    INPUT_451_FINAL_DIR / "neural_sensitivity_development_summary.parquet"
)
TRANSFORMER_GROUPED_RELIANCE_PATH = (
    INPUT_452_FINAL_DIR / "transformer_grouped_reliance_summary.parquet"
)
TRANSFORMER_DEVELOPMENT_SUMMARY_PATH = (
    INPUT_452_FINAL_DIR / "transformer_development_summary.parquet"
)


# ---------------------------------------------------------------------
# 4.6.2 durable outputs
# ---------------------------------------------------------------------

DIAGNOSTIC_CONTRACT_ID = "champion_holdout_error_diagnostic_v2"

DIAGNOSTIC_SURFACE_PATH = (
    INTERMEDIATE_DIR / "champion_holdout_diagnostic_surface.parquet"
)
DIAGNOSTIC_SURFACE_AUDIT_PATH = (
    INTERMEDIATE_DIR / "champion_holdout_diagnostic_surface_audit.parquet"
)

ERROR_BY_TEMPORAL_BUCKET_PATH = (
    INTERMEDIATE_DIR / "error_by_temporal_bucket.parquet"
)
ERROR_BY_BOROUGH_PATH = (
    INTERMEDIATE_DIR / "error_by_borough.parquet"
)
ERROR_BY_POLICY_GEOGRAPHY_PATH = (
    INTERMEDIATE_DIR / "error_by_policy_geography.parquet"
)
ERROR_BY_HISTORY_LEVEL_PATH = (
    INTERMEDIATE_DIR / "error_by_history_level_quantile.parquet"
)
ERROR_BY_HISTORY_VOLATILITY_PATH = (
    INTERMEDIATE_DIR / "error_by_history_volatility_quantile.parquet"
)
ERROR_BY_WEEK_PATH = (
    INTERMEDIATE_DIR / "error_by_calendar_week.parquet"
)

# ---------------------------------------------------------------------
# Final 4.6.2 evaluation handoff
# ---------------------------------------------------------------------
# These compact outputs let 4.7.1 consume the final evaluation evidence
# without rebuilding analyses or copying large intermediate surfaces.

JOB_RELIABILITY_SUMMARY_PATH = (
    FINAL_DIR / "forecast_job_reliability_summary.parquet"
)
FAILURE_MECHANISM_SUMMARY_PATH = (
    FINAL_DIR / "forecast_failure_mechanism_summary.parquet"
)
FAILURE_CASE_STUDIES_PATH = (
    FINAL_DIR / "forecast_failure_case_studies.parquet"
)
FAILURE_CASE_CONTEXT_PATH = (
    FINAL_DIR / "forecast_failure_case_context.parquet"
)
INTERPRETATION_COVERAGE_PATH = (
    FINAL_DIR / "forecast_model_interpretation_coverage.parquet"
)
TREE_HISTORY_SENSITIVITY_PATH = (
    FINAL_DIR / "tree_training_history_sensitivity.parquet"
)
RELIABILITY_GUIDANCE_PATH = (
    FINAL_DIR / "forecast_reliability_guidance.parquet"
)
RECORD_DIAGNOSTICS_PATH = (
    FINAL_DIR / "forecast_record_diagnostics.parquet"
)
EVALUATION_HANDOFF_MANIFEST_PATH = (
    FINAL_DIR / "forecast_evaluation_handoff_manifest.parquet"
)

# ---------------------------------------------------------------------
# Formal Tree information-source ablation
# ---------------------------------------------------------------------
# Versioning prevents a later change in feature membership or training
# semantics from silently reusing results produced under another experiment.

TREE_ABLATION_CONTRACT_ID = "tree_information_source_ablation_v1"

TREE_ABLATION_ROOT = INTERMEDIATE_DIR / "tree_feature_group_ablation"
TREE_ABLATION_CONTRACT_DIR = (
    TREE_ABLATION_ROOT / TREE_ABLATION_CONTRACT_ID
)
TREE_ABLATION_PARTS_DIR = TREE_ABLATION_CONTRACT_DIR / "parts"

TREE_ABLATION_FEATURE_CONTRACT_PATH = (
    TREE_ABLATION_CONTRACT_DIR / "feature_contract.parquet"
)
TREE_ABLATION_WORKLOAD_PATH = (
    TREE_ABLATION_CONTRACT_DIR / "workload.parquet"
)
TREE_ABLATION_FINAL_PATH = (
    FINAL_DIR / "tree_feature_group_ablation.parquet"
)

# Recovery is automatic, but fitting is always explicit. A normal Run All must
# never start retraining merely because one or more ablation units are missing.
RUN_TREE_ABLATION = True
RESET_TREE_ABLATION_CHECKPOINTS = False


# ---------------------------------------------------------------------
# Execution controls
# ---------------------------------------------------------------------

WRITE_INTERMEDIATES = True
WRITE_FINAL_OUTPUTS = True
RUN_VISUALS = True

RUN_DIAGNOSTIC_SURFACE_BUILD = True
REBUILD_DIAGNOSTIC_SURFACE = True




diagnostic_surface_built_this_run = False



In [4]:
# ---------------------------------------------------------------------
# 4.6.2.0C — Schema contracts
# ---------------------------------------------------------------------

# 4.6.2 freezes WHICH upstream artifacts it requires. Their exact physical
# locations come from the authoritative 4.6.1 handoff manifest.
EXPECTED_HANDOFF_ARTIFACT_IDS = {
    "champion_registry",
    "champion_holdout_predictions",
    "champion_holdout_scorecard",
    "validation_holdout_generalization",
    "pre_holdout_runner_up_registry",
    "champion_runner_up_holdout_rows",
    "champion_runner_up_holdout_scorecard",
    "champion_runner_up_robustness",
    "final_champion_board",
    "zone_holdout_reliability",
    "troublesome_zone_surfaces",
}


CHAMPION_REGISTRY_REQUIRED_COLUMNS = [
    "metric", "horizon", "champion_family", "Family", "Representative",
    "modeling_track", "configuration_id", "selection_basis",
    "selection_contract_id", "champion_frozen",
    "holdout_used_for_selection",
]

HOLDOUT_REQUIRED_COLUMNS = [
    "runtime_unit_id", "holdout_runtime_contract_id", "champion_family",
    "modeling_track", "metric", "horizon", "forecast_sequence_id",
    "taxi_zone_id", "origin_observation_sequence_id",
    "target_observation_sequence_id", "target_date", "actual",
    "benchmark_prediction", "prediction_raw", "prediction_final",
    "model_prediction_available", "benchmark_prediction_available",
    "used_benchmark_fallback", "system_prediction",
    "primary_evaluation_eligible",
]

SCORECARD_REQUIRED_COLUMNS = [
    "metric", "horizon", "champion_family", "evaluation_rows",
    "model_available_rows", "fallback_rows", "model_coverage_pct",
    "mae", "rmse", "bias", "abs_bias", "smape_pct",
    "observed_abs_mean", "relative_mae_pct", "benchmark_mae",
    "benchmark_skill_pct",
]

RUNNER_UP_ROBUSTNESS_REQUIRED_COLUMNS = [
    "metric", "horizon", "champion_family", "runner_up_family",
    "comparison_status", "point_advantage_benchmark_pct",
    "ci_low_benchmark_pct", "ci_high_benchmark_pct",
]

ZONE_RELIABILITY_REQUIRED_COLUMNS = [
    "metric", "horizon", "taxi_zone_id", "champion_family",
    "evaluation_rows", "champion_mae", "benchmark_mae",
    "observed_abs_mean", "bias", "job_median_rows", "support_ratio",
    "support_eligible", "benchmark_skill_pct", "relative_mae_pct",
    "zone", "borough",
]

# Only the forecast-time context needed for this first diagnostic phase is read
# from the much wider 4.2.3 forecasting matrices.
MATRIX_DIAGNOSTIC_COLUMNS = [
    "modeling_track", "metric", "horizon", "forecast_sequence_id",
    "taxi_zone_id", "origin_observation_sequence_id",
    "target_observation_sequence_id", "target_date",
    "target_daypart", "target_daypart_order", "target_temporal_bucket",
    "target_temporal_bucket_order", "target_is_weekend", "target_month",
    "target_observed_value", "origin_value",
    "seq_lag_1", "seq_lag_2", "seq_lag_3", "seq_lag_4", "seq_delta_1",
    "same_daypart_lag_1d", "same_daypart_lag_7d",
    "same_daypart_roll_mean_28d", "same_daypart_roll_std_28d",
    "same_daypart_roll_obs_share_28d",
]

ZONE_CONTEXT_REQUIRED_COLUMNS = [
    "location_id", "zone", "borough", "cbd_spatial_category",
]

CANONICAL_CLUSTER_REQUIRED_COLUMNS = [
    "taxi_zone_id",
    "pre_post_cp",
    "cluster_label",
    "canonical_cluster_name",
]

DIAGNOSTIC_SURFACE_REQUIRED_COLUMNS = [
    "diagnostic_contract_id", "champion_family", "modeling_track",
    "metric", "horizon", "forecast_sequence_id", "taxi_zone_id",
    "origin_observation_sequence_id", "target_observation_sequence_id",
    "target_date", "target_daypart", "target_temporal_bucket",
    "actual", "benchmark_prediction", "prediction_raw", "prediction_final",
    "system_prediction", "signed_error", "absolute_error",
    "benchmark_absolute_error", "squared_error", "observed_abs",
    "origin_value", "same_daypart_roll_mean_28d",
    "same_daypart_roll_std_28d", "same_daypart_roll_obs_share_28d",
    "history_28d_complete", "is_transaction_trip_target",
    "actual_zero_event", "origin_zero_event", "history_28d_all_zero_event",
    "raw_prediction_nonpositive", "prediction_was_floored",
    "zone", "borough", "policy_geography", "week_start",
]
# ---------------------------------------------------------------------
# Champion interpretation evidence
# ---------------------------------------------------------------------

TREE_FEATURE_IMPORTANCE_REQUIRED_COLUMNS = [
    "modeling_track", "metric", "horizon", "feature",
    "gain_share", "shap_share",
]

FEATURE_CATALOG_INTERPRETATION_REQUIRED_COLUMNS = [
    "feature_name", "feature_family",
]

NEURAL_GROUPED_IMPORTANCE_REQUIRED_COLUMNS = [
    "metric", "horizon", "input_group", "permutation_identifiable",
    "normalization_available", "reliance_share_pct",
]

TRANSFORMER_GROUPED_RELIANCE_REQUIRED_COLUMNS = [
    "track", "metric", "horizon", "input_group",
    "permutation_identifiable", "reliance_share_pct",
]


TREE_FULL_AUDIT_MODEL_REGISTRY_REQUIRED_COLUMNS = [
    "modeling_track", "metric", "horizon", "status",
    "mae", "nmae", "benchmark_mae", "skill_vs_benchmark_pct",
]

In [5]:
# ---------------------------------------------------------------------
# 4.6.2.0D — Shared data and scoring helpers
# ---------------------------------------------------------------------

def require_file(path: Path, label: str | None = None) -> Path:
    """Fail at the boundary instead of silently substituting another artifact."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing required {label or 'artifact'}: {path}")
    return path


def require_columns(frame: pd.DataFrame, required: list[str], label: str) -> None:
    """Validate an upstream contract before downstream analysis depends on it."""
    missing = sorted(set(required) - set(frame.columns))
    if missing:
        raise ValueError(f"{label} is missing required columns: {missing}")


def atomic_write_parquet(frame: pd.DataFrame, path: Path) -> None:
    """
    Write a DataFrame atomically.

    A notebook interruption should never leave a partial file that looks like
    completed analytical evidence on the next run.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    temp_path = path.with_name(
        f".{path.stem}.{uuid.uuid4().hex}.tmp.parquet"
    )
    frame.to_parquet(temp_path, index=False)
    os.replace(temp_path, path)


def read_champion_holdout_slice(
    path: Path,
    columns: list[str],
    champion_jobs: set[tuple[str, int]],
) -> pd.DataFrame:
    """
    Read only final-holdout rows needed by the frozen champions on one track.

    The filters are passed directly to pd.read_parquet, so the large forecasting
    matrix is narrowed before it becomes an in-memory DataFrame.
    """
    require_file(path)

    filters = [
        [
            ("target_date", ">=", FINAL_HOLDOUT_START_DATE),
            ("target_date", "<=", FINAL_HOLDOUT_END_DATE),
            ("metric", "==", metric),
            ("horizon", "==", int(horizon)),
        ]
        for metric, horizon in sorted(champion_jobs)
    ]

    return pd.read_parquet(path, columns=columns, filters=filters)


def expected_target_horizon_pairs() -> set[tuple[str, int]]:
    """Return the frozen 15-job Target × Horizon roster."""
    return {
        (metric, horizon)
        for metric in FORECAST_TARGETS
        for horizon in FORECAST_HORIZONS
    }


def add_job_labels(frame: pd.DataFrame) -> pd.DataFrame:
    """Attach reader-facing labels without changing the analytical keys."""
    result = frame.copy()
    result["Target"] = result["metric"].map(TARGET_LABELS)
    result["Target type"] = result["metric"].map(TARGET_TYPES)

    result["Job"] = result.apply(
        lambda row: (
            f"{TARGET_SHORT_LABELS[row['metric']]} · h={int(row['horizon'])}"
        ),
        axis=1,
    )
    return result


def summarize_errors(frame: pd.DataFrame, group_columns: list[str]) -> pd.DataFrame:
    """
    Calculate grouped error from additive quantities.

    Relative MAE uses aggregate absolute error divided by aggregate observed
    scale. We deliberately avoid row-level percentage error because legitimate
    zero outcomes would make it unstable or undefined.
    """
    result = (
        frame.groupby(group_columns, observed=True, dropna=False)
        .agg(
            rows=("absolute_error", "size"),
            absolute_error_sum=("absolute_error", "sum"),
            squared_error_sum=("squared_error", "sum"),
            signed_error_sum=("signed_error", "sum"),
            benchmark_absolute_error_sum=("benchmark_absolute_error", "sum"),
            observed_abs_sum=("observed_abs", "sum"),
        )
        .reset_index()
    )

    result["mae"] = result["absolute_error_sum"] / result["rows"]
    result["rmse"] = np.sqrt(result["squared_error_sum"] / result["rows"])
    result["bias"] = result["signed_error_sum"] / result["rows"]
    result["benchmark_mae"] = (
        result["benchmark_absolute_error_sum"] / result["rows"]
    )

    result["benchmark_skill_pct"] = np.where(
        result["benchmark_mae"].gt(0),
        100 * (result["benchmark_mae"] - result["mae"]) / result["benchmark_mae"],
        np.nan,
    )
    result["relative_mae_pct"] = np.where(
        result["observed_abs_sum"].gt(0),
        100 * result["absolute_error_sum"] / result["observed_abs_sum"],
        np.nan,
    )
    return result


def attach_job_error_index(
    frame: pd.DataFrame,
    job_scorecard: pd.DataFrame,
) -> pd.DataFrame:
    """
    Normalize each subgroup against the overall error for the same forecasting job.

    An index of 100 means job-typical difficulty. This lets Taxi trips, Subway
    ridership, and speed appear together without comparing their native MAEs.
    """
    baselines = (
        job_scorecard[["metric", "horizon", "relative_mae_pct"]]
        .rename(columns={"relative_mae_pct": "job_relative_mae_pct"})
    )

    result = frame.merge(
        baselines,
        on=["metric", "horizon"],
        how="left",
        validate="many_to_one",
    )

    result["error_index"] = np.where(
        result["job_relative_mae_pct"].gt(0),
        100 * result["relative_mae_pct"] / result["job_relative_mae_pct"],
        np.nan,
    )
    return result


def build_quantile_error_summary(
    frame: pd.DataFrame,
    value_column: str,
    job_scorecard: pd.DataFrame,
    quantile_count: int = 5,
) -> pd.DataFrame:
    """
    Compare forecast error across low-to-high values of one pre-origin signal.

    These quantiles are descriptive only. Exact all-zero history is analyzed
    separately so tied zeros are not mistaken for a proposed cutoff.
    """
    needed = [
        "metric", "horizon", value_column,
        "absolute_error", "squared_error", "signed_error",
        "benchmark_absolute_error", "observed_abs",
    ]
    working = frame.loc[frame[value_column].notna(), needed].copy()

    pct_rank = (
        working.groupby(["metric", "horizon"], observed=True)[value_column]
        .rank(method="average", pct=True)
    )
    working["quantile"] = (
        np.ceil(pct_rank * quantile_count)
        .clip(1, quantile_count)
        .astype("int8")
    )

    summary = summarize_errors(
        working,
        ["metric", "horizon", "quantile"],
    )

    bounds = (
        working.groupby(
            ["metric", "horizon", "quantile"],
            observed=True,
        )[value_column]
        .agg(
            diagnostic_min="min",
            diagnostic_max="max",
            diagnostic_mean="mean",
        )
        .reset_index()
    )

    summary = summary.merge(
        bounds,
        on=["metric", "horizon", "quantile"],
        how="left",
        validate="one_to_one",
    )
    return attach_job_error_index(summary, job_scorecard)


def hardest_group_per_job(frame: pd.DataFrame) -> pd.DataFrame:
    """Keep the highest normalized-error subgroup for every forecasting job."""
    ranked = frame.sort_values(
        ["metric", "horizon", "error_index", "rows"],
        ascending=[True, True, False, False],
    )

    return (
        ranked.groupby(
            ["metric", "horizon"],
            observed=True,
            as_index=False,
        )
        .head(1)
        .copy()
    )


def dataframe_memory_mb(frame: pd.DataFrame) -> float:
    """Report approximate DataFrame memory so large steps remain transparent."""
    return frame.memory_usage(index=True, deep=True).sum() / (1024 ** 2)


In [6]:
# ---------------------------------------------------------------------
# 4.6.2.0E — Shared visualization helpers
# ---------------------------------------------------------------------

def make_error_index_heatmap(
    frame: pd.DataFrame,
    column_name: str,
    column_order: list[str],
    title: str,
    subtitle: str,
):
    """
    Plot subgroup difficulty relative to each job's own overall holdout error.

    Centering the visual at 100 makes the question intuitive: which conditions
    are easier or harder than normal for that particular forecasting job?
    """
    plot_df = add_job_labels(frame)

    pivot = (
        plot_df.pivot(
            index="Job",
            columns=column_name,
            values="error_index",
        )
        .reindex(
            index=JOB_LABEL_ORDER,
            columns=column_order,
        )
    )

    fig = go.Figure(
        go.Heatmap(
            z=pivot.to_numpy(),
            x=list(pivot.columns),
            y=list(pivot.index),
            colorscale=BRAND_DIVERGING_SEQUENCE,
            zmid=100,
            colorbar={"title": "Error index"},
            hovertemplate=(
                "%{y}<br>%{x}<br>"
                "Error index=%{z:.1f}<extra></extra>"
            ),
        )
    )

    fig.update_layout(
        title={"text": f"{title}<br><sup>{subtitle}</sup>", "x": 0.02},
        height=560,
        margin={"l": 150, "r": 40, "t": 90, "b": 80},
    )
    fig.update_xaxes(title_text="")
    fig.update_yaxes(title_text="")

    apply_branding(fig)
    return fig


def make_quantile_pattern_figure(
    level_frame: pd.DataFrame,
    volatility_frame: pd.DataFrame,
):
    """Compare normalized difficulty across recent target level and volatility."""
    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=(
            "Prior 28-day same-Daypart level",
            "Prior 28-day same-Daypart volatility",
        ),
    )

    for col_idx, frame in enumerate(
        [level_frame, volatility_frame],
        start=1,
    ):
        plot_df = frame.copy()
        plot_df["Target type"] = plot_df["metric"].map(TARGET_TYPES)

        summary = (
            plot_df.groupby(
                ["Target type", "quantile"],
                observed=True,
                as_index=False,
            )
            .agg(median_error_index=("error_index", "median"))
        )

        for color_idx, target_type in enumerate(["Demand", "Speed"]):
            subset = (
                summary.loc[summary["Target type"].eq(target_type)]
                .sort_values("quantile")
            )

            fig.add_trace(
                go.Scatter(
                    x=subset["quantile"],
                    y=subset["median_error_index"],
                    mode="lines+markers",
                    name=target_type,
                    legendgroup=target_type,
                    showlegend=(col_idx == 1),
                    line={"color": BRAND_COLOR_SEQUENCE[color_idx]},
                    hovertemplate=(
                        f"{target_type}<br>"
                        "Quantile=%{x}<br>"
                        "Median error index=%{y:.1f}"
                        "<extra></extra>"
                    ),
                ),
                row=1,
                col=col_idx,
            )

        fig.add_hline(
            y=100,
            line_dash="dot",
            row=1,
            col=col_idx,
        )

    fig.update_xaxes(title_text="Low → high quantile")
    fig.update_yaxes(
        title_text="Median error index",
        row=1,
        col=1,
    )

    fig.update_layout(
        title={
            "text": (
                "Forecast difficulty versus recent historical conditions"
                "<br><sup>"
                "Each job is normalized to 100; values above 100 are harder "
                "than that job's typical holdout error."
                "</sup>"
            ),
            "x": 0.02,
        },
        height=500,
        margin={"l": 70, "r": 30, "t": 95, "b": 60},
    )

    apply_branding(fig)
    return fig

def format_hover_number(value, decimals=3, *, signed=False, suffix=""):
    """Format one reader-facing hover value without changing analytical precision."""
    if pd.isna(value):
        return "Not available"

    sign = "+" if signed else ""
    return f"{value:{sign},.{decimals}f}{suffix}"

## 4.6.2.1 Freeze the Deep-Evaluation Contract

Notebook 4.6.1 has already answered **which model advances for each Target × Horizon job**. That question stays closed.

Before analyzing why forecasts succeed or fail, this section confirms that 4.6.2 is attached to the intended 4.6.1 handoff, that the champion registry still contains the same 15 jobs, and that the final holdout was not used to choose those champions.

This is contract validation, not another model comparison.


**4.6.2.1A — Validate the frozen 4.6.1 evaluation handoff.**

The 4.6.1 handoff manifest is the authoritative directory for downstream evaluation evidence. We use it directly rather than searching the project or reconstructing completed upstream work.

These checks are deliberately lightweight: confirm the expected handoff membership and paths, then validate the small champion registry that defines the forecasting system.


In [7]:
# ---------------------------------------------------------------------
# 4.6.2.1A — Validate the frozen 4.6.1 evaluation handoff
# ---------------------------------------------------------------------

require_file(
    FORECAST_EVALUATION_HANDOFF_PATH,
    "4.6.1 evaluation handoff manifest",
)

forecast_evaluation_handoff_df = pd.read_parquet(
    FORECAST_EVALUATION_HANDOFF_PATH
)

require_columns(
    forecast_evaluation_handoff_df,
    ["artifact_id", "path", "rows", "columns", "purpose"],
    "4.6.1 evaluation handoff manifest",
)


# ---------------------------------------------------------------------
# 1) Prove that this is the complete frozen 4.6.1 handoff
# ---------------------------------------------------------------------
# 4.6.2 defines which evidence it requires. The producer's manifest defines
# where that evidence actually lives. Keeping those responsibilities separate
# prevents downstream code from guessing final_tables versus intermediate_tables.

if forecast_evaluation_handoff_df["artifact_id"].duplicated().any():
    duplicate_ids = (
        forecast_evaluation_handoff_df.loc[
            forecast_evaluation_handoff_df["artifact_id"]
            .duplicated(keep=False),
            "artifact_id",
        ]
        .astype(str)
        .unique()
        .tolist()
    )

    raise RuntimeError(
        "4.6.1 handoff manifest contains duplicate artifact IDs: "
        f"{sorted(duplicate_ids)}"
    )


actual_artifact_ids = set(
    forecast_evaluation_handoff_df["artifact_id"].astype(str)
)

if actual_artifact_ids != EXPECTED_HANDOFF_ARTIFACT_IDS:
    missing_ids = sorted(
        EXPECTED_HANDOFF_ARTIFACT_IDS - actual_artifact_ids
    )
    unexpected_ids = sorted(
        actual_artifact_ids - EXPECTED_HANDOFF_ARTIFACT_IDS
    )

    raise RuntimeError(
        "4.6.1 handoff membership no longer matches the frozen "
        "4.6.2 contract.\n"
        f"  Missing artifact IDs:    {missing_ids}\n"
        f"  Unexpected artifact IDs: {unexpected_ids}"
    )


# ---------------------------------------------------------------------
# 2) Resolve every downstream artifact from the producer's manifest
# ---------------------------------------------------------------------
# The rest of the notebook receives familiar named Path objects, but those
# paths now come from the producer itself rather than a second hardcoded map.

HANDOFF_PATHS = {
    str(row.artifact_id): Path(row.path)
    for row in forecast_evaluation_handoff_df.itertuples(index=False)
}

for artifact_id, artifact_path in HANDOFF_PATHS.items():
    require_file(
        artifact_path,
        f"4.6.1 handoff artifact '{artifact_id}'",
    )


CHAMPION_REGISTRY_PATH = (
    HANDOFF_PATHS["champion_registry"]
)
CHAMPION_HOLDOUT_PREDICTIONS_PATH = (
    HANDOFF_PATHS["champion_holdout_predictions"]
)
CHAMPION_HOLDOUT_SCORECARD_PATH = (
    HANDOFF_PATHS["champion_holdout_scorecard"]
)
VALIDATION_HOLDOUT_GENERALIZATION_PATH = (
    HANDOFF_PATHS["validation_holdout_generalization"]
)

RUNNER_UP_REGISTRY_PATH = (
    HANDOFF_PATHS["pre_holdout_runner_up_registry"]
)
RUNNER_UP_HOLDOUT_ROWS_PATH = (
    HANDOFF_PATHS["champion_runner_up_holdout_rows"]
)
RUNNER_UP_HOLDOUT_SCORECARD_PATH = (
    HANDOFF_PATHS["champion_runner_up_holdout_scorecard"]
)
RUNNER_UP_ROBUSTNESS_PATH = (
    HANDOFF_PATHS["champion_runner_up_robustness"]
)

FINAL_CHAMPION_BOARD_PATH = (
    HANDOFF_PATHS["final_champion_board"]
)
ZONE_RELIABILITY_PATH = (
    HANDOFF_PATHS["zone_holdout_reliability"]
)
TROUBLESOME_SURFACES_PATH = (
    HANDOFF_PATHS["troublesome_zone_surfaces"]
)


# ---------------------------------------------------------------------
# 3) Validate the frozen champion registry
# ---------------------------------------------------------------------

champion_registry_df = pd.read_parquet(
    CHAMPION_REGISTRY_PATH
)

require_columns(
    champion_registry_df,
    CHAMPION_REGISTRY_REQUIRED_COLUMNS,
    "4.6.1 champion registry",
)

actual_jobs = {
    (str(metric), int(horizon))
    for metric, horizon
    in champion_registry_df[
        ["metric", "horizon"]
    ].itertuples(index=False)
}


if (
    len(champion_registry_df) != EXPECTED_JOB_COUNT
    or actual_jobs != expected_target_horizon_pairs()
):
    raise RuntimeError(
        "Champion registry does not contain the exact frozen "
        "15-job Target × Horizon roster."
    )


if champion_registry_df.duplicated(
    ["metric", "horizon"]
).any():
    raise RuntimeError(
        "Champion registry contains duplicate Target × Horizon jobs."
    )


if not (
    champion_registry_df["champion_frozen"]
    .fillna(False)
    .astype(bool)
    .all()
):
    raise RuntimeError(
        "At least one 4.6.1 champion is no longer marked frozen."
    )


if (
    champion_registry_df["holdout_used_for_selection"]
    .fillna(False)
    .astype(bool)
    .any()
):
    raise RuntimeError(
        "The frozen registry indicates that final-holdout evidence "
        "entered model selection."
    )


expected_family_counts = {
    "tree": 8,
    "transformer": 4,
    "neural": 3,
}

actual_family_counts = (
    champion_registry_df["champion_family"]
    .value_counts()
    .to_dict()
)

if actual_family_counts != expected_family_counts:
    raise RuntimeError(
        "Frozen champion-family counts changed. "
        f"Expected {expected_family_counts}; "
        f"observed {actual_family_counts}."
    )


# ---------------------------------------------------------------------
# 4) Show the compact handoff and champion contracts
# ---------------------------------------------------------------------

handoff_display_df = (
    forecast_evaluation_handoff_df[
        [
            "artifact_id",
            "path",
            "rows",
            "columns",
            "purpose",
        ]
    ]
    .rename(
        columns={
            "artifact_id": "Artifact",
            "path": "Producer path",
            "rows": "Rows",
            "columns": "Columns",
            "purpose": "Downstream purpose",
        }
    )
)


champion_display_df = (
    add_job_labels(champion_registry_df)[
        [
            "Target",
            "horizon",
            "champion_family",
            "modeling_track",
            "Representative",
        ]
    ]
    .rename(
        columns={
            "horizon": "Horizon",
            "champion_family": "Champion",
            "modeling_track": "Track",
        }
    )
    .sort_values(["Target", "Horizon"])
)

champion_display_df["Champion"] = (
    champion_display_df["Champion"]
    .map(FAMILY_LABELS)
)


display(handoff_display_df)
display(champion_display_df)


print(
    "PASS — 4.6.2 is bound to the producer-defined "
    "4.6.1 evaluation handoff.\n"
    f"  Handoff artifacts: {len(forecast_evaluation_handoff_df)}\n"
    f"  Frozen jobs:       {len(champion_registry_df)}\n"
    f"  Family counts:     {actual_family_counts}",
    flush=True,
)

,Artifact,Producer path,Rows,Columns,Downstream purpose
0,champion_registry,pipeline_data/4.6.1.final_tables/forecasting_c...,15,17,Frozen validation-selected family for all 15 jobs
1,champion_holdout_predictions,pipeline_data/4.6.1.final_tables/champion_fina...,1427340,20,Row-level unseen-holdout champion forecasts
2,champion_holdout_scorecard,pipeline_data/4.6.1.final_tables/champion_fina...,15,16,15-job unseen-holdout accuracy and benchmark s...
3,validation_holdout_generalization,pipeline_data/4.6.1.final_tables/champion_vali...,15,25,Validation-to-holdout transfer for frozen cham...
4,pre_holdout_runner_up_registry,pipeline_data/4.6.1.intermediate_tables/frozen...,15,28,One strongest pre-holdout alternative per cham...
5,champion_runner_up_holdout_rows,pipeline_data/4.6.1.final_tables/champion_runn...,1421436,20,Row-level champion / runner-up / benchmark com...
6,champion_runner_up_holdout_scorecard,pipeline_data/4.6.1.final_tables/champion_runn...,15,23,15-job champion-versus-runner-up holdout point...
7,champion_runner_up_robustness,pipeline_data/4.6.1.final_tables/champion_runn...,15,34,Weekly-block uncertainty and practical-equival...
8,final_champion_board,pipeline_data/4.6.1.final_tables/final_champio...,15,30,Reader-facing 5 × 3 champion summary
9,zone_holdout_reliability,pipeline_data/4.6.1.final_tables/champion_hold...,3345,23,Target × Horizon × Taxi Zone reliability diagn...


,Target,Horizon,Champion,Track,Representative
9,FHVHV average speed,1,Transformer,full_history_regime_aware,PatchTST-inspired Transformer
10,FHVHV average speed,2,Transformer,full_history_regime_aware,PatchTST-inspired Transformer
11,FHVHV average speed,5,Transformer,full_history_regime_aware,PatchTST-inspired Transformer
6,FHVHV trips,1,Tree,full_history_regime_aware,LightGBM
7,FHVHV trips,2,Tree,full_history_regime_aware,LightGBM
8,FHVHV trips,5,Tree,full_history_regime_aware,LightGBM
12,Subway ridership,1,Tree,full_history_regime_aware,LightGBM
13,Subway ridership,2,Transformer,full_history_regime_aware,PatchTST-inspired Transformer
14,Subway ridership,5,Tree,full_history_regime_aware,LightGBM
3,Taxi average speed,1,Neural,post_cp_primary,Lag-window MLP system


PASS — 4.6.2 is bound to the producer-defined 4.6.1 evaluation handoff.
  Handoff artifacts: 11
  Frozen jobs:       15
  Family counts:     {'tree': 8, 'transformer': 4, 'neural': 3}


**Findings.** Successful execution establishes that 4.6.2 is consuming the intended **11-artifact 4.6.1 evaluation handoff**, the champion registry still contains **15 unique Target × Horizon jobs**, the frozen system remains **8 Tree / 4 Transformer / 3 Neural**, and no champion records the final holdout as selection evidence. Model selection therefore remains closed for this notebook.


## 4.6.2.2 Build the Champion Diagnostic Surface

The row-level 4.6.1 handoff is intentionally lean: it contains the observed outcome, benchmark, champion forecast, model identity, and keys needed for fair evaluation.

Error analysis needs a little more context. This section attaches only what is needed to explain forecasting difficulty: target timing, recent legally available target history, and static Taxi Zone geography.

The **forecast values and evaluation denominator remain exactly those frozen in 4.6.1**. The added fields provide diagnostic context only; they do not change any forecast or redefine which rows belong in the evaluation.

**4.6.2.2A — Validate the diagnostic inputs before the large join.**

The diagnostic surface combines several authoritative sources. Before allocating memory to the large forecast-context join, we confirm that every required file exists and validate the small supporting tables whose schemas can be checked cheaply.

The wide 4.2.3 matrix columns are requested explicitly when they are read. If that upstream contract has changed, Pandas will fail at the read rather than silently substituting another field.


In [8]:
# ---------------------------------------------------------------------
# 4.6.2.2A — Preflight the diagnostic inputs
# ---------------------------------------------------------------------

required_input_paths = {
    "4.6.1 holdout predictions": CHAMPION_HOLDOUT_PREDICTIONS_PATH,
    "4.6.1 holdout scorecard": CHAMPION_HOLDOUT_SCORECARD_PATH,
    "Taxi Zone spatial context": TAXI_ZONE_SPATIAL_CONTEXT_PATH,
}

for track, matrix_path in FORECASTING_MATRIX_PATHS.items():
    required_input_paths[f"4.2.3 {track} matrix"] = matrix_path

for label, path in required_input_paths.items():
    require_file(path, label)


holdout_scorecard_df = pd.read_parquet(
    CHAMPION_HOLDOUT_SCORECARD_PATH
)

require_columns(
    holdout_scorecard_df,
    SCORECARD_REQUIRED_COLUMNS,
    "4.6.1 holdout scorecard",
)

if len(holdout_scorecard_df) != EXPECTED_JOB_COUNT:
    raise RuntimeError(
        "Holdout scorecard does not contain exactly 15 jobs."
    )


zone_context_df = pd.read_parquet(
    TAXI_ZONE_SPATIAL_CONTEXT_PATH,
    columns=ZONE_CONTEXT_REQUIRED_COLUMNS,
)

require_columns(
    zone_context_df,
    ZONE_CONTEXT_REQUIRED_COLUMNS,
    "Taxi Zone spatial context",
)


expected_primary_rows = int(
    holdout_scorecard_df["evaluation_rows"].sum()
)

print(
    "PASS — required diagnostic inputs are available.\n"
    f"  Frozen evaluation rows expected: {expected_primary_rows:,}\n"
    f"  Forecast matrices registered:    {len(FORECASTING_MATRIX_PATHS)}",
    flush=True,
)

PASS — required diagnostic inputs are available.
  Frozen evaluation rows expected: 1,421,436
  Forecast matrices registered:    2


**4.6.2.2B — Build or recover the row-level diagnostic surface.**

The 4.2.3 forecasting matrices contain far more history and features than this notebook needs. Each matrix is therefore read only for the **final holdout dates, frozen champion jobs, and required diagnostic columns**. Everything after that point is ordinary Pandas DataFrame work.

The same physical forecast keys used upstream attach each champion prediction to the history that was available at its forecast origin. Static Taxi Zone geography is then added as descriptive context.

For Taxi and FHVHV trips, the notebook preserves three different zero conditions:

- the **future actual is zero**;
- the **forecast-origin count is zero**;
- all observed same-Daypart values in the prior 28-day history are zero.

Observed-history coverage remains separate, so a fully observed zero history cannot be mistaken for missing data.

In [9]:
# ---------------------------------------------------------------------
# 4.6.2.2B — Build or recover the row-level diagnostic surface
# ---------------------------------------------------------------------

if DIAGNOSTIC_SURFACE_PATH.exists() and not REBUILD_DIAGNOSTIC_SURFACE:
    diagnostic_surface_df = pd.read_parquet(
        DIAGNOSTIC_SURFACE_PATH
    )

    require_columns(
        diagnostic_surface_df,
        DIAGNOSTIC_SURFACE_REQUIRED_COLUMNS,
        "4.6.2 diagnostic surface",
    )

    if not DIAGNOSTIC_SURFACE_AUDIT_PATH.exists():
        raise RuntimeError(
            "Diagnostic surface exists without its successful-contract audit. "
            "Set REBUILD_DIAGNOSTIC_SURFACE=True to recreate it safely."
        )

    diagnostic_surface_audit_df = pd.read_parquet(
        DIAGNOSTIC_SURFACE_AUDIT_PATH
    )

    require_columns(
        diagnostic_surface_audit_df,
        ["diagnostic_contract_id", "rows", "status"],
        "4.6.2 diagnostic-surface audit",
    )

    current_audit = diagnostic_surface_audit_df.loc[
        diagnostic_surface_audit_df["diagnostic_contract_id"]
        .eq(DIAGNOSTIC_CONTRACT_ID)
    ]

    if (
        len(current_audit) != 1
        or current_audit.iloc[0]["status"] != "success"
        or int(current_audit.iloc[0]["rows"]) != len(diagnostic_surface_df)
    ):
        raise RuntimeError(
            "Existing diagnostic surface does not match one successful "
            "current-contract audit."
        )

    diagnostic_surface_built_this_run = False

    print(
        "RECOVERED — validated diagnostic surface loaded directly.\n"
        f"  Rows:   {len(diagnostic_surface_df):,}\n"
        f"  Memory: {dataframe_memory_mb(diagnostic_surface_df):,.1f} MB",
        flush=True,
    )

else:
    if not RUN_DIAGNOSTIC_SURFACE_BUILD:
        raise RuntimeError(
            "Diagnostic surface is missing or marked for rebuild, but "
            "RUN_DIAGNOSTIC_SURFACE_BUILD=False."
        )

    if REBUILD_DIAGNOSTIC_SURFACE:
        DIAGNOSTIC_SURFACE_PATH.unlink(missing_ok=True)
        DIAGNOSTIC_SURFACE_AUDIT_PATH.unlink(missing_ok=True)

    holdout_df = pd.read_parquet(
        CHAMPION_HOLDOUT_PREDICTIONS_PATH,
        columns=HOLDOUT_REQUIRED_COLUMNS,
    )

    require_columns(
        holdout_df,
        HOLDOUT_REQUIRED_COLUMNS,
        "4.6.1 champion holdout predictions",
    )

    holdout_df = holdout_df.loc[
        holdout_df["primary_evaluation_eligible"].astype(bool)
    ].copy()


    # Read only the Target × Horizon jobs actually used by each frozen modeling
    # track. This keeps the 4.2.3 matrix footprint small before the Pandas join.
    champion_jobs_by_track = {}

    for track, group in champion_registry_df.groupby(
        "modeling_track",
        observed=True,
    ):
        champion_jobs_by_track[str(track)] = {
            (str(metric), int(horizon))
            for metric, horizon
            in group[["metric", "horizon"]].itertuples(index=False)
        }


    matrix_parts = []

    for track, champion_jobs in sorted(
        champion_jobs_by_track.items()
    ):
        if track not in FORECASTING_MATRIX_PATHS:
            raise RuntimeError(
                "No authoritative 4.2.3 matrix is registered for "
                f"track: {track}"
            )

        matrix_part = read_champion_holdout_slice(
            FORECASTING_MATRIX_PATHS[track],
            MATRIX_DIAGNOSTIC_COLUMNS,
            champion_jobs,
        )

        require_columns(
            matrix_part,
            MATRIX_DIAGNOSTIC_COLUMNS,
            f"4.2.3 {track} holdout context",
        )

        matrix_parts.append(matrix_part)

        print(
            f"Loaded {track}: {len(matrix_part):,} rows "
            f"× {len(matrix_part.columns)} columns "
            f"({dataframe_memory_mb(matrix_part):,.1f} MB).",
            flush=True,
        )


    matrix_context_df = pd.concat(
        matrix_parts,
        ignore_index=True,
    )

    del matrix_parts
    gc.collect()


    join_keys = [
        "modeling_track",
        "metric",
        "horizon",
        "forecast_sequence_id",
        "taxi_zone_id",
        "origin_observation_sequence_id",
        "target_observation_sequence_id",
    ]

    if matrix_context_df.duplicated(join_keys).any():
        raise RuntimeError(
            "4.2.3 context is not unique on the frozen physical forecast keys."
        )


    # target_date is already frozen in 4.6.1. Keeping that authoritative copy
    # prevents the join from creating competing target-date columns.
    matrix_context_df = matrix_context_df.drop(
        columns=["target_date"]
    )

    diagnostic_surface_df = holdout_df.merge(
        matrix_context_df,
        on=join_keys,
        how="left",
        validate="one_to_one",
    )

    del holdout_df, matrix_context_df
    gc.collect()


    zone_context_for_join_df = zone_context_df.rename(
        columns={"location_id": "taxi_zone_id"}
    )

    if zone_context_for_join_df.duplicated(
        ["taxi_zone_id"]
    ).any():
        raise RuntimeError(
            "Taxi Zone spatial context is not unique by location_id."
        )

    diagnostic_surface_df = diagnostic_surface_df.merge(
        zone_context_for_join_df,
        on="taxi_zone_id",
        how="left",
        validate="many_to_one",
    )



    diagnostic_surface_df["diagnostic_contract_id"] = (
        DIAGNOSTIC_CONTRACT_ID
    )

    diagnostic_surface_df["target_date"] = pd.to_datetime(
        diagnostic_surface_df["target_date"]
    )


    # All diagnostics remain tied to the exact system prediction frozen in
    # 4.6.1; contextual joins never alter a forecast or evaluation denominator.
    diagnostic_surface_df["signed_error"] = (
        diagnostic_surface_df["system_prediction"]
        - diagnostic_surface_df["actual"]
    )

    diagnostic_surface_df["absolute_error"] = (
        diagnostic_surface_df["signed_error"].abs()
    )

    diagnostic_surface_df["squared_error"] = (
        diagnostic_surface_df["signed_error"].pow(2)
    )

    diagnostic_surface_df["benchmark_absolute_error"] = (
        diagnostic_surface_df["benchmark_prediction"]
        - diagnostic_surface_df["actual"]
    ).abs()

    diagnostic_surface_df["observed_abs"] = (
        diagnostic_surface_df["actual"].abs()
    )


    trip_mask = diagnostic_surface_df["metric"].isin(
        TRANSACTION_EVENT_TARGETS
    )

    observed_history_mask = (
        trip_mask
        & diagnostic_surface_df[
            "same_daypart_roll_obs_share_28d"
        ].gt(0)
    )


    diagnostic_surface_df["is_transaction_trip_target"] = (
        trip_mask
    )

    diagnostic_surface_df["actual_zero_event"] = (
        diagnostic_surface_df["actual"]
        .eq(0)
        .where(trip_mask)
        .astype("boolean")
    )

    diagnostic_surface_df["origin_zero_event"] = (
        diagnostic_surface_df["origin_value"]
        .eq(0)
        .where(trip_mask)
        .astype("boolean")
    )

    diagnostic_surface_df["history_28d_all_zero_event"] = (
        diagnostic_surface_df["same_daypart_roll_mean_28d"]
        .eq(0)
        .where(observed_history_mask)
        .astype("boolean")
    )

    diagnostic_surface_df["history_28d_complete"] = (
        diagnostic_surface_df["same_daypart_roll_obs_share_28d"]
        .ge(0.999999)
    )

    diagnostic_surface_df["raw_prediction_nonpositive"] = (
        diagnostic_surface_df["prediction_raw"].le(0)
    )

    diagnostic_surface_df["prediction_was_floored"] = (
        diagnostic_surface_df["prediction_raw"].lt(0)
        & diagnostic_surface_df["prediction_final"].eq(0)
    )


    diagnostic_surface_df["policy_geography"] = (
        diagnostic_surface_df["cbd_spatial_category"]
        .map(POLICY_GEOGRAPHY_LABELS)
        .fillna("Unknown")
    )


    # Monday-based weeks let us distinguish shared difficult periods from
    # isolated row-level misses without introducing another rolling window.
    diagnostic_surface_df["week_start"] = (
        diagnostic_surface_df["target_date"]
        - pd.to_timedelta(
            diagnostic_surface_df["target_date"].dt.weekday,
            unit="D",
        )
    ).dt.normalize()


    atomic_write_parquet(
        diagnostic_surface_df,
        DIAGNOSTIC_SURFACE_PATH,
    )

    # A newly written artifact becomes recoverable only after 2C proves that
    # the joins preserved the frozen forecasting problem.
    DIAGNOSTIC_SURFACE_AUDIT_PATH.unlink(
        missing_ok=True
    )

    diagnostic_surface_built_this_run = True

    print(
        "BUILT — champion diagnostic surface materialized.\n"
        f"  Rows:   {len(diagnostic_surface_df):,}\n"
        f"  Memory: {dataframe_memory_mb(diagnostic_surface_df):,.1f} MB\n"
        "  Full semantic QA follows before this becomes the recovery path.",
        flush=True,
    )

Loaded full_history_regime_aware: 1,197,120 rows × 26 columns (600.1 MB).
Loaded post_cp_primary: 243,810 rows × 26 columns (120.9 MB).
BUILT — champion diagnostic surface materialized.
  Rows:   1,421,436
  Memory: 1,656.5 MB
  Full semantic QA follows before this becomes the recovery path.


**4.6.2.2C — Prove that the diagnostic join preserved the forecasting problem.**

When the diagnostic surface is first created, we run the expensive semantic checks once. Its row count must reproduce the frozen 4.6.1 evaluation denominator, the 4.2.3 target value must reproduce the frozen actual outcome, forecast keys must remain unique, and transaction-derived trip counts must remain valid nonnegative whole counts.

Only after those checks pass do we save the small success audit used by later reruns.


In [10]:
# ---------------------------------------------------------------------
# 4.6.2.2C — Validate the diagnostic surface
# ---------------------------------------------------------------------

require_columns(
    diagnostic_surface_df,
    DIAGNOSTIC_SURFACE_REQUIRED_COLUMNS,
    "4.6.2 diagnostic surface",
)

diagnostic_rows = len(diagnostic_surface_df)

if diagnostic_rows != expected_primary_rows:
    raise RuntimeError(
        f"Diagnostic surface has {diagnostic_rows:,} rows; "
        f"expected {expected_primary_rows:,}."
    )

if diagnostic_surface_built_this_run:
    missing_temporal_context_rows = (
        diagnostic_surface_df["target_temporal_bucket"].isna().sum()
    )
    missing_matrix_target_rows = (
        diagnostic_surface_df["target_observed_value"].isna().sum()
    )

    matched_target_mask = (
        diagnostic_surface_df["target_observed_value"].notna()
    )
    mismatched_target_value_rows = (
        ~np.isclose(
            diagnostic_surface_df.loc[
                matched_target_mask,
                "target_observed_value",
            ],
            diagnostic_surface_df.loc[
                matched_target_mask,
                "actual",
            ],
            rtol=0,
            atol=1e-8,
        )
    ).sum()

    duplicate_key = [
        "metric",
        "horizon",
        "forecast_sequence_id",
        "taxi_zone_id",
        "origin_observation_sequence_id",
        "target_observation_sequence_id",
    ]
    duplicate_rows = (
        diagnostic_surface_df.duplicated(duplicate_key).sum()
    )

    trip_rows = diagnostic_surface_df.loc[
        diagnostic_surface_df["is_transaction_trip_target"]
    ]
    invalid_transaction_count_rows = (
        trip_rows["actual"].lt(0)
        | ~np.isclose(
            trip_rows["actual"],
            np.round(trip_rows["actual"]),
            rtol=0,
            atol=1e-8,
        )
    ).sum()

    if missing_temporal_context_rows:
        raise RuntimeError(
            f"{missing_temporal_context_rows:,} rows failed to recover "
            "target-time context from 4.2.3."
        )

    if missing_matrix_target_rows or mismatched_target_value_rows:
        raise RuntimeError(
            "4.2.3 target outcomes do not reproduce the frozen "
            "4.6.1 actual values."
        )

    if duplicate_rows:
        raise RuntimeError(
            f"Diagnostic join duplicated {duplicate_rows:,} forecast rows."
        )

    if invalid_transaction_count_rows:
        raise RuntimeError(
            "Taxi/FHVHV trip outcomes contain invalid negative "
            "or fractional values."
        )

    zone_context_coverage_pct = (
        100 * diagnostic_surface_df["zone"].notna().mean()
    )
    diagnostic_surface_audit_df = pd.DataFrame(
        [{
            "diagnostic_contract_id": DIAGNOSTIC_CONTRACT_ID,
            "rows": diagnostic_rows,
            "min_target_date": diagnostic_surface_df["target_date"].min(),
            "max_target_date": diagnostic_surface_df["target_date"].max(),
            "zone_context_coverage_pct": zone_context_coverage_pct,
            "status": "success",
        }]
    )

    atomic_write_parquet(
        diagnostic_surface_audit_df,
        DIAGNOSTIC_SURFACE_AUDIT_PATH,
    )

else:
    diagnostic_surface_audit_df = pd.read_parquet(
        DIAGNOSTIC_SURFACE_AUDIT_PATH
    )
    diagnostic_surface_audit_df = (
        diagnostic_surface_audit_df.loc[
            diagnostic_surface_audit_df["diagnostic_contract_id"]
            .eq(DIAGNOSTIC_CONTRACT_ID)
        ]
        .copy()
    )

if (
    len(diagnostic_surface_audit_df) != 1
    or diagnostic_surface_audit_df.iloc[0]["status"] != "success"
):
    raise RuntimeError(
        "Diagnostic surface lacks one successful current-contract audit."
    )

display(diagnostic_surface_audit_df.round(3))

print(
    "PASS — champion diagnostic surface is ready for error analysis.\n"
    f"  Primary rows: {diagnostic_rows:,}\n"
    f"  Contract:     {DIAGNOSTIC_CONTRACT_ID}",
    flush=True,
)


,diagnostic_contract_id,rows,min_target_date,max_target_date,zone_context_coverage_pct,status
0,champion_holdout_error_diagnostic_v2,1421436,2026-01-05,2026-03-31,100.000,success


PASS — champion diagnostic surface is ready for error analysis.
  Primary rows: 1,421,436
  Contract:     champion_holdout_error_diagnostic_v2


**Findings.** The diagnostic surface reproduces the frozen 4.6.1 evaluation population exactly: **1,421,436 primary holdout forecasts** spanning January 5 through March 31, 2026. Every forecast recovered its target-time context from 4.2.3, the recovered target values matched the outcomes frozen in 4.6.1, and the join introduced no duplicate forecast rows. Taxi and FHVHV trip outcomes also passed the checks for nonnegative whole-number transaction counts.

Static Taxi Zone context covered **100% of rows**. The forecasting problem therefore remains exactly the one evaluated in 4.6.1; this section has only attached explanatory context.

## 4.6.2.3 Map Where and When Forecast Error Concentrates

The final champion system beats the weekly benchmark overall, but an average score can hide important weak surfaces.

This section asks whether forecasting difficulty concentrates by **Target × Horizon, temporal bucket, borough, policy geography, genuine zero-trip outcomes, recent historical activity, historical volatility, or calendar week**.

Native-unit errors remain useful within a target. Cross-target views use aggregate **Relative MAE** or an **error index** that first normalizes each subgroup against its own Target × Horizon job. We do not use row-level percentage error or MAPE because observed mobility can legitimately be zero.


<img src="image-20260915-002129.png" width="" align="" />

### 4.6.2.3.1 Start with Job-Level Fragility

Before explaining individual errors, we need to see which of the 15 champion jobs have broad reliability and which have weak tails.

The comparison below combines three already-frozen perspectives:

- overall Relative MAE and benchmark skill;
- the share of adequately supported Taxi Zones where the champion beats the weekly benchmark;
- the champion-versus-runner-up holdout robustness classification.

These measures answer different questions. They are shown together to identify diagnostic priorities—not to reopen model selection.


**4.6.2.3.1A — Compare overall accuracy, geographic breadth, and runner-up robustness.**


In [11]:
# ---------------------------------------------------------------------
# 4.6.2.3.1A — Compare job-level fragility
# ---------------------------------------------------------------------

runner_up_robustness_df = pd.read_parquet(
    RUNNER_UP_ROBUSTNESS_PATH
)
zone_reliability_df = pd.read_parquet(
    ZONE_RELIABILITY_PATH
)

require_columns(
    runner_up_robustness_df,
    RUNNER_UP_ROBUSTNESS_REQUIRED_COLUMNS,
    "runner-up robustness",
)
require_columns(
    zone_reliability_df,
    ZONE_RELIABILITY_REQUIRED_COLUMNS,
    "zone reliability",
)

# Reuse 4.6.1's support definition rather than inventing a new geographic
# eligibility rule inside the diagnostic notebook.
supported_zone_summary_df = (
    zone_reliability_df.loc[
        zone_reliability_df["support_eligible"]
        .fillna(False)
        .astype(bool)
    ]
    .groupby(
        ["metric", "horizon", "champion_family"],
        observed=True,
        as_index=False,
    )
    .agg(
        supported_zones=("taxi_zone_id", "nunique"),
        positive_skill_zones=(
            "benchmark_skill_pct",
            lambda s: int(s.gt(0).sum()),
        ),
        median_zone_skill_pct=("benchmark_skill_pct", "median"),
        p10_zone_skill_pct=(
            "benchmark_skill_pct",
            lambda s: s.quantile(0.10),
        ),
    )
)

supported_zone_summary_df["positive_skill_zone_pct"] = (
    100
    * supported_zone_summary_df["positive_skill_zones"]
    / supported_zone_summary_df["supported_zones"]
)

job_fragility_df = (
    holdout_scorecard_df
    .merge(
        supported_zone_summary_df,
        on=["metric", "horizon", "champion_family"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        runner_up_robustness_df[
            [
                "metric",
                "horizon",
                "champion_family",
                "runner_up_family",
                "comparison_status",
                "point_advantage_benchmark_pct",
                "ci_low_benchmark_pct",
                "ci_high_benchmark_pct",
            ]
        ],
        on=["metric", "horizon", "champion_family"],
        how="left",
        validate="one_to_one",
    )
)

job_fragility_df = add_job_labels(job_fragility_df)

fragility_display_df = (
    job_fragility_df[
        [
            "Target",
            "horizon",
            "champion_family",
            "relative_mae_pct",
            "benchmark_skill_pct",
            "positive_skill_zone_pct",
            "median_zone_skill_pct",
            "p10_zone_skill_pct",
            "runner_up_family",
            "comparison_status",
        ]
    ]
    .rename(
        columns={
            "horizon": "Horizon",
            "champion_family": "Champion",
            "relative_mae_pct": "Relative MAE (%)",
            "benchmark_skill_pct": "Benchmark skill (%)",
            "positive_skill_zone_pct":
                "Supported zones beating benchmark (%)",
            "median_zone_skill_pct": "Median zone skill (%)",
            "p10_zone_skill_pct": "10th-percentile zone skill (%)",
            "runner_up_family": "Runner-up",
            "comparison_status": "Runner-up audit",
        }
    )
    .sort_values("Supported zones beating benchmark (%)")
)

fragility_display_df["Champion"] = (
    fragility_display_df["Champion"].map(FAMILY_LABELS)
)
fragility_display_df["Runner-up"] = (
    fragility_display_df["Runner-up"].map(FAMILY_LABELS)
)

display(fragility_display_df.round(3))


if RUN_VISUALS:
    # Use an explicit marker-safe mapping so every mobility target keeps one
    # visible color. Ice is intentionally excluded because the chart background
    # itself uses Ice, making Ice-filled markers effectively disappear.
    target_marker_colors = {
        TARGET_LABELS["taxi_trip_count"]: BRAND_COLORS["dark_teal"],
        TARGET_LABELS["taxi_avg_trip_speed"]: BRAND_COLORS["terracotta"],
        TARGET_LABELS["fhvhv_trip_count"]: BRAND_COLORS["seafoam"],
        TARGET_LABELS["fhvhv_avg_trip_speed"]: BRAND_CLUSTER_PALETTE[3],
        TARGET_LABELS["subway_ridership"]: BRAND_CLUSTER_PALETTE[4],
    }

    fig = px.scatter(
        job_fragility_df,
        x="positive_skill_zone_pct",
        y="relative_mae_pct",
        color="Target",
        symbol="horizon",
        text="Job",
        color_discrete_map=target_marker_colors,
        custom_data=[
            "champion_family",
            "benchmark_skill_pct",
            "comparison_status",
            "runner_up_family",
            "median_zone_skill_pct",
            "supported_zones",
        ],
    )

    fig.update_traces(
        textposition="top center",
        marker={
            "size": 11,
            "line": {
                "color": BRAND_COLORS["dark_teal"],
                "width": 1,
            },
        },
        hovertemplate=(
            "<b>%{text}</b><br>"
            "Supported zones beating benchmark=%{x:.1f}%<br>"
            "Relative MAE=%{y:.2f}%<br>"
            "Champion=%{customdata[0]}<br>"
            "Overall benchmark skill=%{customdata[1]:.1f}%<br>"
            "Runner-up audit=%{customdata[2]}<br>"
            "Runner-up=%{customdata[3]}<br>"
            "Median zone skill=%{customdata[4]:.1f}%<br>"
            "Supported zones=%{customdata[5]}"
            "<extra></extra>"
        ),
    )

    fig.update_layout(
        title={
            "text": (
                "Which champion jobs are broadly reliable, "
                "and which have weak geographic tails?"
                "<br><sup>"
                "Lower Relative MAE and broader positive zone skill "
                "indicate a stronger reliability surface."
                "</sup>"
            ),
            "x": 0.02,
        },
        xaxis_title="Supported Taxi Zones beating weekly benchmark (%)",
        yaxis_title="Relative MAE (%)",
        height=600,
        margin={"l": 70, "r": 30, "t": 100, "b": 70},
    )

    apply_branding(fig)
    fig.show()

,Target,Horizon,Champion,Relative MAE (%),Benchmark skill (%),Supported zones beating benchmark (%),Median zone skill (%),10th-percentile zone skill (%),Runner-up,Runner-up audit
7,Subway ridership,2,Transformer,11.130,19.223,70.588,12.886,-33.942,Tree,runner_up_clear_win
13,Taxi trips,2,Tree,15.829,26.304,73.256,9.176,-83.937,Neural,champion_clear_win
12,Taxi trips,1,Tree,14.676,31.669,80.233,12.455,-64.519,Transformer,uncertain_difference
14,Taxi trips,5,Tree,15.192,29.270,86.822,18.397,-18.356,Transformer,champion_clear_win
5,FHVHV trips,5,Tree,12.109,28.491,87.549,24.697,-12.818,Transformer,champion_clear_win
4,FHVHV trips,2,Tree,12.043,28.885,91.440,23.780,1.735,Transformer,uncertain_difference
3,FHVHV trips,1,Tree,10.556,37.665,93.774,31.925,12.675,Transformer,uncertain_difference
6,Subway ridership,1,Tree,8.841,35.836,95.425,33.806,15.940,Transformer,uncertain_difference
8,Subway ridership,5,Tree,9.871,28.361,96.078,27.983,15.329,Transformer,champion_clear_win
1,FHVHV average speed,2,Transformer,6.205,32.428,100.000,33.638,27.611,Tree,uncertain_difference


**Findings.** Overall accuracy and geographic reliability tell somewhat different stories. The speed forecasts are the most geographically dependable: both FHVHV and Taxi speed beat the weekly benchmark in **100% of support-qualified Taxi Zones** at every horizon. FHVHV speed is especially strong, with Relative MAE of roughly **5.8%–6.4%** and even its weaker geographic tail remaining comfortably positive versus the benchmark.

Demand forecasts have much weaker geographic tails. FHVHV trips still beat the benchmark in about **88%–94%** of supported zones, while Taxi trips fall to about **73%–87%**. This means a respectable citywide Taxi-trip score can hide severe local failures.

**Subway h=2 remains the clearest multi-dimensional fragility case.** Its Relative MAE is **11.1%**, benchmark skill falls to **19.2%**, only **70.6%** of supported zones beat the benchmark, and its weakest 10% of zones have benchmark skill of **−33.9%** or worse. It is also the only job where the frozen runner-up clearly wins on the unseen holdout. That does not reopen model selection, but it reinforces that Subway h=2 deserves separate failure analysis.

### 4.6.2.3.2 Locate error across time and mobility geography

Job-level averages can hide whether forecast difficulty is concentrated in particular parts of the week or particular kinds of places.

This subsection therefore looks at the same frozen holdout errors through three complementary lenses:

- **temporal bucket**, to identify recurring times of week when each target becomes harder to forecast;
- **borough and congestion-pricing geography**, to identify conventional geographic concentration;
- **mobility environment**, using the canonical five-cluster solution selected in 3.2.2 to test whether forecasting difficulty aligns with recurring multimodal mobility structure rather than administrative boundaries alone.

The mobility-environment assignments are used here only as **retrospective descriptive context**. They help explain where errors concentrate, but they are not treated as leakage-safe forecast-time inputs or automatic reliability rules.

**4.6.2.3.2A — Compare error across the ten temporal buckets.**


In [12]:
# ---------------------------------------------------------------------
# 4.6.2.3.2A — Compare error across temporal buckets
# ---------------------------------------------------------------------

temporal_error_df = summarize_errors(
    diagnostic_surface_df,
    ["metric", "horizon", "target_temporal_bucket"],
)
temporal_error_df = attach_job_error_index(
    temporal_error_df,
    holdout_scorecard_df,
)

temporal_error_df["Temporal bucket"] = (
    temporal_error_df["target_temporal_bucket"]
    .map(TEMPORAL_BUCKET_LABELS)
)

if WRITE_INTERMEDIATES:
    atomic_write_parquet(
        temporal_error_df,
        ERROR_BY_TEMPORAL_BUCKET_PATH,
    )

hardest_temporal_df = hardest_group_per_job(
    temporal_error_df
)

hardest_temporal_display_df = (
    add_job_labels(hardest_temporal_df)[
        [
            "Target",
            "horizon",
            "Temporal bucket",
            "rows",
            "relative_mae_pct",
            "error_index",
            "benchmark_skill_pct",
            "bias",
        ]
    ]
    .rename(
        columns={
            "horizon": "Horizon",
            "rows": "Rows",
            "relative_mae_pct": "Relative MAE (%)",
            "error_index": "Error index",
            "benchmark_skill_pct": "Benchmark skill (%)",
            "bias": "Bias",
        }
    )
)

display(hardest_temporal_display_df.round(3))

if RUN_VISUALS:
    fig = make_error_index_heatmap(
        temporal_error_df,
        column_name="Temporal bucket",
        column_order=[
            TEMPORAL_BUCKET_LABELS[bucket]
            for bucket in TEMPORAL_BUCKET_ORDER
        ],
        title="When during the week does each forecasting job become harder?",
        subtitle=(
            "Each temporal bucket is compared with that job's own "
            "overall holdout error; 100 = typical."
        ),
    )
    fig.show()


,Target,Horizon,Temporal bucket,Rows,Relative MAE (%),Error index,Benchmark skill (%),Bias
0,FHVHV average speed,1,Weekday · AM peak,15890,7.481,128.211,44.272,0.171
10,FHVHV average speed,2,Weekday · AM peak,15890,7.917,127.596,41.021,0.187
20,FHVHV average speed,5,Weekday · AM peak,15890,8.313,130.516,38.073,0.166
33,FHVHV trips,1,Weekday · Overnight,15797,14.071,133.301,45.502,2.984
43,FHVHV trips,2,Weekday · Overnight,15797,16.187,134.411,37.308,8.656
53,FHVHV trips,5,Weekday · Overnight,15797,16.511,136.352,36.051,-10.120
68,Subway ridership,1,Weekend · Overnight,3646,19.075,215.767,-10.440,48.392
78,Subway ridership,2,Weekend · Overnight,3646,25.144,225.914,-45.574,39.446
88,Subway ridership,5,Weekend · Overnight,3646,19.129,193.797,-10.751,-5.189
94,Taxi average speed,1,Weekday · PM peak,11454,25.225,157.525,23.892,0.072


**Findings.** Each mobility target has a consistent time of week when forecasting becomes hardest, and that pattern is remarkably stable across horizons.

For **Taxi and FHVHV trip counts**, the hardest period is **weekday overnight** at h=1, h=2, and h=5. Taxi is especially sensitive: its error index rises to roughly **205–213**, meaning overnight error is about twice the job's typical level, while Relative MAE reaches about **30%–33%**. FHVHV trip forecasts also weaken overnight, but less dramatically, with error indices around **133–136**.

The speed targets struggle at different commute periods. **Taxi speed is hardest during the weekday PM peak**, with error indices around **154–158**, while **FHVHV speed is hardest during the weekday AM peak**, at roughly **128–131**. Despite these increases, both speed models still beat the weekly benchmark in their hardest periods.

**Subway is the clearest temporal failure case.** Weekend overnight is the hardest bucket at all three horizons, with error indices of roughly **194–226**. Unlike the other targets, the champion actually loses to the weekly benchmark there: benchmark skill is about **−10% at h=1 and h=5**, and falls to **−45.6% at h=2**.

The main lesson is that forecast difficulty is not spread randomly across the week. Each mode has a repeatable weak period, and **Subway weekend overnight—especially h=2—stands out as the only case where the hardest temporal regime consistently overturns the benchmark advantage.**

**4.6.2.3.2B — Compare error across geography.**

In [13]:
# ---------------------------------------------------------------------
# 4.6.2.3.2B — Compare geographic error
# ---------------------------------------------------------------------

borough_mask = (
    diagnostic_surface_df["borough"].notna()
    & diagnostic_surface_df["borough"].ne("Unknown")
)

policy_mask = (
    diagnostic_surface_df["policy_geography"].notna()
    & diagnostic_surface_df["policy_geography"].ne("Unknown")
)


borough_error_df = attach_job_error_index(
    summarize_errors(
        diagnostic_surface_df.loc[borough_mask],
        ["metric", "horizon", "borough"],
    ),
    holdout_scorecard_df,
)

policy_error_df = attach_job_error_index(
    summarize_errors(
        diagnostic_surface_df.loc[policy_mask],
        ["metric", "horizon", "policy_geography"],
    ),
    holdout_scorecard_df,
)


if WRITE_INTERMEDIATES:
    atomic_write_parquet(
        borough_error_df,
        ERROR_BY_BOROUGH_PATH,
    )

    atomic_write_parquet(
        policy_error_df,
        ERROR_BY_POLICY_GEOGRAPHY_PATH,
    )


hardest_borough = (
    hardest_group_per_job(borough_error_df)
    .rename(
        columns={
            "borough": "Hardest borough",
            "error_index": "Borough error index",
        }
    )
)

hardest_policy = (
    hardest_group_per_job(policy_error_df)
    .rename(
        columns={
            "policy_geography": "Hardest policy geography",
            "error_index": "Policy error index",
        }
    )
)


geography_summary_df = (
    hardest_borough[
        [
            "metric",
            "horizon",
            "Hardest borough",
            "Borough error index",
        ]
    ]
    .merge(
        hardest_policy[
            [
                "metric",
                "horizon",
                "Hardest policy geography",
                "Policy error index",
            ]
        ],
        on=["metric", "horizon"],
        how="outer",
        validate="one_to_one",
    )
)


geography_summary_df = (
    add_job_labels(geography_summary_df)[
        [
            "Target",
            "horizon",
            "Hardest borough",
            "Borough error index",
            "Hardest policy geography",
            "Policy error index",
        ]
    ]
    .rename(
        columns={"horizon": "Horizon"}
    )
)

display(
    geography_summary_df.round(3)
)

,Target,Horizon,Hardest borough,Borough error index,Hardest policy geography,Policy error index
0,FHVHV average speed,1,Staten Island,119.223,Outside,100.004
1,FHVHV average speed,2,Staten Island,115.600,Outside,99.865
2,FHVHV average speed,5,Staten Island,114.515,CBD,99.823
3,FHVHV trips,1,Staten Island,137.281,Gateway + adjacent,104.973
4,FHVHV trips,2,Staten Island,127.503,Gateway + adjacent,104.238
5,FHVHV trips,5,Staten Island,143.189,Gateway + adjacent,102.672
6,Subway ridership,1,Staten Island,234.907,Outside,103.731
7,Subway ridership,2,Staten Island,357.898,Outside,109.745
8,Subway ridership,5,Staten Island,182.762,Gateway + adjacent,103.140
9,Taxi average speed,1,Bronx,133.162,Outside,112.189


**Findings.** Geographically, **Staten Island is the hardest borough for 12 of the 15 forecasting jobs**—every horizon for Taxi trips, FHVHV trips, FHVHV speed, and Subway. Taxi speed is the exception, with the Bronx hardest at all three horizons. The concentration is particularly extreme for Taxi trips: Staten Island's normalized error is roughly **25 to 41 times** the job-wide level depending on horizon.

The broader congestion-pricing geography categories are much less extreme. Most CBD, gateway/adjacent, and outside-CBD error levels remain relatively close to each job's overall average, although outside-CBD Taxi-trip forecasts are consistently weaker. The largest geographic failures therefore appear to be **localized borough- and zone-level problems**, not simply an inside-versus-outside-CBD effect.

**4.6.2.3.2C — Compare forecast error across mobility environments.**

Boroughs and policy geography describe where a Taxi Zone is located, but they do not necessarily describe how that area behaves as part of the mobility system.

The canonical Chapter 3 clustering provides a second geographic lens: five mobility environments defined from multimodal behavior. Because the entire final holdout occurs after congestion pricing began, each forecast can be attached to its frozen **post-CP** mobility-environment assignment.

This block compares each Target × Horizon job across those five environments using the same error measures already used elsewhere in this section. The heatmap emphasizes the **error index**, where 100 represents the job's overall holdout error level, values above 100 indicate a harder-than-usual environment, and values below 100 indicate an easier-than-usual environment.

In [14]:
# ---------------------------------------------------------------------
# 4.6.2.3.2C — Compare forecast error across mobility environments
# ---------------------------------------------------------------------

canonical_cluster_assignments_df = pd.read_parquet(
    CANONICAL_CLUSTER_ASSIGNMENTS_PATH,
    columns=CANONICAL_CLUSTER_REQUIRED_COLUMNS,
)

require_columns(
    canonical_cluster_assignments_df,
    CANONICAL_CLUSTER_REQUIRED_COLUMNS,
    "canonical cluster assignments",
)

# The final holdout is entirely Post-CP, so each Taxi Zone should have exactly
# one applicable mobility-environment assignment. The pre-CP assignments are
# not needed for this diagnostic population.
post_cp_cluster_df = (
    canonical_cluster_assignments_df.loc[
        canonical_cluster_assignments_df["pre_post_cp"].eq("post_cp"),
        ["taxi_zone_id", "cluster_label", "canonical_cluster_name"],
    ]
    .sort_values(["cluster_label", "taxi_zone_id"])
    .reset_index(drop=True)
)

assert len(post_cp_cluster_df) == 263, (
    "Expected one Post-CP cluster assignment for each of the 263 Taxi Zones."
)
assert post_cp_cluster_df["taxi_zone_id"].is_unique, (
    "Post-CP cluster assignments must be unique by taxi_zone_id."
)
assert post_cp_cluster_df["canonical_cluster_name"].nunique() == 5, (
    "Expected the five canonical mobility environments selected in 3.2.2."
)

cluster_error_rows_df = diagnostic_surface_df.merge(
    post_cp_cluster_df,
    on="taxi_zone_id",
    how="left",
    validate="many_to_one",
)

assert cluster_error_rows_df["canonical_cluster_name"].notna().all(), (
    "Every holdout forecast must resolve to a Post-CP mobility environment."
)

# Recompute the grouped metrics from the frozen row-level predictions so every
# mobility environment is evaluated on its own observed scale and denominator.
cluster_error_rows_df["absolute_error"] = (
    cluster_error_rows_df["prediction_final"]
    - cluster_error_rows_df["actual"]
).abs()

cluster_error_rows_df["benchmark_absolute_error"] = (
    cluster_error_rows_df["benchmark_prediction"]
    - cluster_error_rows_df["actual"]
).abs()

cluster_error_rows_df["signed_error"] = (
    cluster_error_rows_df["prediction_final"]
    - cluster_error_rows_df["actual"]
)

cluster_error_summary_df = (
    cluster_error_rows_df
    .groupby(
        [
            "metric",
            "horizon",
            "cluster_label",
            "canonical_cluster_name",
        ],
        observed=True,
        as_index=False,
    )
    .agg(
        rows=("actual", "size"),
        absolute_error_sum=("absolute_error", "sum"),
        benchmark_absolute_error_sum=("benchmark_absolute_error", "sum"),
        observed_abs_sum=("actual", lambda s: s.abs().sum()),
        bias=("signed_error", "mean"),
    )
)

cluster_error_summary_df["relative_mae_pct"] = np.where(
    cluster_error_summary_df["observed_abs_sum"].gt(0),
    100
    * cluster_error_summary_df["absolute_error_sum"]
    / cluster_error_summary_df["observed_abs_sum"],
    np.nan,
)

cluster_error_summary_df["benchmark_skill_pct"] = np.where(
    cluster_error_summary_df["benchmark_absolute_error_sum"].gt(0),
    100
    * (
        cluster_error_summary_df["benchmark_absolute_error_sum"]
        - cluster_error_summary_df["absolute_error_sum"]
    )
    / cluster_error_summary_df["benchmark_absolute_error_sum"],
    np.nan,
)

# Normalize each environment against that Target × Horizon job's overall
# Relative MAE. This makes "hard" environments comparable across targets whose
# native units and normal error scales differ substantially.
cluster_error_summary_df = cluster_error_summary_df.merge(
    holdout_scorecard_df[
        ["metric", "horizon", "relative_mae_pct"]
    ].rename(
        columns={
            "relative_mae_pct": "job_relative_mae_pct",
        }
    ),
    on=["metric", "horizon"],
    how="left",
    validate="many_to_one",
)

cluster_error_summary_df["error_index"] = (
    100
    * cluster_error_summary_df["relative_mae_pct"]
    / cluster_error_summary_df["job_relative_mae_pct"]
)

cluster_error_summary_df = add_job_labels(cluster_error_summary_df)

# Keep cluster order tied to the canonical numeric labels rather than sorting
# the descriptive names alphabetically.
cluster_order = (
    post_cp_cluster_df[
        ["cluster_label", "canonical_cluster_name"]
    ]
    .drop_duplicates()
    .sort_values("cluster_label")["canonical_cluster_name"]
    .tolist()
)

hardest_cluster_df = (
    cluster_error_summary_df
    .sort_values(
        ["metric", "horizon", "error_index"],
        ascending=[True, True, False],
    )
    .groupby(["metric", "horizon"], observed=True, as_index=False)
    .first()
)

hardest_cluster_display_df = (
    hardest_cluster_df[
        [
            "Target",
            "horizon",
            "canonical_cluster_name",
            "rows",
            "relative_mae_pct",
            "error_index",
            "benchmark_skill_pct",
            "bias",
        ]
    ]
    .rename(
        columns={
            "horizon": "Horizon",
            "canonical_cluster_name": "Hardest mobility environment",
            "rows": "Rows",
            "relative_mae_pct": "Relative MAE (%)",
            "error_index": "Error index",
            "benchmark_skill_pct": "Benchmark skill (%)",
            "bias": "Bias",
        }
    )
    .sort_values(["Target", "Horizon"])
)

display(hardest_cluster_display_df.round(3))


if RUN_VISUALS:
    cluster_heatmap_df = (
        cluster_error_summary_df.pivot(
            index="Job",
            columns="canonical_cluster_name",
            values="error_index",
        )
        .reindex(columns=cluster_order)
    )

    fig = px.imshow(
        cluster_heatmap_df,
        aspect="auto",
        color_continuous_scale=BRAND_SEQUENTIAL_HEATMAP_SCALE,
        labels={
            "x": "Mobility environment",
            "y": "Forecasting job",
            "color": "Error index",
        },
    )

    fig.update_layout(
        title={
            "text": (
                "Which mobility environments are hardest to forecast?"
                "<br><sup>"
                "100 = each job's overall holdout error; "
                "values above 100 indicate a harder-than-usual environment."
                "</sup>"
            ),
            "x": 0.02,
        },
        height=650,
        margin={"l": 140, "r": 40, "t": 100, "b": 150},
    )

    fig.update_xaxes(tickangle=-30)

    apply_branding(fig)
    fig.show()

,Target,Horizon,Hardest mobility environment,Rows,Relative MAE (%),Error index,Benchmark skill (%),Bias
0,FHVHV average speed,1,Long-Trip Fast-Mobility Zones,7772,10.506,180.063,28.701,-0.001
1,FHVHV average speed,2,Long-Trip Fast-Mobility Zones,7772,10.673,171.998,27.574,-0.025
2,FHVHV average speed,5,Long-Trip Fast-Mobility Zones,7772,10.784,169.315,26.814,-0.041
3,FHVHV trips,1,Staten Island Fast-Mobility,6391,15.609,147.877,19.117,-0.941
4,FHVHV trips,2,Staten Island Fast-Mobility,6391,16.358,135.832,15.240,-0.971
5,FHVHV trips,5,Staten Island Fast-Mobility,6391,18.883,155.937,2.156,-10.609
6,Subway ridership,1,Long-Trip Fast-Mobility Zones,1265,14.437,163.297,22.416,20.370
7,Subway ridership,2,Long-Trip Fast-Mobility Zones,1265,19.236,172.837,-3.378,93.377
8,Subway ridership,5,Long-Trip Fast-Mobility Zones,1265,14.677,148.693,21.124,56.301
9,Taxi average speed,1,Lower-Transit Neighborhoods,15274,19.248,120.199,29.117,-0.209


**Findings.** Forecast difficulty also has a clear **mobility-environment structure**, and the hardest environment depends strongly on the target being forecast.

The most extreme result is **Staten Island Fast-Mobility**, which is the hardest environment for every Taxi- and FHVHV-trip horizon. The effect is especially severe for Taxi trips: Relative MAE reaches roughly **389%–627%**, with error indices of about **2,564–4,116**. In other words, Taxi-trip error in this environment is roughly **26 to 41 times the job-wide level**. The champion also loses decisively to the weekly benchmark there, with benchmark skill between approximately **−156% and −313%**. FHVHV trips show the same directional weakness, but at a much smaller scale, with error indices around **136–156**.

A different environment is difficult for FHVHV speed and Subway. **Long-Trip Fast-Mobility Zones** are the hardest cluster at all three horizons for both targets. FHVHV speed remains better than the benchmark despite elevated error, but Subway h=2 again stands out: its Relative MAE rises to **19.2%**, its error index reaches about **173**, and benchmark skill becomes slightly negative at **−3.4%**.

Taxi speed shows the mildest cluster concentration. **Lower-Transit Neighborhoods** are hardest at all three horizons, but error is only about **19%–20% above the job-wide level**, and the champion continues to beat the benchmark comfortably.

The cluster view therefore adds information beyond conventional geography. **Staten Island Fast-Mobility is a highly concentrated trip-demand failure environment, while Long-Trip Fast-Mobility Zones expose a separate cross-mode weakness affecting FHVHV speed and Subway.** Mobility environment is useful explanatory context, but the clusters themselves do not establish a forecast-time reliability rule.

### 4.6.2.3.3 Treat Zero Trips as Real Outcomes

Taxi and FHVHV trip counts come from transaction records, so **zero means zero trips occurred**. The forecasting question is not whether those rows are missing; it is whether the model should have learned that some Zone × Time surfaces frequently remain at zero.

For true zero-event rows, percentage error is not meaningful because the observed denominator is zero. We instead inspect native-unit error, signed bias, the raw model output, the nonnegative final output, and the weekly benchmark.

This also lets us distinguish two very different explanations for a weak surface such as Freshkills Park: the series may have been too low-information to be a useful forecasting target, or the global regressor may simply have failed to learn its near-zero behavior.


**4.6.2.3.3A — Compare real zero-event periods with nonzero trip periods.**


In [15]:
# ---------------------------------------------------------------------
# 4.6.2.3.3A — Compare real zero-event periods with nonzero trip periods
# ---------------------------------------------------------------------

trip_error_df = diagnostic_surface_df.loc[
    diagnostic_surface_df["is_transaction_trip_target"]
]

zero_event_df = (
    trip_error_df.groupby(
        ["metric", "horizon", "actual_zero_event"],
        observed=True,
        dropna=False,
    )
    .agg(
        rows=("actual", "size"),
        mean_actual=("actual", "mean"),
        mean_prediction_raw=("prediction_raw", "mean"),
        mean_prediction_final=("prediction_final", "mean"),
        mean_benchmark_prediction=("benchmark_prediction", "mean"),
        raw_nonpositive_share=("raw_prediction_nonpositive", "mean"),
        final_zero_share=("prediction_final", lambda s: s.eq(0).mean()),
        champion_mae=("absolute_error", "mean"),
        benchmark_mae=("benchmark_absolute_error", "mean"),
        bias=("signed_error", "mean"),
    )
    .reset_index()
)

zero_event_df["raw_nonpositive_pct"] = (
    100 * zero_event_df["raw_nonpositive_share"]
)
zero_event_df["final_zero_pct"] = (
    100 * zero_event_df["final_zero_share"]
)
zero_event_df["job_row_share_pct"] = (
    100
    * zero_event_df["rows"]
    / zero_event_df.groupby(
        ["metric", "horizon"],
        observed=True,
    )["rows"].transform("sum")
)

zero_event_display_df = add_job_labels(
    zero_event_df
)

zero_event_display_df["Observed state"] = np.where(
    zero_event_display_df["actual_zero_event"],
    "True zero-event period",
    "At least one observed trip",
)

zero_event_display_df = (
    zero_event_display_df[
        [
            "Target",
            "horizon",
            "Observed state",
            "rows",
            "job_row_share_pct",
            "mean_actual",
            "mean_prediction_raw",
            "mean_prediction_final",
            "mean_benchmark_prediction",
            "raw_nonpositive_pct",
            "final_zero_pct",
            "champion_mae",
            "benchmark_mae",
            "bias",
        ]
    ]
    .rename(
        columns={
            "horizon": "Horizon",
            "rows": "Rows",
            "job_row_share_pct": "Job rows (%)",
            "mean_actual": "Mean actual",
            "mean_prediction_raw": "Mean raw forecast",
            "mean_prediction_final": "Mean final forecast",
            "mean_benchmark_prediction": "Mean benchmark",
            "raw_nonpositive_pct": "Raw forecast ≤ 0 (%)",
            "final_zero_pct": "Final forecast = 0 (%)",
            "champion_mae": "Champion MAE",
            "benchmark_mae": "Benchmark MAE",
            "bias": "Bias",
        }
    )
)

display(zero_event_display_df.round(3))


,Target,Horizon,Observed state,Rows,Job rows (%),Mean actual,Mean raw forecast,Mean final forecast,Mean benchmark,Raw forecast ≤ 0 (%),Final forecast = 0 (%),Champion MAE,Benchmark MAE,Bias
0,FHVHV trips,1,At least one observed trip,110041,99.586,548.156,544.213,544.240,544.316,0.541,0.541,57.856,92.823,-3.916
1,FHVHV trips,1,True zero-event period,458,0.414,0.000,-0.180,1.279,0.279,55.022,55.022,1.279,0.279,1.279
2,FHVHV trips,2,At least one observed trip,110041,99.586,548.156,545.062,545.082,544.316,0.475,0.475,65.999,92.823,-3.074
3,FHVHV trips,2,True zero-event period,458,0.414,0.000,2.020,3.078,0.279,26.201,26.201,3.078,0.279,3.078
4,FHVHV trips,5,At least one observed trip,110041,99.586,548.156,530.472,530.696,544.316,3.082,3.082,66.376,92.823,-17.460
5,FHVHV trips,5,True zero-event period,458,0.414,0.000,-8.359,0.269,0.279,96.070,96.070,0.269,0.279,0.269
6,Taxi trips,1,At least one observed trip,93705,84.802,112.450,114.956,114.965,111.169,0.037,0.037,16.235,24.003,2.515
7,Taxi trips,1,True zero-event period,16794,15.198,0.000,1.496,1.498,0.835,0.464,0.464,1.498,0.835,1.498
8,Taxi trips,2,At least one observed trip,93705,84.802,112.450,114.965,114.966,111.169,0.014,0.014,17.505,24.003,2.516
9,Taxi trips,2,True zero-event period,16794,15.198,0.000,1.619,1.644,0.835,0.435,0.435,1.644,0.835,1.644


**Findings.** Genuine zero-trip periods are primarily a **Taxi forecasting problem**. Taxi trips are exactly zero on **15.2%** of holdout rows, compared with only **0.4%** for FHVHV trips.

When Taxi demand is actually zero, the champion usually still predicts a small positive count. Its mean final forecast is roughly **1.2–1.6 trips**, and it predicts an exact zero on only about **0.4%–1.7%** of these rows. The simple weekly benchmark is much more naturally zero-sensitive, averaging only **0.835 trips**, and it performs better on the zero-event cases.

This is not mainly caused by the nonnegative output rule. The Taxi model's **raw forecasts are already positive almost all of the time** when the actual count is zero, so clipping negative predictions to zero cannot explain the error.

FHVHV behaves differently because true zeros are rare and the pattern changes by horizon. At h=5, for example, the raw model is already nonpositive on **96.1%** of zero-event rows and performs slightly better than the benchmark. Zero-event weakness is therefore not universal across trip forecasting; it is strongest and most persistent for Taxi.

**4.6.2.3.3B — Quantify the Freshkills Park Taxi h=1 failure directly.**


In [16]:
# ---------------------------------------------------------------------
# 4.6.2.3.3B — Quantify Freshkills Park Taxi h=1
# ---------------------------------------------------------------------

freshkills_mask = (
    diagnostic_surface_df["metric"].eq("taxi_trip_count")
    & diagnostic_surface_df["horizon"].eq(1)
    & diagnostic_surface_df["zone"].eq(FRESHKILLS_ZONE_NAME)
)

freshkills_rows_df = (
    diagnostic_surface_df.loc[freshkills_mask]
)

if freshkills_rows_df.empty:
    raise RuntimeError(
        f"No Taxi h=1 diagnostic rows were found for {FRESHKILLS_ZONE_NAME}."
    )

zero_actual_rows = freshkills_rows_df.loc[
    freshkills_rows_df["actual_zero_event"].fillna(False)
]

observed_abs_sum = freshkills_rows_df["observed_abs"].sum()
champion_mae = freshkills_rows_df["absolute_error"].mean()
benchmark_mae = freshkills_rows_df["benchmark_absolute_error"].mean()

freshkills_summary_df = pd.DataFrame(
    [{
        "Rows": len(freshkills_rows_df),
        "Actual zero-event rows (%)":
            100 * freshkills_rows_df["actual_zero_event"].mean(),
        "Origin zero-event rows (%)":
            100 * freshkills_rows_df["origin_zero_event"].mean(),
        "Prior 28-day all-zero history (%)":
            100 * freshkills_rows_df["history_28d_all_zero_event"].mean(),
        "Complete prior 28-day history (%)":
            100 * freshkills_rows_df["history_28d_complete"].mean(),
        "Mean actual":
            freshkills_rows_df["actual"].mean(),
        "Median actual":
            freshkills_rows_df["actual"].median(),
        "Mean raw forecast":
            freshkills_rows_df["prediction_raw"].mean(),
        "Mean final forecast":
            freshkills_rows_df["prediction_final"].mean(),
        "Mean raw forecast when actual = 0":
            zero_actual_rows["prediction_raw"].mean(),
        "Mean final forecast when actual = 0":
            zero_actual_rows["prediction_final"].mean(),
        "Raw forecast ≤ 0 (%)":
            100 * freshkills_rows_df["raw_prediction_nonpositive"].mean(),
        "Final forecast = 0 (%)":
            100 * freshkills_rows_df["prediction_final"].eq(0).mean(),
        "Weekly benchmark = 0 (%)":
            100 * freshkills_rows_df["benchmark_prediction"].eq(0).mean(),
        "Champion MAE":
            champion_mae,
        "Benchmark MAE":
            benchmark_mae,
        "Bias":
            freshkills_rows_df["signed_error"].mean(),
        "Relative MAE (%)":
            (
                100
                * freshkills_rows_df["absolute_error"].sum()
                / observed_abs_sum
                if observed_abs_sum > 0
                else np.nan
            ),
        "Benchmark skill (%)":
            (
                100 * (benchmark_mae - champion_mae) / benchmark_mae
                if benchmark_mae > 0
                else np.nan
            ),
    }]
)

display(
    freshkills_summary_df
    .T
    .rename(columns={0: "Value"})
    .round(3)
)


,Value
Rows,381.000
Actual zero-event rows (%),99.738
Origin zero-event rows (%),93.701
Prior 28-day all-zero history (%),79.790
Complete prior 28-day history (%),30.184
Mean actual,0.003
Median actual,0.000
Mean raw forecast,1.427
Mean final forecast,1.435
Mean raw forecast when actual = 0,1.426


**Findings.** Freshkills Park is an extreme case where **both forecast support and model behavior are questionable**. Across 381 Taxi h=1 forecasts, the actual trip count is zero **99.7%** of the time, the forecast-origin count is zero **93.7%** of the time, and all observed values in the recent same-Daypart history are zero on **79.8%** of rows.

At the same time, only **30.2%** of rows have a complete 28-day same-Daypart history, so this is not simply a fully observed low-demand series. Historical support is frequently limited.

The Tree champion nevertheless produces an average raw forecast of about **1.43 trips**, including **1.43 trips when the actual outcome is zero**, and its raw prediction is nonpositive only **3.2%** of the time. The weekly benchmark, by contrast, predicts zero on **99.7%** of these rows. Champion MAE is therefore **1.432 trips** versus only **0.005** for the benchmark.

Freshkills looks like a **combination of weak forecast support and a model that tends to regress toward a small positive count**, rather than purely one problem or the other. It is therefore a strong candidate for testing whether some forecasts should be reliability-qualified before downstream use.

### 4.6.2.3.4 Relate Error to Legal Pre-Origin History

A difficult forecast can arise from several different historical conditions:

- the target has been consistently near zero;
- the target has meaningful activity but is highly volatile;
- the recent history is incompletely observed;
- the target has moved away from its usual level.

These conditions were available at the forecast origin, so they are legitimate candidates for the later forecastability analysis.

We first keep **zero-heavy observed history** separate from **incomplete history**. Then we ask whether error changes systematically as recent historical level or volatility increases.


**4.6.2.3.4A — Separate all-zero history from incomplete history.**


In [17]:
# ---------------------------------------------------------------------
# 4.6.2.3.4A — Separate zero activity from incomplete history
# ---------------------------------------------------------------------

zero_history_rows = trip_error_df.loc[
    trip_error_df["history_28d_all_zero_event"].notna()
]

history_zero_df = summarize_errors(
    zero_history_rows,
    ["metric", "horizon", "history_28d_all_zero_event"],
)
history_zero_df = attach_job_error_index(
    history_zero_df,
    holdout_scorecard_df,
)

history_zero_df["History state"] = np.where(
    history_zero_df["history_28d_all_zero_event"],
    "All observed prior 28-day same-Daypart values were zero",
    "At least one prior 28-day same-Daypart value was nonzero",
)

history_zero_df["job_row_share_pct"] = (
    100
    * history_zero_df["rows"]
    / history_zero_df.groupby(
        ["metric", "horizon"],
        observed=True,
    )["rows"].transform("sum")
)

history_zero_display_df = (
    add_job_labels(history_zero_df)[
        [
            "Target",
            "horizon",
            "History state",
            "rows",
            "job_row_share_pct",
            "relative_mae_pct",
            "error_index",
            "benchmark_skill_pct",
            "bias",
        ]
    ]
    .rename(
        columns={
            "horizon": "Horizon",
            "rows": "Rows",
            "job_row_share_pct": "Job rows (%)",
            "relative_mae_pct": "Relative MAE (%)",
            "error_index": "Error index",
            "benchmark_skill_pct": "Benchmark skill (%)",
            "bias": "Bias",
        }
    )
)

display(history_zero_display_df.round(3))


history_completeness_df = summarize_errors(
    diagnostic_surface_df,
    ["metric", "horizon", "history_28d_complete"],
)
history_completeness_df = attach_job_error_index(
    history_completeness_df,
    holdout_scorecard_df,
)
history_completeness_df["Target type"] = (
    history_completeness_df["metric"].map(TARGET_TYPES)
)

history_completeness_summary_df = (
    history_completeness_df
    .groupby(
        ["Target type", "history_28d_complete"],
        observed=True,
        as_index=False,
    )
    .agg(
        median_error_index=("error_index", "median"),
        median_relative_mae_pct=("relative_mae_pct", "median"),
        job_groups=("metric", "size"),
    )
)

history_completeness_summary_df["History completeness"] = np.where(
    history_completeness_summary_df["history_28d_complete"],
    "Complete 28-day same-Daypart history",
    "Incomplete 28-day same-Daypart history",
)

display(
    history_completeness_summary_df[
        [
            "Target type",
            "History completeness",
            "job_groups",
            "median_error_index",
            "median_relative_mae_pct",
        ]
    ]
    .rename(
        columns={
            "job_groups": "Job-groups",
            "median_error_index": "Median error index",
            "median_relative_mae_pct": "Median Relative MAE (%)",
        }
    )
    .round(3)
)


,Target,Horizon,History state,Rows,Job rows (%),Relative MAE (%),Error index,Benchmark skill (%),Bias
0,FHVHV trips,1,At least one prior 28-day same-Daypart value w...,110073,99.614,10.555,99.993,37.669,-3.914
1,FHVHV trips,1,All observed prior 28-day same-Daypart values ...,426,0.386,NaN,NaN,NaN,0.995
2,FHVHV trips,2,At least one prior 28-day same-Daypart value w...,110073,99.614,12.041,99.984,28.896,-3.070
3,FHVHV trips,2,All observed prior 28-day same-Daypart values ...,426,0.386,NaN,NaN,NaN,2.730
4,FHVHV trips,5,At least one prior 28-day same-Daypart value w...,110073,99.614,12.109,100.000,28.491,-17.454
5,FHVHV trips,5,All observed prior 28-day same-Daypart values ...,426,0.386,NaN,NaN,NaN,0.011
6,Taxi trips,1,At least one prior 28-day same-Daypart value w...,108120,97.847,14.655,99.856,31.762,2.392
7,Taxi trips,1,All observed prior 28-day same-Daypart values ...,2379,2.153,"2,129.785","14,511.637","-1,156.334",0.922
8,Taxi trips,2,At least one prior 28-day same-Daypart value w...,108111,97.839,15.806,99.857,26.403,2.414
9,Taxi trips,2,All observed prior 28-day same-Daypart values ...,2388,2.161,"1,729.925","10,929.060","-1,050.524",0.988


,Target type,History completeness,Job-groups,Median error index,Median Relative MAE (%)
0,Demand,Incomplete 28-day same-Daypart history,9,536.221,60.468
1,Demand,Complete 28-day same-Daypart history,9,99.795,12.015
2,Speed,Incomplete 28-day same-Daypart history,6,206.575,19.911
3,Speed,Complete 28-day same-Daypart history,6,93.545,10.105


**Findings.** Two different conditions need to remain separate: **genuinely zero recent activity** and **incomplete historical observation**.

For Taxi trips, roughly **2.2%** of holdout rows have recent same-Daypart history in which every observed value is zero. Those rows are extraordinarily difficult: Relative MAE rises to roughly **1,690%–2,130%**, and the weekly benchmark substantially outperforms the champion. FHVHV all-zero histories are much rarer—about **0.4%** of rows—and their percentage error is undefined because the observed scale itself is zero.

Incomplete history is a broader warning sign across the forecasting system. For demand targets, incomplete 28-day same-Daypart history has a median error index of **536**, versus about **100** for complete history; median Relative MAE rises from about **12.0% to 60.5%**. Speed forecasts show the same direction: the median error index rises from about **94 to 207**, while Relative MAE rises from about **10.1% to 19.9%**.

Observed-history completeness is therefore one of the strongest pre-origin forecastability signals found so far. Importantly, this does **not** mean zero activity is missing data: a fully observed sequence of zeros remains genuine zero demand.

**4.6.2.3.4B — Compare forecast error across recent historical level and volatility.**


In [18]:
# ---------------------------------------------------------------------
# 4.6.2.3.4B — Compare error across prior level and volatility
# ---------------------------------------------------------------------

history_level_df = build_quantile_error_summary(
    diagnostic_surface_df,
    "same_daypart_roll_mean_28d",
    holdout_scorecard_df,
)

history_volatility_df = build_quantile_error_summary(
    diagnostic_surface_df,
    "same_daypart_roll_std_28d",
    holdout_scorecard_df,
)

if WRITE_INTERMEDIATES:
    atomic_write_parquet(
        history_level_df,
        ERROR_BY_HISTORY_LEVEL_PATH,
    )
    atomic_write_parquet(
        history_volatility_df,
        ERROR_BY_HISTORY_VOLATILITY_PATH,
    )


# Summarize demand and speed separately. Their native error scales differ, but
# the within-job error index lets us ask whether the same historical condition
# tends to make each broad target type easier or harder than usual.
quantile_summary_parts = []

for diagnostic_name, frame in [
    ("Prior 28-day same-Daypart level", history_level_df),
    ("Prior 28-day same-Daypart volatility", history_volatility_df),
]:
    temp = frame.copy()
    temp["Target type"] = temp["metric"].map(TARGET_TYPES)

    summary = (
        temp.groupby(
            ["Target type", "quantile"],
            observed=True,
            as_index=False,
        )
        .agg(
            median_error_index=("error_index", "median"),
            median_relative_mae_pct=("relative_mae_pct", "median"),
        )
    )

    summary.insert(
        0,
        "Diagnostic",
        diagnostic_name,
    )
    quantile_summary_parts.append(summary)

history_quantile_summary_df = pd.concat(
    quantile_summary_parts,
    ignore_index=True,
)

display(history_quantile_summary_df.round(3))


if RUN_VISUALS:
    fig = make_quantile_pattern_figure(
        history_level_df,
        history_volatility_df,
    )
    fig.show()


,Diagnostic,Target type,quantile,median_error_index,median_relative_mae_pct
0,Prior 28-day same-Daypart level,Demand,1,176.821,18.268
1,Prior 28-day same-Daypart level,Demand,2,129.511,12.784
2,Prior 28-day same-Daypart level,Demand,3,116.240,11.938
3,Prior 28-day same-Daypart level,Demand,4,98.063,11.451
4,Prior 28-day same-Daypart level,Demand,5,89.752,11.330
5,Prior 28-day same-Daypart level,Speed,1,86.278,9.639
6,Prior 28-day same-Daypart level,Speed,2,97.049,10.887
7,Prior 28-day same-Daypart level,Speed,3,98.154,10.560
8,Prior 28-day same-Daypart level,Speed,4,100.215,10.964
9,Prior 28-day same-Daypart level,Speed,5,108.241,11.537


**Findings.** Recent history matters differently for demand and speed forecasts.

For **demand**, low recent activity is the difficult regime. The lowest activity fifth has a median error index of about **177**, while the highest activity fifth falls below the job-wide average at about **90**. Low recent volatility shows a similar pattern: the lowest-volatility fifth has an error index of about **162**, compared with about **87** for the highest-volatility fifth. For demand, low volatility often accompanies very quiet or near-zero series, so “stable” does not necessarily mean “easy to forecast.”

For **speed**, recent level matters much less, but volatility has a clear relationship with error. The speed error index rises steadily from about **66** in the lowest-volatility fifth to **137** in the highest-volatility fifth.

There is therefore no single universal rule such as “high volatility means unreliable.” Candidate reliability rules should respect the type of target: **low level and near-zero behavior are more informative for demand, while high recent volatility is more informative for speed.**

### 4.6.2.3.5 Look for Shared Hard Weeks and the Subway h=2 Exception

Some failures may belong to a particular series or neighborhood. Others may occur during weeks when the broader mobility system became harder to predict.

We therefore normalize weekly error within each Target × Horizon job before comparing weeks. A broad increase across many jobs suggests a shared difficult period; an isolated spike points toward a more target-specific problem.

We then isolate **Subway h=2**, because 4.6.1 already showed that it is unusual in two independent ways: weaker geographic reliability and the only clear champion-versus-runner-up holdout reversal. The goal is to understand that fragility, not to replace its frozen champion.


**4.6.2.3.5A — Identify unusually difficult weeks across the champion system.**


In [19]:
# ---------------------------------------------------------------------
# 4.6.2.3.5A — Identify shared difficult weeks
# ---------------------------------------------------------------------

weekly_error_df = summarize_errors(
    diagnostic_surface_df,
    ["metric", "horizon", "week_start"],
)

weekly_error_df = attach_job_error_index(
    weekly_error_df,
    holdout_scorecard_df,
)

if WRITE_INTERMEDIATES:
    atomic_write_parquet(
        weekly_error_df,
        ERROR_BY_WEEK_PATH,
    )


# Each job first gets its own normalized weekly error. We then summarize across
# the 15 jobs so a citywide difficult week is not dominated by the native scale
# of Taxi trips, Subway riders, or speed.
weekly_system_summary_df = (
    weekly_error_df.groupby(
        "week_start",
        observed=True,
        as_index=False,
    )
    .agg(
        median_error_index=("error_index", "median"),
        q25_error_index=("error_index", lambda s: s.quantile(0.25)),
        q75_error_index=("error_index", lambda s: s.quantile(0.75)),
        max_error_index=("error_index", "max"),
    )
    .sort_values("week_start")
)

hardest_weeks_df = (
    weekly_system_summary_df
    .nlargest(8, "median_error_index")
    .copy()
)

display(hardest_weeks_df.round(3))


if RUN_VISUALS:
    fig = go.Figure()

    # The shaded band shows the middle half of forecasting jobs. A week is more
    # plausibly a shared system shock when the entire band rises, not just one job.
    fig.add_trace(
        go.Scatter(
            x=weekly_system_summary_df["week_start"],
            y=weekly_system_summary_df["q75_error_index"],
            mode="lines",
            line={
                "color": BRAND_COLOR_SEQUENCE[1],
                "width": 0,
            },
            showlegend=False,
            hoverinfo="skip",
        )
    )

    fig.add_trace(
        go.Scatter(
            x=weekly_system_summary_df["week_start"],
            y=weekly_system_summary_df["q25_error_index"],
            mode="lines",
            fill="tonexty",
            line={
                "color": BRAND_COLOR_SEQUENCE[1],
                "width": 0,
            },
            opacity=0.18,
            name="Middle 50% of jobs",
            hoverinfo="skip",
        )
    )

    fig.add_trace(
        go.Scatter(
            x=weekly_system_summary_df["week_start"],
            y=weekly_system_summary_df["median_error_index"],
            mode="lines+markers",
            name="Median job",
            line={
                "color": BRAND_COLOR_SEQUENCE[0],
                "width": 2.5,
            },
            marker={"size": 6},
            hovertemplate=(
                "Week of %{x|%Y-%m-%d}<br>"
                "Median error index=%{y:.1f}"
                "<extra></extra>"
            ),
        )
    )

    fig.add_hline(
        y=100,
        line_dash="dot",
    )

    fig.update_layout(
        title={
            "text": (
                "Did the forecasting system share unusually difficult weeks?"
                "<br><sup>"
                "Error is normalized within each Target × Horizon job "
                "before weeks are compared."
                "</sup>"
            ),
            "x": 0.02,
        },
        xaxis_title="Calendar week",
        yaxis_title="Error index",
        height=500,
        margin={
            "l": 70,
            "r": 30,
            "t": 95,
            "b": 60,
        },
    )

    apply_branding(fig)
    fig.show()


,week_start,median_error_index,q25_error_index,q75_error_index,max_error_index
7,2026-02-23,150.170,118.520,189.705,200.860
2,2026-01-19,133.063,123.258,148.805,163.638
3,2026-01-26,124.514,114.495,137.834,159.907
6,2026-02-16,113.798,106.433,119.272,136.252
0,2026-01-05,92.204,82.968,100.130,115.255
5,2026-02-09,89.415,75.785,90.330,97.451
4,2026-02-02,88.307,84.378,102.201,105.216
10,2026-03-16,87.639,83.247,89.337,102.360


**Findings.** Several difficult weeks affected much of the forecasting system at the same time rather than only one isolated target. The week of **February 23, 2026** is the clearest example, with a median error index of about **150** across the 15 forecasting jobs.

The weeks of **January 19, January 26, and February 16** are also broadly difficult. In all four cases, even the 25th-percentile job has an error index above 100, meaning the weakness extends well beyond one or two unusually bad forecasts.

This suggests that some forecast failures come from **temporary system-wide conditions** rather than from a particular model, target, or neighborhood. These weeks are therefore good candidates for later failure-case analysis using external context such as weather, mobility stress, major disruptions, or unusual events. The weekly error pattern alone does not tell us what caused them.

**4.6.2.3.5B — Compare Subway h=1, h=2, and h=5 directly.**


In [20]:
# ---------------------------------------------------------------------
# 4.6.2.3.5B — Diagnose the Subway h=2 reliability exception
# ---------------------------------------------------------------------

subway_job_df = job_fragility_df.loc[
    job_fragility_df["metric"].eq("subway_ridership")
].copy()

subway_temporal_df = hardest_group_per_job(
    temporal_error_df.loc[
        temporal_error_df["metric"].eq("subway_ridership")
    ]
)


# Combine overall accuracy, geographic breadth, runner-up robustness, and the
# hardest temporal bucket. These dimensions describe different kinds of
# weakness, so seeing them together helps determine whether h=2 is broadly
# fragile or merely difficult under one particular time condition.
subway_fragility_df = (
    subway_job_df[
        [
            "metric",
            "horizon",
            "champion_family",
            "relative_mae_pct",
            "benchmark_skill_pct",
            "positive_skill_zone_pct",
            "median_zone_skill_pct",
            "p10_zone_skill_pct",
            "runner_up_family",
            "comparison_status",
        ]
    ]
    .merge(
        subway_temporal_df[
            [
                "metric",
                "horizon",
                "Temporal bucket",
                "error_index",
                "benchmark_skill_pct",
            ]
        ]
        .rename(
            columns={
                "Temporal bucket": "Hardest temporal bucket",
                "error_index": "Hardest temporal error index",
                "benchmark_skill_pct":
                    "Hardest temporal benchmark skill (%)",
            }
        ),
        on=["metric", "horizon"],
        how="left",
        validate="one_to_one",
    )
    .sort_values("horizon")
)


subway_fragility_display_df = (
    subway_fragility_df.rename(
        columns={
            "horizon": "Horizon",
            "champion_family": "Champion",
            "relative_mae_pct": "Relative MAE (%)",
            "benchmark_skill_pct": "Overall benchmark skill (%)",
            "positive_skill_zone_pct":
                "Supported zones beating benchmark (%)",
            "median_zone_skill_pct": "Median zone skill (%)",
            "p10_zone_skill_pct": "10th-percentile zone skill (%)",
            "runner_up_family": "Runner-up",
            "comparison_status": "Runner-up audit",
        }
    )
)

subway_fragility_display_df["Champion"] = (
    subway_fragility_display_df["Champion"]
    .map(FAMILY_LABELS)
)

subway_fragility_display_df["Runner-up"] = (
    subway_fragility_display_df["Runner-up"]
    .map(FAMILY_LABELS)
)

display(
    subway_fragility_display_df[
        [
            "Horizon",
            "Champion",
            "Relative MAE (%)",
            "Overall benchmark skill (%)",
            "Supported zones beating benchmark (%)",
            "Median zone skill (%)",
            "10th-percentile zone skill (%)",
            "Runner-up",
            "Runner-up audit",
            "Hardest temporal bucket",
            "Hardest temporal error index",
            "Hardest temporal benchmark skill (%)",
        ]
    ].round(3)
)


if RUN_VISUALS:
    subway_temporal_plot_df = temporal_error_df.loc[
        temporal_error_df["metric"].eq("subway_ridership")
    ].copy()

    subway_temporal_plot_df["Horizon"] = (
        subway_temporal_plot_df["horizon"]
        .astype(int)
        .astype(str)
    )

    subway_temporal_plot_df["Temporal bucket"] = pd.Categorical(
        subway_temporal_plot_df["Temporal bucket"],
        categories=[
            TEMPORAL_BUCKET_LABELS[bucket]
            for bucket in TEMPORAL_BUCKET_ORDER
        ],
        ordered=True,
    )

    subway_temporal_plot_df = subway_temporal_plot_df.sort_values(
        ["Temporal bucket", "horizon"]
    )

    fig = px.bar(
        subway_temporal_plot_df,
        x="Temporal bucket",
        y="error_index",
        color="Horizon",
        barmode="group",
        color_discrete_sequence=BRAND_COLOR_SEQUENCE,
        custom_data=[
            "relative_mae_pct",
            "benchmark_skill_pct",
            "rows",
        ],
    )

    fig.add_hline(
        y=100,
        line_dash="dot",
    )

    fig.update_traces(
        hovertemplate=(
            "%{x}<br>"
            "h=%{fullData.name}<br>"
            "Error index=%{y:.1f}<br>"
            "Relative MAE=%{customdata[0]:.2f}%<br>"
            "Benchmark skill=%{customdata[1]:.1f}%<br>"
            "Rows=%{customdata[2]:,}"
            "<extra></extra>"
        )
    )

    fig.update_layout(
        title={
            "text": (
                "When does Subway h=2 become unusually difficult?"
                "<br><sup>"
                "Each temporal bucket is normalized against its own "
                "Subway horizon's overall holdout error."
                "</sup>"
            ),
            "x": 0.02,
        },
        xaxis_title="Temporal bucket",
        yaxis_title="Error index",
        height=520,
        margin={
            "l": 70,
            "r": 30,
            "t": 95,
            "b": 120,
        },
    )

    apply_branding(fig)
    fig.show()


,Horizon,Champion,Relative MAE (%),Overall benchmark skill (%),Supported zones beating benchmark (%),Median zone skill (%),10th-percentile zone skill (%),Runner-up,Runner-up audit,Hardest temporal bucket,Hardest temporal error index,Hardest temporal benchmark skill (%)
0,1,Tree,8.841,35.836,95.425,33.806,15.940,Transformer,uncertain_difference,Weekend · Overnight,215.767,-10.440
1,2,Transformer,11.130,19.223,70.588,12.886,-33.942,Tree,runner_up_clear_win,Weekend · Overnight,225.914,-45.574
2,5,Tree,9.871,28.361,96.078,27.983,15.329,Transformer,champion_clear_win,Weekend · Overnight,193.797,-10.751


**Findings.** Subway h=2 stands out as a distinct reliability problem rather than simply a harder version of the other horizons. Its Relative MAE is **11.1%**, compared with **8.8%** at h=1 and **9.9%** at h=5, while benchmark skill falls to **19.2%**.

Its weakness is also geographic: only **70.6%** of support-qualified zones beat the benchmark, and the weakest geographic tail performs substantially worse than the benchmark. The Tree runner-up also clearly beats the frozen Transformer on the unseen holdout, making Subway h=2 the only clear champion-versus-runner-up reversal from 4.6.1.

Weekend overnight is the hardest temporal bucket for all three Subway horizons, but the problem is much more severe at h=2. Benchmark skill falls to **−45.6%** in that period, compared with roughly **−10%** at h=1 and h=5.

Subway h=2 is therefore **broadly more fragile across several independent measures, with weekend overnight making that weakness even worse**. This should remain a separate model/horizon failure category in the next stage rather than being folded into the general low-support or low-demand cases.

### 4.6.2.3.6 Quantify How Much Each Failure Mechanism Matters

The earlier diagnostics showed several conditions where forecast error becomes unusually large. Severity alone, however, does not tell us which problems matter most overall.

This subsection first identifies **where error actually begins to separate** for historical support, demand level, and speed volatility. It then measures how often each resulting failure condition occurs and how much of the forecasting system's error burden is concentrated there.

Because Taxi trips, Subway riders, and average speed use different units, systemwide error burden gives each Target × Horizon job equal weight before combining them.

**4.6.2.3.6A — Calibrate the diagnostic failure thresholds from the observed error surface.**

Rather than declaring that 90% history is “weak” or that the lowest 20% of activity is automatically “low,” we test a range of plausible boundaries and ask where forecast error actually separates.

For each candidate boundary, we compare MAE on the **risk side** with MAE on the remaining forecasts from the same Target × Horizon. A candidate must have enough observations across most applicable jobs and show higher error in at least two-thirds of the supported jobs.

If several thresholds perform almost equally well, we keep the **more conservative** definition. These are retrospective diagnostic thresholds for describing the holdout—not forecast-time abstention rules.

In [21]:
# ---------------------------------------------------------------------
# 4.6.2.3.6A — Prepare the diagnostic surface and calibrate thresholds
# ---------------------------------------------------------------------

FAILURE_JOB_KEYS = ["metric", "horizon"]

# These guardrails define how much evidence a candidate threshold needs before
# we trust it. They do NOT define what "weak" or "volatile" means; the actual
# failure thresholds are selected from the error patterns below.
CALIBRATION_MIN_RISK_SHARE = 0.05
CALIBRATION_MIN_SAFE_SHARE = 0.20
CALIBRATION_MIN_SUPPORTED_JOB_SHARE = 0.80
CALIBRATION_MIN_WORSE_JOB_SHARE = 2 / 3
CALIBRATION_NEAR_BEST_FRACTION = 0.95

# "Severe error" is used later only to ask how much of each job's worst tail a
# mechanism captures. It is not one of the failure thresholds calibrated here.
SEVERE_ERROR_QUANTILE = 0.90


# Build one reusable row-level surface so all later accounting uses the same
# predictions, outcomes, benchmark, and leakage-safe historical signals.
failure_burden_df = diagnostic_surface_df.copy()

failure_burden_df["absolute_error"] = (
    failure_burden_df["prediction_final"] - failure_burden_df["actual"]
).abs()

failure_burden_df["benchmark_absolute_error"] = (
    failure_burden_df["benchmark_prediction"] - failure_burden_df["actual"]
).abs()

failure_burden_df["target_date"] = pd.to_datetime(
    failure_burden_df["target_date"]
).dt.normalize()

failure_burden_df["week_start"] = (
    failure_burden_df["target_date"]
    - pd.to_timedelta(failure_burden_df["target_date"].dt.weekday, unit="D")
)

failure_burden_df["target_type"] = np.where(
    failure_burden_df["metric"].str.endswith("_speed"), "speed", "demand"
)

trip_count_metrics = {"taxi_trip_count", "fhvhv_trip_count"}
failure_burden_df["is_trip_count"] = failure_burden_df["metric"].isin(
    trip_count_metrics
)

numeric_signal_columns = [
    "same_daypart_roll_obs_share_28d",
    "same_daypart_roll_mean_28d",
    "same_daypart_roll_std_28d",
    "origin_value",
]

for column in numeric_signal_columns:
    failure_burden_df[column] = pd.to_numeric(
        failure_burden_df[column], errors="coerce"
    )

# Normalizing error within Target × Horizon lets us compare very different
# outcomes without pretending that trips, riders, and mph share native units.
failure_burden_df["job_mae"] = (
    failure_burden_df.groupby(FAILURE_JOB_KEYS, observed=True)["absolute_error"]
    .transform("mean")
)

failure_burden_df["row_error_index"] = np.where(
    failure_burden_df["job_mae"].gt(0),
    100.0 * failure_burden_df["absolute_error"] / failure_burden_df["job_mae"],
    np.nan,
)


def evaluate_diagnostic_candidates_4636(
    frame,
    *,
    signal_column,
    candidates,
    applicable_mask,
    threshold_kind,
    risk_direction,
):
    """
    Compare candidate risk boundaries across the forecasting jobs they apply to.

    History completeness has a common 0–1 scale, so it can use an absolute
    cutoff. Activity and volatility use within-job percentiles because their
    physical scales differ substantially across mobility targets.
    """
    applicable_df = frame.loc[applicable_mask].copy()
    applicable_jobs = applicable_df[FAILURE_JOB_KEYS].drop_duplicates().shape[0]
    candidate_records = []

    for candidate in candidates:
        job_records = []

        for _, job_df in applicable_df.groupby(FAILURE_JOB_KEYS, observed=True):
            values = pd.to_numeric(job_df[signal_column], errors="coerce")
            usable = values.notna() & np.isfinite(
                job_df["absolute_error"].to_numpy(dtype=float)
            )
            local_df = job_df.loc[usable].copy()

            if local_df.empty:
                continue

            values = local_df[signal_column]
            threshold = (
                float(candidate)
                if threshold_kind == "absolute"
                else float(values.quantile(candidate))
            )

            if risk_direction == "low_absolute":
                risk_mask = values.lt(threshold)
            elif risk_direction == "low_quantile":
                # Ties stay together so a large set of genuine zeros is not
                # split artificially just to match a nominal percentile.
                risk_mask = values.le(threshold)
            elif risk_direction == "high_quantile":
                risk_mask = values.ge(threshold)
            else:
                raise ValueError(f"Unknown risk direction: {risk_direction}")

            safe_mask = ~risk_mask
            risk_share, safe_share = float(risk_mask.mean()), float(safe_mask.mean())

            support_ok = (
                risk_share >= CALIBRATION_MIN_RISK_SHARE
                and safe_share >= CALIBRATION_MIN_SAFE_SHARE
            )

            risk_mae = (
                float(local_df.loc[risk_mask, "absolute_error"].mean())
                if risk_mask.any()
                else np.nan
            )
            safe_mae = (
                float(local_df.loc[safe_mask, "absolute_error"].mean())
                if safe_mask.any()
                else np.nan
            )

            job_records.append(
                {
                    "risk_share": risk_share,
                    "support_ok": support_ok,
                    "mae_ratio": (
                        risk_mae / safe_mae
                        if support_ok and safe_mae > 0
                        else np.nan
                    ),
                }
            )

        job_result_df = pd.DataFrame(job_records)
        supported_df = job_result_df.loc[
            job_result_df["support_ok"] & job_result_df["mae_ratio"].notna()
        ]

        supported_jobs = len(supported_df)
        supported_job_share = (
            supported_jobs / applicable_jobs if applicable_jobs else 0.0
        )
        worse_job_share = (
            float(supported_df["mae_ratio"].gt(1.0).mean())
            if supported_jobs
            else np.nan
        )

        candidate_records.append(
            {
                "candidate": float(candidate),
                "applicable_jobs": applicable_jobs,
                "supported_jobs": supported_jobs,
                "supported_job_share": supported_job_share,
                "worse_job_share": worse_job_share,
                "median_risk_share": (
                    float(supported_df["risk_share"].median())
                    if supported_jobs
                    else np.nan
                ),
                "median_mae_ratio": (
                    float(supported_df["mae_ratio"].median())
                    if supported_jobs
                    else np.nan
                ),
            }
        )

    result_df = pd.DataFrame(candidate_records)
    result_df["eligible"] = (
        result_df["supported_job_share"].ge(CALIBRATION_MIN_SUPPORTED_JOB_SHARE)
        & result_df["worse_job_share"].ge(CALIBRATION_MIN_WORSE_JOB_SHARE)
        & result_df["median_mae_ratio"].gt(1.0)
    )

    return result_df


def select_diagnostic_candidate_4636(candidate_df, *, conservative_direction):
    """
    Select a stable boundary without chasing tiny numerical improvements.

    Candidates within 5% of the strongest median separation are treated as
    practically equivalent. The stricter failure definition wins that tie.
    """
    eligible_df = candidate_df.loc[candidate_df["eligible"]].copy()

    if eligible_df.empty:
        return np.nan

    best_ratio = float(eligible_df["median_mae_ratio"].max())
    near_best_df = eligible_df.loc[
        eligible_df["median_mae_ratio"].ge(
            best_ratio * CALIBRATION_NEAR_BEST_FRACTION
        )
    ]

    if conservative_direction == "lower":
        return float(near_best_df["candidate"].min())
    if conservative_direction == "higher":
        return float(near_best_df["candidate"].max())

    raise ValueError(f"Unknown conservative direction: {conservative_direction}")


# History completeness uses its shared physical scale. We scan broadly enough
# to detect whether weakness begins at, for example, 30%, 50%, 70%, or 90%.
history_candidate_values = np.arange(0.20, 0.91, 0.10)

history_candidate_df = evaluate_diagnostic_candidates_4636(
    failure_burden_df,
    signal_column="same_daypart_roll_obs_share_28d",
    candidates=history_candidate_values,
    applicable_mask=failure_burden_df["same_daypart_roll_obs_share_28d"].notna(),
    threshold_kind="absolute",
    risk_direction="low_absolute",
)

DIAGNOSTIC_HISTORY_SHARE_CUTOFF = select_diagnostic_candidate_4636(
    history_candidate_df, conservative_direction="lower"
)


# Demand activity uses job-specific percentiles so Taxi, FHVHV, and Subway are
# each judged relative to their own normal level of activity.
activity_candidates = np.arange(0.05, 0.41, 0.05)

activity_candidate_df = evaluate_diagnostic_candidates_4636(
    failure_burden_df,
    signal_column="same_daypart_roll_mean_28d",
    candidates=activity_candidates,
    applicable_mask=(
        failure_burden_df["target_type"].eq("demand")
        & failure_burden_df["same_daypart_roll_mean_28d"].notna()
    ),
    threshold_kind="quantile",
    risk_direction="low_quantile",
)

DIAGNOSTIC_LOW_ACTIVITY_QUANTILE = select_diagnostic_candidate_4636(
    activity_candidate_df, conservative_direction="lower"
)


# Speed volatility is also job-relative. Here the risky side is the unusually
# volatile upper tail rather than the unusually quiet lower tail.
volatility_candidates = np.arange(0.60, 0.96, 0.05)

volatility_candidate_df = evaluate_diagnostic_candidates_4636(
    failure_burden_df,
    signal_column="same_daypart_roll_std_28d",
    candidates=volatility_candidates,
    applicable_mask=(
        failure_burden_df["target_type"].eq("speed")
        & failure_burden_df["same_daypart_roll_std_28d"].notna()
    ),
    threshold_kind="quantile",
    risk_direction="high_quantile",
)

DIAGNOSTIC_HIGH_VOLATILITY_QUANTILE = select_diagnostic_candidate_4636(
    volatility_candidate_df, conservative_direction="higher"
)


# A shared-shock week must be difficult broadly, not merely disastrous for one
# target. Requiring the 25th-percentile job to exceed 100 means roughly three-
# quarters of observed jobs are performing worse than their own normal level.
job_week_error_df = (
    failure_burden_df.groupby(
        [*FAILURE_JOB_KEYS, "week_start"], observed=True, as_index=False
    )
    .agg(
        week_rows=("absolute_error", "size"),
        week_mae=("absolute_error", "mean"),
        job_mae=("job_mae", "first"),
    )
)

job_week_error_df["week_error_index"] = (
    100.0 * job_week_error_df["week_mae"] / job_week_error_df["job_mae"]
)

expected_jobs = failure_burden_df[FAILURE_JOB_KEYS].drop_duplicates().shape[0]
minimum_week_jobs = int(
    np.ceil(expected_jobs * CALIBRATION_MIN_SUPPORTED_JOB_SHARE)
)

weekly_system_error_df = (
    job_week_error_df.groupby("week_start", observed=True, as_index=False)
    .agg(
        jobs=("week_error_index", "size"),
        q25_error_index=("week_error_index", lambda s: s.quantile(0.25)),
        median_error_index=("week_error_index", "median"),
    )
)

weekly_system_error_df["shared_shock"] = (
    weekly_system_error_df["jobs"].ge(minimum_week_jobs)
    & weekly_system_error_df["q25_error_index"].gt(100.0)
)

SHARED_SHOCK_WEEKS = set(
    weekly_system_error_df.loc[
        weekly_system_error_df["shared_shock"], "week_start"
    ]
)

The calibration is easier to audit when the chosen boundaries are shown beside the alternatives. The compact table gives the final diagnostic definitions, while the three curves show how strongly error separates as each candidate threshold moves.

In [22]:
# ---------------------------------------------------------------------
# 4.6.2.3.6A — Summarize and visualize the calibrated thresholds
# ---------------------------------------------------------------------

def format_percent_4636(value):
    """Format a calibrated proportion while keeping failed calibration visible."""
    return f"{100 * value:.0f}%" if np.isfinite(value) else "No stable threshold"


diagnostic_thresholds_display_df = pd.DataFrame(
    [
        {
            "Failure mechanism": "Weak historical support",
            "Diagnostic definition": (
                f"History observed < {format_percent_4636(DIAGNOSTIC_HISTORY_SHARE_CUTOFF)}"
            ),
        },
        {
            "Failure mechanism": "Structural low / zero demand",
            "Diagnostic definition": (
                "Recent activity ≤ job-specific "
                f"{format_percent_4636(DIAGNOSTIC_LOW_ACTIVITY_QUANTILE)} percentile, "
                "or exact trip-count zero warning"
            ),
        },
        {
            "Failure mechanism": "Unstable recent conditions",
            "Diagnostic definition": (
                "Recent speed volatility ≥ job-specific "
                f"{format_percent_4636(DIAGNOSTIC_HIGH_VOLATILITY_QUANTILE)} percentile"
            ),
        },
        {
            "Failure mechanism": "Temporary shared shock",
            "Diagnostic definition": (
                "Weekly 25th-percentile job error index > 100 "
                f"({len(SHARED_SHOCK_WEEKS)} weeks)"
            ),
        },
    ]
)

display(diagnostic_thresholds_display_df)


def plot_threshold_scan_4636(
    candidate_df, *, selected_candidate, title, x_title
):
    """Show how strongly the risky side separates as the boundary moves."""
    plot_df = candidate_df.copy()

    fig = go.Figure(
        go.Scatter(
            x=100 * plot_df["candidate"],
            y=plot_df["median_mae_ratio"],
            mode="lines+markers",
            line={"color": BRAND_COLORS["seafoam"], "width": 3},
            marker={"color": BRAND_COLORS["dark_teal"], "size": 9},
            customdata=np.column_stack(
                [
                    100 * plot_df["median_risk_share"],
                    100 * plot_df["supported_job_share"],
                    100 * plot_df["worse_job_share"],
                ]
            ),
            hovertemplate=(
                "Candidate: %{x:.0f}%<br>"
                "Risk / safe MAE: %{y:.2f}×<br>"
                "Median forecasts flagged: %{customdata[0]:.1f}%<br>"
                "Jobs with adequate support: %{customdata[1]:.1f}%<br>"
                "Supported jobs where risk is worse: %{customdata[2]:.1f}%"
                "<extra></extra>"
            ),
        )
    )

    fig.add_hline(
        y=1.0,
        line_dash="dot",
        line_color=BRAND_COLORS["dark_teal"],
        opacity=0.55,
        annotation_text="No error separation",
        annotation_position="bottom right",
    )

    if np.isfinite(selected_candidate):
        selected_row = plot_df.loc[
            np.isclose(plot_df["candidate"], selected_candidate)
        ].iloc[0]

        fig.add_trace(
            go.Scatter(
                x=[100 * selected_candidate],
                y=[selected_row["median_mae_ratio"]],
                mode="markers",
                name="Selected threshold",
                marker={
                    "color": BRAND_COLORS["terracotta"],
                    "size": 15,
                    "symbol": "diamond",
                    "line": {"color": BRAND_COLORS["dark_teal"], "width": 1},
                },
                hovertemplate=(
                    "<b>Selected diagnostic threshold</b><br>"
                    "Candidate: %{x:.0f}%<br>"
                    "Risk / safe MAE: %{y:.2f}×"
                    "<extra></extra>"
                ),
            )
        )

    fig = apply_branding(fig)
    fig.update_layout(
        title=title,
        height=430,
        xaxis_title=x_title,
        yaxis_title="Median risk-side MAE ÷ safe-side MAE",
        margin={"l": 75, "r": 35, "t": 75, "b": 60},
        showlegend=bool(np.isfinite(selected_candidate)),
    )
    fig.update_xaxes(ticksuffix="%")

    return fig


if RUN_VISUALS:
    plot_threshold_scan_4636(
        history_candidate_df,
        selected_candidate=DIAGNOSTIC_HISTORY_SHARE_CUTOFF,
        title="Where does incomplete history begin to separate forecast error?",
        x_title="Observed-history cutoff",
    ).show()

    plot_threshold_scan_4636(
        activity_candidate_df,
        selected_candidate=DIAGNOSTIC_LOW_ACTIVITY_QUANTILE,
        title="How deep into the low-activity tail does demand error separate?",
        x_title="Low-activity percentile cutoff",
    ).show()

    plot_threshold_scan_4636(
        volatility_candidate_df,
        selected_candidate=DIAGNOSTIC_HIGH_VOLATILITY_QUANTILE,
        title="How extreme must speed volatility become before error separates?",
        x_title="High-volatility percentile cutoff",
    ).show()

shared_shock_labels = [
    timestamp.strftime("%Y-%m-%d") for timestamp in sorted(SHARED_SHOCK_WEEKS)
]
print("Shared-shock weeks:", ", ".join(shared_shock_labels) or "None")

,Failure mechanism,Diagnostic definition
0,Weak historical support,History observed < No stable threshold
1,Structural low / zero demand,Recent activity ≤ job-specific No stable thres...
2,Unstable recent conditions,Recent speed volatility ≥ job-specific 95% per...
3,Temporary shared shock,Weekly 25th-percentile job error index > 100 (...


Shared-shock weeks: 2026-01-19, 2026-01-26, 2026-02-16, 2026-02-23


**Findings.** The threshold calibration separates the earlier hypotheses into two different kinds of evidence.

**Extreme speed volatility produces the clearest stable boundary.** The risky side has higher MAE in all six speed jobs across the full candidate range, and the separation strengthens toward the tail. The selected **95th-percentile volatility threshold** flags about 5% of speed forecasts and has roughly **2.23× the MAE** of the remaining forecasts.

By contrast, **history completeness does not support one systemwide hard cutoff**. Only the 90% candidate has enough risky observations to show meaningful separation—and even there, adequate support exists for only about 20% of jobs. Historical completeness can therefore remain a useful continuous warning signal without labeling every forecast below one universal percentage as “weak.”

Low recent activity also does **not** produce a stable absolute-MAE failure threshold. Across every tested demand cutoff from the 5th through 40th percentile, the low-activity side has *lower* native-unit MAE than the remaining forecasts. This does not contradict the earlier Relative-MAE finding: quiet demand can be proportionally difficult while still producing small absolute errors because the underlying counts are small.

Finally, four weeks meet the broad shared-shock definition: **January 19, January 26, February 16, and February 23, 2026**.

**4.6.2.3.6B — Measure how common each mechanism is and how much error is concentrated there.**

With the diagnostic boundaries frozen, we can now distinguish **rare but severe** failures from problems that account for a large share of the forecasting system's total error.

The first step applies those definitions row by row and creates a common accounting scale. Each Target × Horizon receives equal weight so a large Subway error cannot numerically overwhelm a smaller speed error simply because the units differ.

In [23]:
# ---------------------------------------------------------------------
# 4.6.2.3.6B — Apply the taxonomy and build equal-job error weights
# ---------------------------------------------------------------------

history_share = failure_burden_df["same_daypart_roll_obs_share_28d"]
recent_level = failure_burden_df["same_daypart_roll_mean_28d"]
recent_volatility = failure_burden_df["same_daypart_roll_std_28d"]
origin_value = failure_burden_df["origin_value"]


# Percentile-based definitions must first be translated back into the physical
# activity or volatility scale for each individual Target × Horizon.
if np.isfinite(DIAGNOSTIC_LOW_ACTIVITY_QUANTILE):
    activity_threshold_df = (
        failure_burden_df.loc[failure_burden_df["target_type"].eq("demand")]
        .groupby(FAILURE_JOB_KEYS, observed=True)["same_daypart_roll_mean_28d"]
        .quantile(DIAGNOSTIC_LOW_ACTIVITY_QUANTILE)
        .rename("diagnostic_activity_threshold")
        .reset_index()
    )
    failure_burden_df = failure_burden_df.merge(
        activity_threshold_df,
        on=FAILURE_JOB_KEYS,
        how="left",
        validate="many_to_one",
    )
else:
    failure_burden_df["diagnostic_activity_threshold"] = np.nan


if np.isfinite(DIAGNOSTIC_HIGH_VOLATILITY_QUANTILE):
    volatility_threshold_df = (
        failure_burden_df.loc[failure_burden_df["target_type"].eq("speed")]
        .groupby(FAILURE_JOB_KEYS, observed=True)["same_daypart_roll_std_28d"]
        .quantile(DIAGNOSTIC_HIGH_VOLATILITY_QUANTILE)
        .rename("diagnostic_volatility_threshold")
        .reset_index()
    )
    failure_burden_df = failure_burden_df.merge(
        volatility_threshold_df,
        on=FAILURE_JOB_KEYS,
        how="left",
        validate="many_to_one",
    )
else:
    failure_burden_df["diagnostic_volatility_threshold"] = np.nan


# Weak support is used only if calibration found a stable boundary.
failure_burden_df["weak_support_applicable"] = (
    history_share.notna() & np.isfinite(DIAGNOSTIC_HISTORY_SHARE_CUTOFF)
)
failure_burden_df["weak_support"] = (
    failure_burden_df["weak_support_applicable"]
    & history_share.lt(DIAGNOSTIC_HISTORY_SHARE_CUTOFF)
)


# Exact zero states are literal observations rather than fitted thresholds.
# A recent zero-history flag requires at least one observed historical value,
# keeping genuine inactivity separate from missing historical information.
failure_burden_df["origin_zero"] = (
    failure_burden_df["is_trip_count"]
    & origin_value.notna()
    & np.isclose(origin_value, 0.0, atol=1e-12)
)

zero_history_known = history_share.gt(0) & recent_level.notna()
failure_burden_df["recent_all_zero"] = (
    failure_burden_df["is_trip_count"]
    & zero_history_known
    & np.isclose(recent_level, 0.0, atol=1e-12)
)

low_activity_known = (
    failure_burden_df["target_type"].eq("demand")
    & recent_level.notna()
    & failure_burden_df["diagnostic_activity_threshold"].notna()
)

failure_burden_df["low_activity"] = (
    low_activity_known
    & recent_level.le(failure_burden_df["diagnostic_activity_threshold"])
)

# Taxi/FHVHV exact-zero warnings remain meaningful even if the broader
# low-activity calibration fails to produce a stable percentile boundary.
failure_burden_df["low_zero_demand_applicable"] = (
    low_activity_known
    | (
        failure_burden_df["is_trip_count"]
        & (origin_value.notna() | zero_history_known)
    )
)

failure_burden_df["low_zero_demand"] = (
    failure_burden_df["low_zero_demand_applicable"]
    & (
        failure_burden_df["low_activity"]
        | failure_burden_df["origin_zero"]
        | failure_burden_df["recent_all_zero"]
    )
)


failure_burden_df["unstable_conditions_applicable"] = (
    failure_burden_df["target_type"].eq("speed")
    & recent_volatility.notna()
    & failure_burden_df["diagnostic_volatility_threshold"].notna()
)

failure_burden_df["unstable_conditions"] = (
    failure_burden_df["unstable_conditions_applicable"]
    & recent_volatility.ge(failure_burden_df["diagnostic_volatility_threshold"])
)

failure_burden_df["shared_shock_applicable"] = True
failure_burden_df["shared_shock"] = failure_burden_df["week_start"].isin(
    SHARED_SHOCK_WEEKS
)


# Give each of the 15 jobs exactly 1/15 of the systemwide prevalence and error
# budget. Rows then divide that job's contribution internally.
job_error_totals_df = (
    failure_burden_df.groupby(FAILURE_JOB_KEYS, observed=True, as_index=False)
    .agg(
        job_rows=("absolute_error", "size"),
        job_absolute_error=("absolute_error", "sum"),
        job_mae=("absolute_error", "mean"),
    )
)

job_count = len(job_error_totals_df)
if job_count != 15:
    raise RuntimeError(f"Expected 15 Target × Horizon jobs; found {job_count}.")

failure_burden_df = (
    failure_burden_df.drop(columns="job_mae")
    .merge(
        job_error_totals_df,
        on=FAILURE_JOB_KEYS,
        how="left",
        validate="many_to_one",
    )
)

failure_burden_df["equal_job_row_weight"] = (
    1.0 / (job_count * failure_burden_df["job_rows"])
)

failure_burden_df["equal_job_error_weight"] = np.where(
    failure_burden_df["job_absolute_error"].gt(0),
    failure_burden_df["absolute_error"]
    / failure_burden_df["job_absolute_error"]
    / job_count,
    0.0,
)

failure_burden_df["row_error_index"] = np.where(
    failure_burden_df["job_mae"].gt(0),
    100.0 * failure_burden_df["absolute_error"] / failure_burden_df["job_mae"],
    np.nan,
)

# "Severe" means the worst 10% of errors within each job, so every forecasting
# task contributes its own tail rather than competing in incompatible units.
severe_thresholds = (
    failure_burden_df.groupby(FAILURE_JOB_KEYS, observed=True)["absolute_error"]
    .transform(lambda s: s.quantile(SEVERE_ERROR_QUANTILE))
)

failure_burden_df["severe_error"] = failure_burden_df["absolute_error"].ge(
    severe_thresholds
)
failure_burden_df["benchmark_loss"] = (
    failure_burden_df["absolute_error"]
    > failure_burden_df["benchmark_absolute_error"]
)


# A hypothesis that failed calibration is not quietly resurrected here.
mechanism_specs = []

if np.isfinite(DIAGNOSTIC_HISTORY_SHARE_CUTOFF):
    mechanism_specs.append(
        {
            "mechanism": "Weak historical support",
            "short_label": "Weak support",
            "flag": "weak_support",
            "applicable": "weak_support_applicable",
        }
    )

mechanism_specs.append(
    {
        "mechanism": "Structural low / zero demand",
        "short_label": "Low / zero demand",
        "flag": "low_zero_demand",
        "applicable": "low_zero_demand_applicable",
    }
)

if np.isfinite(DIAGNOSTIC_HIGH_VOLATILITY_QUANTILE):
    mechanism_specs.append(
        {
            "mechanism": "Unstable recent conditions",
            "short_label": "High speed volatility",
            "flag": "unstable_conditions",
            "applicable": "unstable_conditions_applicable",
        }
    )

mechanism_specs.append(
    {
        "mechanism": "Temporary shared shocks",
        "short_label": "Shared-shock weeks",
        "flag": "shared_shock",
        "applicable": "shared_shock_applicable",
    }
)

**4.6.2.3.6B — Summarize how common, severe, and consequential each failure mechanism is.**

A failure condition can matter in several different ways. It may occur often, produce unusually large errors when it occurs, or account for a disproportionate share of the system's total error. These measures separate those ideas:

- **Applicable prevalence** — how often the condition occurs among forecasts where that condition is meaningful.
- **Median error index** — how severe the flagged forecasts are relative to an ordinary forecast in the same Target × Horizon job. **100 means typical error; 150 means 50% more error than usual.**
- **System error burden** — the condition's share of the total equal-job-weighted absolute error budget.
- **Burden / forecast share** — whether the condition contributes more error than its frequency alone would suggest. **1× means proportional; 2× means twice as much error burden as forecast share.**
- **Severe-error capture** — what share of the worst 10% of errors falls inside the condition.
- **Benchmark-loss capture** — what share of rows where the champion loses to the weekly benchmark falls inside the condition.
- **Associated excess burden** — a descriptive estimate of how much of the total error budget is associated with the condition having higher error than comparable unflagged forecasts. It is **not a causal effect**.

The bubble chart uses **frequency on the x-axis, severity on the y-axis, and total error burden as bubble size**, so conditions can be important for different reasons.

In [58]:
# ---------------------------------------------------------------------
# 4.6.2.3.6B — Summarize prevalence, severity, and error burden
# ---------------------------------------------------------------------

def summarize_failure_mechanism_4636(frame, spec):
    """
    Summarize one failure mechanism at both job and systemwide scales.

    Excess burden compares flagged and unflagged forecasts within each job,
    while equal-job weighting prevents native-unit scale from dominating.
    """
    flag_column = spec["flag"]
    applicable_column = spec["applicable"]
    job_records = []

    for (metric, horizon), job_df in frame.groupby(
        FAILURE_JOB_KEYS, observed=True
    ):
        applicable_df = job_df.loc[job_df[applicable_column]]

        if applicable_df.empty:
            continue

        flagged_df = applicable_df.loc[applicable_df[flag_column]]
        unflagged_df = applicable_df.loc[~applicable_df[flag_column]]

        prevalence_pct = 100.0 * len(flagged_df) / len(applicable_df)
        flagged_mae = (
            float(flagged_df["absolute_error"].mean())
            if len(flagged_df)
            else np.nan
        )
        job_mae = float(job_df["absolute_error"].mean())

        error_index_pct = (
            100.0 * flagged_mae / job_mae
            if len(flagged_df) and job_mae > 0
            else np.nan
        )

        severe_total = int(applicable_df["severe_error"].sum())
        benchmark_loss_total = int(applicable_df["benchmark_loss"].sum())

        severe_capture_pct = (
            100.0 * flagged_df["severe_error"].sum() / severe_total
            if severe_total
            else np.nan
        )
        benchmark_loss_capture_pct = (
            100.0 * flagged_df["benchmark_loss"].sum() / benchmark_loss_total
            if benchmark_loss_total
            else np.nan
        )

        excess_error_share_pct = np.nan
        if len(flagged_df) and len(unflagged_df):
            unflagged_mae = float(unflagged_df["absolute_error"].mean())
            excess_absolute_error = len(flagged_df) * (
                flagged_mae - unflagged_mae
            )
            job_absolute_error = float(job_df["absolute_error"].sum())

            if job_absolute_error > 0:
                excess_error_share_pct = (
                    100.0 * excess_absolute_error / job_absolute_error
                )

        job_records.append(
            {
                "metric": metric,
                "horizon": horizon,
                "prevalence_pct": prevalence_pct,
                "error_index_pct": error_index_pct,
                "severe_capture_pct": severe_capture_pct,
                "benchmark_loss_capture_pct": benchmark_loss_capture_pct,
                "excess_error_share_pct": excess_error_share_pct,
            }
        )

    job_summary_df = pd.DataFrame(job_records)
    flagged = frame[flag_column].fillna(False).astype(bool)

    system_forecast_share_pct = (
        100.0 * frame.loc[flagged, "equal_job_row_weight"].sum()
    )
    system_error_burden_pct = (
        100.0 * frame.loc[flagged, "equal_job_error_weight"].sum()
    )

    burden_concentration = (
        system_error_burden_pct / system_forecast_share_pct
        if system_forecast_share_pct > 0
        else np.nan
    )

    # Jobs where the mechanism is not applicable contribute zero rather than
    # silently receiving extra systemwide weight.
    associated_excess_burden_pct = (
        job_summary_df["excess_error_share_pct"].fillna(0.0).sum() / job_count
    )

    return {
        "Mechanism": spec["mechanism"],
        "Short label": spec["short_label"],
        "Applicable jobs": len(job_summary_df),
        "Applicable prevalence (%)": job_summary_df["prevalence_pct"].mean(),
        "System forecast share (%)": system_forecast_share_pct,
        "Median error index": job_summary_df["error_index_pct"].median(),
        "System error burden (%)": system_error_burden_pct,
        "Burden / forecast share (×)": burden_concentration,
        "Severe-error capture (%)": job_summary_df["severe_capture_pct"].mean(),
        "Benchmark-loss capture (%)": (
            job_summary_df["benchmark_loss_capture_pct"].mean()
        ),
        "Associated excess error burden (%)": associated_excess_burden_pct,
    }


failure_mechanism_summary_df = pd.DataFrame(
    [
        summarize_failure_mechanism_4636(failure_burden_df, spec)
        for spec in mechanism_specs
    ]
)

failure_mechanism_display_df = failure_mechanism_summary_df[
    [
        "Mechanism",
        "Applicable jobs",
        "Applicable prevalence (%)",
        "System forecast share (%)",
        "Median error index",
        "System error burden (%)",
        "Burden / forecast share (×)",
        "Severe-error capture (%)",
        "Benchmark-loss capture (%)",
        "Associated excess error burden (%)",
    ]
].round(3)

display(failure_mechanism_display_df)


# The bubble chart separates three ideas visually:
# x = frequency, y = severity, and bubble size = total system error burden.
mechanism_palette = [
    BRAND_COLORS["dark_teal"],
    BRAND_COLORS["terracotta"],
    BRAND_COLORS["seafoam"],
    BRAND_CLUSTER_PALETTE[3],
]

mechanism_colors = {
    label: mechanism_palette[index]
    for index, label in enumerate(failure_mechanism_summary_df["Short label"])
}

if RUN_VISUALS:
    bubble_df = failure_mechanism_summary_df.copy()

    # Preformat only the field that Plotly was rendering with excessive
    # floating-point precision. The remaining hover fields already display
    # correctly in Deepnote.
    bubble_df["Associated excess burden display"] = (
        bubble_df["Associated excess error burden (%)"]
        .map(lambda value: f"{value:+.1f}%" if pd.notna(value) else "—")
    )

    maximum_burden = max(
        float(bubble_df["System error burden (%)"].max()),
        1.0,
    )
    size_reference = 2.0 * maximum_burden / (55.0 ** 2)

    fig = go.Figure(
        go.Scatter(
            x=bubble_df["Applicable prevalence (%)"],
            y=bubble_df["Median error index"],
            mode="markers+text",
            text=bubble_df["Short label"],
            textposition="top center",
            marker={
                "size": bubble_df["System error burden (%)"],
                "sizemode": "area",
                "sizeref": size_reference,
                "sizemin": 12,
                "color": [
                    mechanism_colors[label]
                    for label in bubble_df["Short label"]
                ],
                "line": {
                    "color": BRAND_COLORS["dark_teal"],
                    "width": 1,
                },
            },
            customdata=bubble_df[
                [
                    "Mechanism",
                    "System forecast share (%)",
                    "System error burden (%)",
                    "Burden / forecast share (×)",
                    "Severe-error capture (%)",
                    "Benchmark-loss capture (%)",
                    "Associated excess burden display",
                ]
            ],
            hovertemplate=(
                "<b>%{customdata[0]}</b><br>"
                "Applicable prevalence: %{x:.1f}%<br>"
                "System forecast share: %{customdata[1]:.1f}%<br>"
                "Median error index: %{y:.1f}<br>"
                "System error burden: %{customdata[2]:.1f}%<br>"
                "Burden / forecast share: %{customdata[3]:.2f}×<br>"
                "Severe-error capture: %{customdata[4]:.1f}%<br>"
                "Benchmark-loss capture: %{customdata[5]:.1f}%<br>"
                "Associated excess burden: %{customdata[6]}"
                "<extra></extra>"
            ),
        )
    )

    fig.add_hline(
        y=100,
        line_dash="dot",
        line_color=BRAND_COLORS["dark_teal"],
        opacity=0.5,
        annotation_text="Ordinary job error = 100",
        annotation_position="top left",
    )

    fig = apply_branding(fig)
    fig.update_layout(
        title=(
            "Which forecast-failure mechanisms matter most?"
            "<br><sup>Bubble size = share of the equal-job-weighted error budget.</sup>"
        ),
        height=570,
        xaxis_title="How often the condition occurs when applicable (%)",
        yaxis_title="Median error index (100 = ordinary job error)",
        margin={"l": 75, "r": 45, "t": 100, "b": 65},
        showlegend=False,
    )

    fig.show()

,Mechanism,Applicable jobs,Applicable prevalence (%),System forecast share (%),Median error index,System error burden (%),Burden / forecast share (×),Severe-error capture (%),Benchmark-loss capture (%),Associated excess error burden (%)
0,Structural low / zero demand,6,7.794,3.116,5.682,0.262,0.084,0.038,11.044,-3.352
1,Unstable recent conditions,6,5.001,1.980,207.957,4.086,2.064,15.618,4.908,2.238
2,Temporary shared shocks,15,32.674,32.674,124.632,41.148,1.259,45.194,30.632,12.595


**Findings.** The error-burden accounting shows that the failure mechanisms differ sharply in systemic importance.

**Shared-shock weeks are the dominant identified absolute-error mechanism.** They contain **32.7% of forecasts but 41.1% of the equal-job-weighted error budget**, with a median error index of 124.6. They also capture **45.2% of severe errors** and are associated with roughly **12.6% excess systemwide error burden**.

**Extreme speed volatility is much rarer but highly concentrated.** Only about **5.0% of applicable speed forecasts** meet the calibrated threshold, yet those rows have a median error index of **208.0** and consume about **2.06× their proportional share of error**. They capture 15.6% of severe speed-job errors despite representing only 2.0% of the overall equal-job-weighted forecast population.

The exact low/zero-demand warnings tell a different story. They identify about **7.8% of applicable trip-count forecasts**, but only **0.26% of the systemwide absolute-error budget**. Their negative associated excess-error estimate confirms that these rows generally have small native-unit errors. Their weakness is therefore better understood as **relative-scale and benchmark fragility**, not a major source of aggregate MAE.

**4.6.2.3.6C — Separate overlapping mechanisms so the same error is not counted twice.**

Failure mechanisms can occur together. A forecast may have weak history, very low demand, and occur during a difficult system-wide week at the same time.

The individual mechanism percentages therefore cannot simply be added. This block assigns every forecast to one mutually exclusive combination, making the resulting error-burden shares a complete accounting of the holdout surface.

In [25]:
# ---------------------------------------------------------------------
# 4.6.2.3.6C — Mutually exclusive failure-mechanism combinations
# ---------------------------------------------------------------------

combination_parts = [(spec["flag"], spec["short_label"]) for spec in mechanism_specs]

combination_label = pd.Series(
    "No identified mechanism", index=failure_burden_df.index, dtype="string"
)

for flag_column, label in combination_parts:
    flagged = failure_burden_df[flag_column].fillna(False).astype(bool)
    already_flagged = combination_label.ne("No identified mechanism")

    combination_label.loc[flagged & ~already_flagged] = label
    combination_label.loc[flagged & already_flagged] = (
        combination_label.loc[flagged & already_flagged] + " + " + label
    )

failure_burden_df["failure_combination"] = combination_label

# This weighted index preserves the same equal-job interpretation used in the
# burden analysis, while allowing combinations to be compared on severity.
failure_burden_df["weighted_error_index"] = (
    failure_burden_df["row_error_index"]
    * failure_burden_df["equal_job_row_weight"]
)

combination_summary_df = (
    failure_burden_df.groupby(
        "failure_combination", observed=True, as_index=False
    )
    .agg(
        rows=("absolute_error", "size"),
        system_prevalence_weight=("equal_job_row_weight", "sum"),
        system_error_burden_weight=("equal_job_error_weight", "sum"),
        weighted_error_index_sum=("weighted_error_index", "sum"),
    )
)

combination_summary_df["System forecast share (%)"] = (
    100.0 * combination_summary_df["system_prevalence_weight"]
)
combination_summary_df["System error burden (%)"] = (
    100.0 * combination_summary_df["system_error_burden_weight"]
)

combination_summary_df["Error index"] = np.where(
    combination_summary_df["system_prevalence_weight"].gt(0),
    combination_summary_df["weighted_error_index_sum"]
    / combination_summary_df["system_prevalence_weight"],
    np.nan,
)

combination_summary_df["Burden / forecast share (×)"] = np.where(
    combination_summary_df["System forecast share (%)"].gt(0),
    combination_summary_df["System error burden (%)"]
    / combination_summary_df["System forecast share (%)"],
    np.nan,
)


# Severe-error and benchmark-loss shares are mutually exclusive here, so each
# of these columns legitimately sums to approximately 100% across combinations.
severe_weight_total = failure_burden_df.loc[
    failure_burden_df["severe_error"], "equal_job_row_weight"
].sum()

benchmark_loss_weight_total = failure_burden_df.loc[
    failure_burden_df["benchmark_loss"], "equal_job_row_weight"
].sum()

severe_weight_by_combination = (
    failure_burden_df.loc[failure_burden_df["severe_error"]]
    .groupby("failure_combination", observed=True)["equal_job_row_weight"]
    .sum()
)

benchmark_loss_weight_by_combination = (
    failure_burden_df.loc[failure_burden_df["benchmark_loss"]]
    .groupby("failure_combination", observed=True)["equal_job_row_weight"]
    .sum()
)

combination_summary_df["Severe-error share (%)"] = (
    combination_summary_df["failure_combination"]
    .map(severe_weight_by_combination)
    .fillna(0.0)
    .mul(100.0 / severe_weight_total if severe_weight_total > 0 else np.nan)
)

combination_summary_df["Benchmark-loss share (%)"] = (
    combination_summary_df["failure_combination"]
    .map(benchmark_loss_weight_by_combination)
    .fillna(0.0)
    .mul(
        100.0 / benchmark_loss_weight_total
        if benchmark_loss_weight_total > 0
        else np.nan
    )
)

combination_summary_df = combination_summary_df.sort_values(
    "System error burden (%)", ascending=False
).reset_index(drop=True)

combination_display_df = (
    combination_summary_df[
        [
            "failure_combination",
            "System forecast share (%)",
            "System error burden (%)",
            "Error index",
            "Burden / forecast share (×)",
            "Severe-error share (%)",
            "Benchmark-loss share (%)",
        ]
    ]
    .rename(columns={"failure_combination": "Failure combination"})
    .round(3)
)

# The table stays compact; the chart retains every observed combination.
display(combination_display_df.head(10))


if RUN_VISUALS:
    plot_df = combination_summary_df.iloc[::-1].copy()

    fig = go.Figure(
        go.Bar(
            x=plot_df["System error burden (%)"],
            y=plot_df["failure_combination"],
            orientation="h",
            marker_color=BRAND_COLORS["terracotta"],
            text=plot_df["System error burden (%)"].map(lambda value: f"{value:.1f}%"),
            textposition="outside",
            customdata=plot_df[
                [
                    "System forecast share (%)",
                    "Error index",
                    "Burden / forecast share (×)",
                    "Severe-error share (%)",
                    "Benchmark-loss share (%)",
                ]
            ],
            hovertemplate=(
                "<b>%{y}</b><br>"
                "System error burden: %{x:.1f}%<br>"
                "System forecast share: %{customdata[0]:.1f}%<br>"
                "Error index: %{customdata[1]:.1f}<br>"
                "Burden / forecast share: %{customdata[2]:.2f}×<br>"
                "Share of severe errors: %{customdata[3]:.1f}%<br>"
                "Share of benchmark losses: %{customdata[4]:.1f}%"
                "<extra></extra>"
            ),
        )
    )

    fig = apply_branding(fig)
    fig.update_layout(
        title=(
            "Where does the forecasting system's error burden accumulate?"
            "<br><sup>Each forecast appears in exactly one mechanism combination.</sup>"
        ),
        height=max(500, 42 * len(plot_df) + 130),
        xaxis_title="Share of equal-job-weighted absolute-error budget (%)",
        yaxis_title="",
        margin={"l": 260, "r": 65, "t": 100, "b": 60},
    )
    fig.update_xaxes(ticksuffix="%")
    fig.show()

,Failure combination,System forecast share (%),System error burden (%),Error index,Burden / forecast share (×),Severe-error share (%),Benchmark-loss share (%)
0,No identified mechanism,63.915,56.131,87.820,0.878,51.045,64.774
1,Shared-shock weeks,30.989,39.521,127.532,1.275,42.876,28.643
2,High speed volatility,1.285,2.554,198.742,1.987,3.757,1.079
3,High speed volatility + Shared-shock weeks,0.695,1.532,220.466,2.205,2.307,0.535
4,Low / zero demand,2.126,0.167,7.865,0.079,0.004,3.384
5,Low / zero demand + Shared-shock weeks,0.989,0.095,9.588,0.096,0.011,1.585


**Findings.** The mutually exclusive accounting confirms that **shared shocks explain most of the identified error concentration**, while extreme speed volatility defines the most intense smaller regime.

Shared-shock weeks alone represent **31.0% of forecasts but 39.5% of the systemwide error budget**, with an error index of 127.5. Extreme speed volatility alone represents just **1.3% of forecasts but 2.6% of error**, while the combination of extreme volatility and a shared-shock week is even more concentrated: **0.7% of forecasts, 1.5% of error, and an error index of 220.5**.

Low/zero-demand combinations contribute very little absolute-error burden. Together they represent roughly 3.1% of forecasts but only about 0.3% of error.

Importantly, **63.9% of forecasts receive no row-level mechanism label and still account for 56.1% of total error and 51.0% of severe errors**. The taxonomy therefore identifies meaningful concentrations of risk rather than claiming to explain every individual forecast miss.

**4.6.2.3.6D — Keep Subway h=2 separate as a structural model / horizon fragility.**

Subway h=2 is different from the row-level mechanisms above. It does not appear intermittently within a job; it is one entire forecasting job whose reliability is consistently weaker.

Its importance is therefore better described by **scope and relative weakness**. The compact comparison below places h=2 beside the other Subway horizons using Relative MAE, benchmark skill, and the frequency of losing to the weekly benchmark.

In [26]:
# ---------------------------------------------------------------------
# 4.6.2.3.6D — Scope of the Subway h=2 structural fragility
# ---------------------------------------------------------------------

subway_scope_df = (
    failure_burden_df.loc[
        failure_burden_df["metric"].eq("subway_ridership")
    ]
    .groupby("horizon", observed=True, as_index=False)
    .agg(
        rows=("absolute_error", "size"),
        absolute_error_sum=("absolute_error", "sum"),
        benchmark_absolute_error_sum=("benchmark_absolute_error", "sum"),
        absolute_actual_sum=("actual", lambda s: s.abs().sum()),
        benchmark_loss_rows=("benchmark_loss", "sum"),
    )
)

subway_scope_df["Holdout row share (%)"] = (
    100.0 * subway_scope_df["rows"] / len(failure_burden_df)
)

subway_scope_df["Relative MAE (%)"] = np.where(
    subway_scope_df["absolute_actual_sum"].gt(0),
    100.0
    * subway_scope_df["absolute_error_sum"]
    / subway_scope_df["absolute_actual_sum"],
    np.nan,
)

subway_scope_df["Benchmark skill (%)"] = np.where(
    subway_scope_df["benchmark_absolute_error_sum"].gt(0),
    100.0
    * (
        1.0
        - subway_scope_df["absolute_error_sum"]
        / subway_scope_df["benchmark_absolute_error_sum"]
    ),
    np.nan,
)

subway_scope_df["Benchmark-loss rows (%)"] = (
    100.0 * subway_scope_df["benchmark_loss_rows"] / subway_scope_df["rows"]
)

subway_scope_display_df = (
    subway_scope_df[
        [
            "horizon",
            "rows",
            "Holdout row share (%)",
            "Relative MAE (%)",
            "Benchmark skill (%)",
            "Benchmark-loss rows (%)",
        ]
    ]
    .rename(columns={"horizon": "Horizon", "rows": "Rows"})
    .round(3)
)

display(subway_scope_display_df)


# A single summary number makes the horizon-specific weakness easier to read
# without replacing the richer h=1 / h=2 / h=5 comparison above.
h2_relative_mae = float(
    subway_scope_df.loc[
        subway_scope_df["horizon"].eq(2), "Relative MAE (%)"
    ].iloc[0]
)

peer_relative_mae = float(
    subway_scope_df.loc[
        subway_scope_df["horizon"].isin([1, 5]), "Relative MAE (%)"
    ].mean()
)

h2_relative_mae_uplift = (
    100.0 * (h2_relative_mae - peer_relative_mae) / peer_relative_mae
)

print(
    "Subway h=2 structural scope:",
    "1 of 15 forecasting jobs ·",
    "1 of 3 Subway horizons ·",
    f"Relative MAE is {h2_relative_mae_uplift:+.1f}% "
    "versus the mean of h=1 and h=5.",
)

,Horizon,Rows,Holdout row share (%),Relative MAE (%),Benchmark skill (%),Benchmark-loss rows (%)
0,1,64477,4.536,8.841,35.836,49.016
1,2,64477,4.536,11.130,19.223,59.446
2,5,64477,4.536,9.871,28.361,48.335


Subway h=2 structural scope: 1 of 15 forecasting jobs · 1 of 3 Subway horizons · Relative MAE is +19.0% versus the mean of h=1 and h=5.


**Findings.** Subway h=2 remains a distinct structural fragility rather than a row-level failure condition.

Its **11.13% Relative MAE** is about **19.0% higher than the mean of h=1 and h=5**. Its benchmark skill is also substantially weaker at **19.2%**, compared with 35.8% for h=1 and 28.4% for h=5.

Most strikingly, the h=2 champion loses to the weekly benchmark on **59.4% of holdout rows**, versus roughly 49% for both neighboring horizons. Combined with the earlier geographic, temporal, and runner-up evidence, this supports retaining Subway h=2 as a separate model/horizon reliability concern.

### Section Summary

Forecast weakness takes several different forms, but the burden analysis changes which ones matter most to the system as a whole.

The clearest broad concentration comes from **temporary shared shocks**. The four difficult weeks identified above contain about one-third of forecasts but roughly two-fifths of the equal-job-weighted absolute-error budget. **Extreme recent speed volatility** is much rarer—about the upper 5% of applicable speed conditions—but carries roughly twice its proportional share of speed-forecast error.

The sparse-demand story is more nuanced. Genuine zero and near-zero Taxi/FHVHV conditions can produce spectacular **relative** errors for individual locations such as Freshkills Park, yet the exact low/zero-demand warning states account for only a very small share of total system absolute error. These are important local proportional failures, not a dominant source of aggregate forecasting error.

Incomplete recent history remains useful interpretive context because error rises when support is weak, but the calibration exercise did **not** identify one stable history-completeness cutoff that cleanly separates reliable from unreliable forecasts across the champion system. The same is true for a universal low-activity cutoff. These conditions therefore describe difficult forecasting contexts rather than justify a generic holdout-derived rejection rule.

Geography and mobility environments still help explain **where** weakness concentrates, while **Subway h=2** remains a separate whole-job fragility: it is weaker than the other Subway horizons across several independent reliability views rather than failing only during a particular row-level condition.

Overall, 4.6.2 has identified interpretable contexts associated with forecast error without establishing a defensible basis for removing ordinary forecasts from this evaluation. The next question is different: **what information are the frozen champion models actually using when they make their forecasts?**

> ✅ Decision — Retain the full frozen forecasting surface for deep evaluation

4\.6\.2 will not create a holdout\-derived abstention or forecast\-rejection rule\. Shared shocks, extreme speed volatility, sparse or zero demand, incomplete history, geographic context, and Subway h=2 fragility remain diagnostic evidence used to explain where the champion system is easier or harder to trust\.

Genuine observed zero activity remains valid data, not missingness\. Likewise, difficult or volatile conditions are not grounds for deleting a forecast merely because they are hard to predict\.

If downstream Chapter 5 or Showcase packaging requires a data\-sufficiency gate, 4\.7\.1 will define that separately from authoritative pre\-origin support and availability semantics\. It will not tune an exclusion rule to improve final\-holdout error\.

## 4.6.2.4 Explain What the Champion Models Use

The previous section asked **where and under what conditions forecast errors become large**. That is not the same question as asking **what information the forecasting models use to make their predictions**.

Because the frozen champion system contains three different model families, there is no single honest feature-importance measure for all 15 jobs. The Tree champions expose feature-level **SHAP importance**, while the Neural and Transformer champions were already tested with **grouped permutation reliance**: one coherent input group was scrambled while the fitted model stayed fixed, and the resulting increase in forecast error measured how much the model depended on that information.

These methods answer the same broad question—*what information does this fitted forecasting system rely on?*—but their percentages are not interchangeable units and they do not establish causality. We therefore interpret each family with its own validated evidence rather than forcing all three into one artificial ranking.

This section first restricts the upstream interpretation artifacts to the **15 frozen champion jobs**. It then examines Tree predictors and feature families, Neural input groups, and Transformer input groups before deciding which Tree feature families warrant the more expensive formal ablation test.

**4.6.2.4.1 — Recover the interpretation evidence for the frozen champions.**

The upstream model notebooks already performed the expensive interpretation work. We only need the portions belonging to the families that actually won each Target × Horizon job: eight Tree champions, three Neural champions, and four Transformer champions.

The Tree evidence is also joined back to the authoritative 4.2.3 feature taxonomy. Taxi Zone identity is the one additional panel-model input introduced by the Tree branch, so it retains the same **Series identity** family used upstream.

The guards below make one important promise: after filtering, every one of the 15 frozen champion jobs must have the appropriate interpretation evidence before we draw conclusions from it.

In [27]:
# ---------------------------------------------------------------------
# 4.6.2.4.1 — Recover champion-only interpretation evidence
# ---------------------------------------------------------------------

tree_importance_df = pd.read_parquet(
    require_file(TREE_FEATURE_IMPORTANCE_PATH, "Tree feature importance"),
    columns=TREE_FEATURE_IMPORTANCE_REQUIRED_COLUMNS,
)
feature_catalog_df = pd.read_parquet(
    require_file(FEATURE_CATALOG_PATH, "forecast feature catalog"),
    columns=FEATURE_CATALOG_INTERPRETATION_REQUIRED_COLUMNS,
)
neural_importance_df = pd.read_parquet(
    require_file(NEURAL_GROUPED_IMPORTANCE_PATH, "Neural grouped importance"),
    columns=NEURAL_GROUPED_IMPORTANCE_REQUIRED_COLUMNS,
)
transformer_reliance_df = pd.read_parquet(
    require_file(
        TRANSFORMER_GROUPED_RELIANCE_PATH,
        "Transformer grouped reliance",
    ),
    columns=TRANSFORMER_GROUPED_RELIANCE_REQUIRED_COLUMNS,
)

require_columns(
    tree_importance_df,
    TREE_FEATURE_IMPORTANCE_REQUIRED_COLUMNS,
    "Tree feature importance",
)
require_columns(
    feature_catalog_df,
    FEATURE_CATALOG_INTERPRETATION_REQUIRED_COLUMNS,
    "forecast feature catalog",
)
require_columns(
    neural_importance_df,
    NEURAL_GROUPED_IMPORTANCE_REQUIRED_COLUMNS,
    "Neural grouped importance",
)
require_columns(
    transformer_reliance_df,
    TRANSFORMER_GROUPED_RELIANCE_REQUIRED_COLUMNS,
    "Transformer grouped reliance",
)

if feature_catalog_df["feature_name"].duplicated().any():
    raise RuntimeError(
        "The authoritative feature catalog must contain one family per feature."
    )

# Use the upstream feature taxonomy so later Tree-family ablations correspond
# to information groups that already have an established modeling meaning.
feature_family_lookup_df = feature_catalog_df.rename(
    columns={"feature_name": "feature"}
).copy()

feature_family_lookup_df = pd.concat(
    [
        feature_family_lookup_df,
        pd.DataFrame(
            {
                "feature": ["taxi_zone_id"],
                "feature_family": ["Series identity"],
            }
        ),
    ],
    ignore_index=True,
).drop_duplicates("feature")


def keep_family_champions(
    frame: pd.DataFrame,
    family: str,
) -> pd.DataFrame:
    """Keep only Target × Horizon jobs won by one frozen champion family."""
    family_jobs = (
        champion_registry_df.loc[
            champion_registry_df["champion_family"].eq(family),
            ["metric", "horizon"],
        ]
        .drop_duplicates()
    )

    result = frame.merge(
        family_jobs,
        on=["metric", "horizon"],
        how="inner",
        validate="many_to_one",
    )
    result["champion_family"] = family

    return result


tree_champion_importance_df = keep_family_champions(
    tree_importance_df,
    "tree",
)
neural_champion_importance_df = keep_family_champions(
    neural_importance_df,
    "neural",
)
transformer_champion_reliance_df = keep_family_champions(
    transformer_reliance_df,
    "transformer",
)

# Every LightGBM predictor must belong to an interpretable semantic family
# before grouped summaries or later ablations are allowed.
tree_champion_importance_df = tree_champion_importance_df.merge(
    feature_family_lookup_df,
    on="feature",
    how="left",
    validate="many_to_one",
)

if tree_champion_importance_df["feature_family"].isna().any():
    missing_features = sorted(
        tree_champion_importance_df.loc[
            tree_champion_importance_df["feature_family"].isna(),
            "feature",
        ].unique()
    )
    raise RuntimeError(
        "Tree champion features are missing authoritative family labels: "
        f"{missing_features}"
    )

interpretation_frames = {
    "tree": tree_champion_importance_df,
    "neural": neural_champion_importance_df,
    "transformer": transformer_champion_reliance_df,
}

expected_interpretation_jobs = {
    (row.champion_family, row.metric, int(row.horizon))
    for row in champion_registry_df[
        ["champion_family", "metric", "horizon"]
    ].itertuples(index=False)
}

observed_interpretation_jobs = set()

for family, frame in interpretation_frames.items():
    family_jobs = {
        (family, row.metric, int(row.horizon))
        for row in frame[["metric", "horizon"]]
        .drop_duplicates()
        .itertuples(index=False)
    }
    observed_interpretation_jobs.update(family_jobs)

if observed_interpretation_jobs != expected_interpretation_jobs:
    missing = sorted(
        expected_interpretation_jobs - observed_interpretation_jobs
    )
    unexpected = sorted(
        observed_interpretation_jobs - expected_interpretation_jobs
    )
    raise RuntimeError(
        "Champion interpretation coverage does not match the frozen registry. "
        f"Missing={missing}; unexpected={unexpected}"
    )

coverage_rows = [
    {
        "Family": "Tree",
        "Champion jobs": tree_champion_importance_df[
            ["metric", "horizon"]
        ].drop_duplicates().shape[0],
        "Interpretation method": "SHAP",
        "Evidence rows": len(tree_champion_importance_df),
        "Interpretation grain": "Individual predictor",
    },
    {
        "Family": "Neural",
        "Champion jobs": neural_champion_importance_df[
            ["metric", "horizon"]
        ].drop_duplicates().shape[0],
        "Interpretation method": "Grouped permutation",
        "Evidence rows": len(neural_champion_importance_df),
        "Interpretation grain": "Semantic input group",
    },
    {
        "Family": "Transformer",
        "Champion jobs": transformer_champion_reliance_df[
            ["metric", "horizon"]
        ].drop_duplicates().shape[0],
        "Interpretation method": "Grouped permutation",
        "Evidence rows": len(transformer_champion_reliance_df),
        "Interpretation grain": "Semantic input group",
    },
]

interpretation_coverage_df = pd.DataFrame(coverage_rows)
display(interpretation_coverage_df)

,Family,Champion jobs,Interpretation method,Evidence rows,Interpretation grain
0,Tree,8,SHAP,680,Individual predictor
1,Neural,3,Grouped permutation,24,Semantic input group
2,Transformer,4,Grouped permutation,32,Semantic input group


**Findings.** Interpretation evidence is available for every frozen champion job: **8 Tree**, **3 Neural**, and **4 Transformer** champions.

The evidence is intentionally architecture-specific. Tree champions retain individual-predictor SHAP evidence, while the Neural and Transformer champions use grouped permutation reliance. These measures answer the same broad question—what information the fitted model depends on—but they are not numerically interchangeable across model families.

All 15 champion jobs are therefore covered without forcing incompatible interpretation methods into one common scale.

### 4.6.2.4.2 What Information Drives the Tree Champions?

LightGBM gives us two complementary views of importance. **Gain importance** records which predictors the trees found useful while building splits. **SHAP importance** asks a more prediction-centered question: across held-out rows, how strongly did each predictor move the model's forecast?

For the deep-evaluation story, SHAP is the primary measure because it is closer to the forecasts we ultimately care about. Gain remains visible as a useful cross-check.

We begin at the individual-feature level so a broad family label does not hide the predictors actually carrying the signal.

**4.6.2.4.2A — Identify the strongest individual predictors in each Tree champion.**

Rather than print all 85 predictors for every model, the table keeps the top three SHAP contributors for each of the eight champion jobs. Their authoritative feature-family labels remain attached so the individual and grouped views stay connected.

In [28]:
# 4.6.2.4.2A — Top individual predictors for Tree champions
# ---------------------------------------------------------------------

tree_top_features_df = (
    tree_champion_importance_df
    .sort_values(
        ["metric", "horizon", "shap_share", "feature"],
        ascending=[True, True, False, True],
    )
    .groupby(
        ["metric", "horizon"],
        observed=True,
        group_keys=False,
    )
    .head(3)
    .copy()
)

tree_top_features_df["Rank"] = (
    tree_top_features_df
    .groupby(["metric", "horizon"], observed=True)
    .cumcount()
    .add(1)
)

tree_top_features_df = add_job_labels(tree_top_features_df)

tree_top_features_display_df = tree_top_features_df[
    [
        "Job",
        "Rank",
        "feature",
        "feature_family",
        "shap_share",
        "gain_share",
    ]
].copy()

tree_top_features_display_df["Job"] = pd.Categorical(
    tree_top_features_display_df["Job"],
    categories=JOB_LABEL_ORDER,
    ordered=True,
)
tree_top_features_display_df["SHAP share (%)"] = (
    100.0 * tree_top_features_display_df["shap_share"]
)
tree_top_features_display_df["Gain share (%)"] = (
    100.0 * tree_top_features_display_df["gain_share"]
)

tree_top_features_display_df = (
    tree_top_features_display_df
    .rename(
        columns={
            "feature": "Feature",
            "feature_family": "Feature family",
        }
    )
    .drop(columns=["shap_share", "gain_share"])
    .sort_values(["Job", "Rank"])
    .reset_index(drop=True)
)

display(tree_top_features_display_df.round(3))

,Job,Rank,Feature,Feature family,SHAP share (%),Gain share (%)
0,Taxi trips · h=1,1,seq_lag_4,Recent sequence history,55.184,73.929
1,Taxi trips · h=1,2,taxi_zone_id,Series identity,12.674,14.512
2,Taxi trips · h=1,3,origin_value,Current mobility state,5.236,0.877
3,Taxi trips · h=2,1,seq_lag_3,Recent sequence history,56.598,74.847
4,Taxi trips · h=2,2,taxi_zone_id,Series identity,15.563,14.265
5,Taxi trips · h=2,3,target_temporal_bucket,Target calendar,4.993,2.305
6,Taxi trips · h=5,1,origin_value,Current mobility state,52.796,68.544
7,Taxi trips · h=5,2,same_daypart_roll_mean_7d,Comparable Daypart history,17.603,23.261
8,Taxi trips · h=5,3,same_daypart_roll_mean_28d,Comparable Daypart history,5.658,1.200
9,FHVHV trips · h=1,1,seq_lag_4,Recent sequence history,47.771,66.631


**Findings.** The Tree champions show a clear horizon-dependent pattern in which individual predictors carry the forecast.

At **h=1 and h=2**, the leading predictors are almost always the most recent sequence lags. `seq_lag_4` contributes **47.771%–55.184%** of SHAP importance across the h=1 demand champions, while `seq_lag_3` contributes **48.685%–56.598%** for Taxi and FHVHV trips at h=2. Subway h=1 follows the same pattern, with `seq_lag_4` accounting for **49.822%**.

At **h=5**, the leaders change. Taxi and FHVHV trips rely most heavily on the **current observed mobility level** (`origin_value`), while Subway h=5 is led by the **7-day same-Daypart rolling mean**. Comparable-period rolling history also becomes much more prominent among the top predictors.

Gain importance broadly supports the same story: the predictors with the largest SHAP contributions also tend to receive the largest split gain. Gain is generally more concentrated on the leading predictor, so the two measures should be read as complementary rather than numerically interchangeable.

**4.6.2.4.2B — Step back from individual predictors to feature families.**

Individual features often carry related information. For example, several lag or rolling-history variables may each receive part of the importance that conceptually belongs to the same source of information.

The next view therefore sums SHAP shares within the **authoritative upstream feature families** for each Tree champion. Because SHAP shares already sum to one within a job, the grouped values remain easy to read: 40% means that predictors from that family collectively account for about 40% of that job's measured absolute SHAP importance.

In [29]:
# ---------------------------------------------------------------------
# 4.6.2.4.2B — Aggregate Tree SHAP reliance by feature family
# ---------------------------------------------------------------------

tree_family_share_df = (
    tree_champion_importance_df
    .groupby(
        ["metric", "horizon", "feature_family"],
        observed=True,
        as_index=False,
    )
    .agg(shap_share=("shap_share", "sum"))
)

tree_share_sums = (
    tree_family_share_df
    .groupby(["metric", "horizon"], observed=True)["shap_share"]
    .sum()
)

if not np.allclose(
    tree_share_sums.to_numpy(),
    1.0,
    atol=1e-6,
):
    raise RuntimeError(
        "Tree feature-family SHAP shares must sum to 1 "
        "within every champion job."
    )

tree_family_share_df = add_job_labels(tree_family_share_df)

tree_family_order = (
    tree_family_share_df
    .groupby("feature_family", observed=True)["shap_share"]
    .mean()
    .sort_values(ascending=False)
    .index
    .tolist()
)

tree_job_order = [
    job
    for job in JOB_LABEL_ORDER
    if job in set(tree_family_share_df["Job"])
]

tree_family_matrix_df = (
    tree_family_share_df
    .pivot(
        index="feature_family",
        columns="Job",
        values="shap_share",
    )
    .reindex(
        index=tree_family_order,
        columns=tree_job_order,
    )
    .fillna(0.0)
)

if RUN_VISUALS:
    tree_family_fig = px.imshow(
        100.0 * tree_family_matrix_df,
        color_continuous_scale=BRAND_SEQUENTIAL_HEATMAP_SCALE,
        aspect="auto",
        text_auto=".1f",
        labels={
            "x": "Tree champion job",
            "y": "Feature family",
            "color": "SHAP share (%)",
        },
    )
    tree_family_fig = apply_branding(tree_family_fig)
    tree_family_fig.update_layout(
        title=(
            "What information families drive the Tree champions?"
            "<br><sup>Share of absolute SHAP importance "
            "within each champion job</sup>"
        ),
        height=max(
            480,
            32 * len(tree_family_matrix_df) + 180,
        ),
        margin={
            "l": 175,
            "r": 40,
            "t": 90,
            "b": 70,
        },
    )
    tree_family_fig.show()

**Findings.** Grouping the Tree predictors reveals an especially clear change with forecast horizon.

For **h=1 and h=2**, **Recent sequence history** dominates the Tree champions, contributing roughly **52%–60%** of SHAP importance across Taxi trips, FHVHV trips, and Subway h=1. It is the highest-ranked feature family in **5 of the 8 Tree champion jobs**.

At **h=5**, that immediate sequence history largely gives way to two longer-view signals. **Current mobility state** contributes **52.796%** for Taxi trips and **35.528%** for FHVHV trips, while **Comparable Daypart history** contributes **29.267%**, **37.118%**, and **48.540%** for Taxi trips, FHVHV trips, and Subway respectively. Subway h=5 also retains substantial current-state reliance at **24.656%**.

The Tree champions therefore do not use one fixed forecasting recipe. Shorter-horizon forecasts lean heavily on the immediately preceding mobility sequence, while the five-step forecasts depend more on the current level of activity and recurring same-Daypart patterns. Calendar, Taxi Zone identity, and contextual features remain useful secondary signals rather than the primary source of predictive information.

### 4.6.2.4.3 What Information Drives the Neural Champions?

The Neural family won the three **Taxi average-speed** jobs. Unlike LightGBM, the wider lag-window MLP consumes structured blocks of information rather than a flat list whose individual contributions are naturally summarized with SHAP.

Its upstream grouped-permutation experiment therefore gives the cleaner interpretation. A high reliance share means that breaking the relationship between that input group and the forecast rows caused a large share of the model's measured performance deterioration.

Only the three jobs that actually became final champions are shown here.

In [30]:
# ---------------------------------------------------------------------
# 4.6.2.4.3 — Grouped predictive reliance for Neural champions
# ---------------------------------------------------------------------

if neural_champion_importance_df.duplicated(
    ["metric", "horizon", "input_group"]
).any():
    raise RuntimeError(
        "Neural champion reliance must contain one row "
        "per Job × Input Group."
    )

neural_plot_df = add_job_labels(
    neural_champion_importance_df
)

neural_identifiable_df = neural_plot_df.loc[
    neural_plot_df["permutation_identifiable"].fillna(False)
    & neural_plot_df["normalization_available"].fillna(False)
    & neural_plot_df["reliance_share_pct"].notna()
].copy()

neural_top_idx = (
    neural_identifiable_df
    .groupby(
        ["metric", "horizon"],
        observed=True,
    )["reliance_share_pct"]
    .idxmax()
)

neural_top_group_df = neural_identifiable_df.loc[
    neural_top_idx,
    ["Job", "input_group", "reliance_share_pct"],
].copy()

neural_top_group_df["Job"] = pd.Categorical(
    neural_top_group_df["Job"],
    categories=JOB_LABEL_ORDER,
    ordered=True,
)

neural_top_group_df = (
    neural_top_group_df
    .rename(
        columns={
            "input_group": "Top input group",
            "reliance_share_pct": "Reliance share (%)",
        }
    )
    .sort_values("Job")
    .reset_index(drop=True)
)

display(neural_top_group_df.round(3))

neural_group_order = (
    neural_identifiable_df
    .groupby(
        "input_group",
        observed=True,
    )["reliance_share_pct"]
    .mean()
    .sort_values(ascending=False)
    .index
    .tolist()
)

neural_job_order = [
    job
    for job in JOB_LABEL_ORDER
    if job in set(neural_plot_df["Job"])
]

neural_reliance_matrix_df = (
    neural_plot_df
    .pivot(
        index="input_group",
        columns="Job",
        values="reliance_share_pct",
    )
    .reindex(
        index=neural_group_order,
        columns=neural_job_order,
    )
)

if RUN_VISUALS:
    neural_reliance_fig = px.imshow(
        neural_reliance_matrix_df,
        color_continuous_scale=BRAND_SEQUENTIAL_HEATMAP_SCALE,
        aspect="auto",
        text_auto=".1f",
        labels={
            "x": "Neural champion job",
            "y": "Input group",
            "color": "Reliance share (%)",
        },
    )
    neural_reliance_fig = apply_branding(
        neural_reliance_fig
    )
    neural_reliance_fig.update_layout(
        title=(
            "What information drives the Neural Taxi-speed champions?"
            "<br><sup>Normalized grouped permutation reliance</sup>"
        ),
        height=max(
            460,
            58 * len(neural_reliance_matrix_df) + 170,
        ),
        margin={
            "l": 175,
            "r": 40,
            "t": 90,
            "b": 70,
        },
    )
    neural_reliance_fig.show()

,Job,Top input group,Reliance share (%)
0,Taxi speed · h=1,Recent mobility history,61.986
1,Taxi speed · h=2,Recent mobility history,69.266
2,Taxi speed · h=5,Recent mobility history,49.787


**Findings.** All three Neural Taxi-speed champions rely most heavily on **Recent mobility history**, although the strength of that reliance changes with horizon.

Recent history contributes **61.986% at h=1**, **69.266% at h=2**, and **49.787% at h=5** of normalized grouped reliance. The Neural champions therefore retain a strong recent-history backbone, but the five-step forecast spreads more of its reliance across the remaining calendar and contextual inputs.

This is consistent with the broader forecasting pattern seen elsewhere in the notebook: near-term forecasts can lean heavily on the immediately preceding mobility trajectory, while longer-horizon forecasts benefit from a broader mix of information.

### 4.6.2.4.4 What Information Drives the Transformer Champions?

The Transformer won all three **FHVHV average-speed** horizons plus **Subway h=2**. Its PatchTST-inspired architecture processes recent mobility history differently from the MLP, but the interpretation question is the same: which information groups does the fitted model actually depend on?

The grouped-permutation evidence lets us compare the three closely related FHVHV-speed champions with the structurally weaker Subway h=2 champion without pretending that attention weights themselves are feature importance.

In [31]:
# ---------------------------------------------------------------------
# 4.6.2.4.4 — Grouped predictive reliance for Transformer champions
# ---------------------------------------------------------------------

if transformer_champion_reliance_df.duplicated(
    ["metric", "horizon", "input_group"]
).any():
    raise RuntimeError(
        "Transformer reliance must contain one row "
        "per Job × Input Group."
    )

transformer_plot_df = add_job_labels(
    transformer_champion_reliance_df
)

transformer_identifiable_df = transformer_plot_df.loc[
    transformer_plot_df["permutation_identifiable"].fillna(False)
    & transformer_plot_df["reliance_share_pct"].notna()
].copy()

transformer_top_idx = (
    transformer_identifiable_df
    .groupby(
        ["metric", "horizon"],
        observed=True,
    )["reliance_share_pct"]
    .idxmax()
)

transformer_top_group_df = transformer_identifiable_df.loc[
    transformer_top_idx,
    ["Job", "input_group", "reliance_share_pct"],
].copy()

transformer_top_group_df["Job"] = pd.Categorical(
    transformer_top_group_df["Job"],
    categories=JOB_LABEL_ORDER,
    ordered=True,
)

transformer_top_group_df = (
    transformer_top_group_df
    .rename(
        columns={
            "input_group": "Top input group",
            "reliance_share_pct": "Reliance share (%)",
        }
    )
    .sort_values("Job")
    .reset_index(drop=True)
)

display(transformer_top_group_df.round(3))

transformer_group_order = (
    transformer_identifiable_df
    .groupby(
        "input_group",
        observed=True,
    )["reliance_share_pct"]
    .mean()
    .sort_values(ascending=False)
    .index
    .tolist()
)

transformer_job_order = [
    job
    for job in JOB_LABEL_ORDER
    if job in set(transformer_plot_df["Job"])
]

transformer_reliance_matrix_df = (
    transformer_plot_df
    .pivot(
        index="input_group",
        columns="Job",
        values="reliance_share_pct",
    )
    .reindex(
        index=transformer_group_order,
        columns=transformer_job_order,
    )
)

if RUN_VISUALS:
    transformer_reliance_fig = px.imshow(
        transformer_reliance_matrix_df,
        color_continuous_scale=BRAND_SEQUENTIAL_HEATMAP_SCALE,
        aspect="auto",
        text_auto=".1f",
        labels={
            "x": "Transformer champion job",
            "y": "Input group",
            "color": "Reliance share (%)",
        },
    )
    transformer_reliance_fig = apply_branding(
        transformer_reliance_fig
    )
    transformer_reliance_fig.update_layout(
        title=(
            "What information drives the Transformer champions?"
            "<br><sup>Normalized grouped permutation reliance</sup>"
        ),
        height=max(
            480,
            58 * len(transformer_reliance_matrix_df) + 170,
        ),
        margin={
            "l": 175,
            "r": 40,
            "t": 90,
            "b": 70,
        },
    )
    transformer_reliance_fig.show()

,Job,Top input group,Reliance share (%)
0,FHVHV speed · h=1,Recent mobility history,66.969
1,FHVHV speed · h=2,Recent mobility history,69.353
2,FHVHV speed · h=5,Recent mobility history,74.077
3,Subway · h=2,Recent mobility history,92.126


**Findings.** The Transformer champions are even more strongly centered on recent mobility history.

For FHVHV speed, **Recent mobility history** contributes **66.969% at h=1**, **69.353% at h=2**, and **74.077% at h=5** of normalized grouped reliance. Annual / calendar timing is the strongest consistent secondary source, contributing approximately **13%–19%**, while multimodal context provides a smaller but still meaningful contribution in some horizons.

**Subway h=2 has a notably more concentrated reliance profile.** Recent mobility history accounts for **92.126%** of measured reliance, compared with only **4.301%** for annual/calendar timing and roughly 1% or less for each remaining context group.

That concentration does not by itself explain Subway h=2's weaker forecast reliability; grouped reliance measures what the fitted model uses, not why it makes an error. It does show, however, that the Subway champion depends much more narrowly on its recent ridership history than the FHVHV-speed Transformer champions do.

### 4.6.2.4.5 Identify the Tree Feature Families Worth Ablating

Feature importance tells us where to look; it does not prove that removing a feature family will hurt a newly trained model by the same amount. Correlated predictors can substitute for one another, and a family that appears modest in SHAP may become important once another source of information disappears.

A formal ablation therefore requires a separate experiment: **remove an entire semantic feature family, retrain the otherwise unchanged LightGBM champion, and compare its ordinary-validation accuracy with the intact champion**.

Testing every family would add cost without adding much insight. Before launching that work, we summarize three useful signals across the eight Tree champions: average SHAP share, how broadly the family matters across jobs, and how often it is the single largest family. This evidence will be used to choose a small, interpretable ablation set.

In [32]:
# ---------------------------------------------------------------------
# 4.6.2.4.5 — Rank candidate Tree feature families for formal ablation
# ---------------------------------------------------------------------

tree_family_ranked_df = tree_family_share_df.copy()

job_max_share = (
    tree_family_ranked_df
    .groupby(
        ["metric", "horizon"],
        observed=True,
    )["shap_share"]
    .transform("max")
)

tree_family_ranked_df["is_job_top_family"] = np.isclose(
    tree_family_ranked_df["shap_share"],
    job_max_share,
)
tree_family_ranked_df["share_at_least_10pct"] = (
    tree_family_ranked_df["shap_share"] >= 0.10
)

tree_ablation_candidate_df = (
    tree_family_ranked_df
    .groupby(
        "feature_family",
        observed=True,
        as_index=False,
    )
    .agg(
        champion_jobs=("Job", "nunique"),
        mean_shap_share=("shap_share", "mean"),
        median_shap_share=("shap_share", "median"),
        max_shap_share=("shap_share", "max"),
        top_family_jobs=("is_job_top_family", "sum"),
        jobs_at_least_10pct=("share_at_least_10pct", "sum"),
    )
    .sort_values(
        [
            "mean_shap_share",
            "jobs_at_least_10pct",
            "top_family_jobs",
        ],
        ascending=[False, False, False],
    )
    .reset_index(drop=True)
)

tree_ablation_candidate_display_df = (
    tree_ablation_candidate_df
    .rename(
        columns={
            "feature_family": "Feature family",
            "champion_jobs": "Champion jobs represented",
            "top_family_jobs": "Jobs ranked #1",
            "jobs_at_least_10pct": "Jobs with ≥10% SHAP share",
        }
    )
    .copy()
)

for source, label in [
    ("mean_shap_share", "Mean SHAP share (%)"),
    ("median_shap_share", "Median SHAP share (%)"),
    ("max_shap_share", "Maximum SHAP share (%)"),
]:
    tree_ablation_candidate_display_df[label] = (
        100.0 * tree_ablation_candidate_display_df[source]
    )

tree_ablation_candidate_display_df = (
    tree_ablation_candidate_display_df[
        [
            "Feature family",
            "Champion jobs represented",
            "Mean SHAP share (%)",
            "Median SHAP share (%)",
            "Maximum SHAP share (%)",
            "Jobs ranked #1",
            "Jobs with ≥10% SHAP share",
        ]
    ]
)

display(
    tree_ablation_candidate_display_df.round(3)
)

if RUN_VISUALS:
    tree_candidate_plot_df = (
        tree_ablation_candidate_df
        .assign(
            mean_shap_share_pct=(
                100.0
                * tree_ablation_candidate_df["mean_shap_share"]
            )
        )
        .sort_values("mean_shap_share_pct")
    )

    tree_candidate_fig = px.bar(
        tree_candidate_plot_df,
        x="mean_shap_share_pct",
        y="feature_family",
        orientation="h",
        text="mean_shap_share_pct",
        color_discrete_sequence=[
            BRAND_COLORS["dark_teal"]
        ],
        labels={
            "mean_shap_share_pct":
                "Mean SHAP share across Tree champions (%)",
            "feature_family":
                "Feature family",
        },
    )

    tree_candidate_fig.update_traces(
        texttemplate="%{text:.1f}%",
        textposition="outside",
    )

    tree_candidate_fig = apply_branding(
        tree_candidate_fig
    )
    tree_candidate_fig.update_layout(
        title=(
            "Which feature families are strongest ablation candidates?"
            "<br><sup>Mean SHAP reliance across "
            "the eight Tree champions</sup>"
        ),
        height=max(
            480,
            32 * len(tree_candidate_plot_df) + 180,
        ),
        margin={
            "l": 175,
            "r": 70,
            "t": 90,
            "b": 65,
        },
        showlegend=False,
    )
    tree_candidate_fig.show()

,Feature family,Champion jobs represented,Mean SHAP share (%),Median SHAP share (%),Maximum SHAP share (%),Jobs ranked #1,Jobs with ≥10% SHAP share
0,Recent sequence history,8,35.586,52.977,59.840,5,5
1,Comparable Daypart history,8,17.476,6.966,48.540,2,3
2,Current mobility state,8,16.685,6.537,52.796,1,3
3,Origin calendar,8,8.701,8.064,16.849,0,2
4,Series identity,8,7.974,7.795,15.563,0,3
5,Target calendar,8,7.472,7.030,12.913,0,2
6,multimodal_origin,8,2.179,2.201,4.010,0,0
7,weather,8,1.505,1.207,2.524,0,0
8,transportation_connected,8,1.124,1.018,1.948,0,0
9,Annual calendar cycle,8,0.847,0.789,1.503,0,0


**Findings.** Three Tree feature families stand clearly above the remaining candidates for formal ablation.

**Recent sequence history** is the strongest and broadest candidate. It averages **35.586% of SHAP importance** across the eight Tree champions, ranks first in **5 jobs**, and exceeds 10% importance in those same five jobs.

Two other families capture the complementary information used by the longer-horizon champions. **Comparable Daypart history** averages **17.476%**, ranks first in **2 jobs**, and reaches as high as **48.540%**. **Current mobility state** averages **16.685%**, ranks first in **1 job**, and reaches **52.796%**.

The remaining families are meaningfully smaller. Origin calendar, Series identity, and Target calendar each average roughly **7%–9%**, while every other family averages below 3%.

The strongest ablation candidates therefore also provide useful conceptual contrast: **immediate recent sequence history**, **recurring comparable-Daypart history**, and the **current observed mobility level**. Testing these three families would examine the main information sources supporting both short- and longer-horizon Tree forecasts without expanding the experiment into an exhaustive feature-removal tournament.

### Section Summary

Across the frozen champion system, **recent observed mobility is the central predictive signal**, but different architectures and horizons use that history in different ways. Tree champions rely heavily on immediate sequence history at h=1 and h=2, then shift toward the current mobility state and comparable same-Daypart history at h=5. Neural Taxi-speed champions retain a strong recent-history backbone while drawing more on calendar and contextual information at longer horizons. Transformer champions are similarly history-centered, with Subway h=2 especially concentrated on recent ridership history. Because SHAP and grouped permutation reliance are different measurement systems, their percentages are interpreted within family rather than directly compared. For the Tree branch, **Recent sequence history, Comparable Daypart history, and Current mobility state** emerge as three complementary information sources worth testing with formal retraining ablation.

> ✅ Decision — Ablate three complementary Tree information sources

The formal LightGBM ablation will test three semantic information groups across all eight Tree champion jobs:

\- Recent sequence history — 5 predictors capturing the immediately preceding target trajectory\.
\- Comparable Daypart history — 11 predictors capturing recurring behavior at comparable times\.
\- Current mobility state — \`origin\_value\`, representing the latest observed target level at forecast origin\.

These groups are available with the same predictor inventory across all eight Tree champions and represent three distinct forecasting information sources: where the series is now, how it arrived there, and what typically happens at comparable times\.

Each ablation will remove one complete information source, retrain the otherwise unchanged LightGBM under the frozen validation contract, and compare its accuracy with the intact champion\. The experiment therefore measures how well the remaining predictors can compensate when that source of information is unavailable\.

## 4.6.2.5 Test Tree Information Sources with Formal Ablation

The interpretation evidence above tells us which information the fitted Tree champions **used**. Ablation asks a stronger question:

> **Does forecast accuracy actually suffer when that information is unavailable?**

This distinction matters because LightGBM has many correlated historical predictors. A feature group can receive substantial SHAP importance while other predictors are still able to substitute for it after retraining.

The experiment therefore removes one complete information source at a time, retrains the otherwise unchanged LightGBM, and evaluates it on the same ordinary-validation forecast opportunities used by the intact model.

Everything else remains frozen: **Full-History Regime-Aware training, the selected 400-tree LightGBM definition, raw target representation, validation-start-only fitting, complete legally observable training history, and the final nonnegative output rule**. The January–March 2026 final holdout remains outside the experiment.


**4.6.2.5.1A — Freeze exactly what each information-source ablation removes.**

An ablation is only interpretable if the reader can see exactly what has been taken away from the model.

The three experiments represent complementary forecasting questions:

- **Current mobility state:** Where is the series now?
- **Recent sequence history:** How did it just get here?
- **Comparable Daypart history:** What does comparable recent history look like?

The last group is intentionally broader than simple “seasonality.” It includes same-Daypart lags and changes, recent historical levels and variability, exponentially weighted history, and indicators describing how much usable recent history was actually observed.

The code below derives the membership from the authoritative forecasting feature taxonomy, checks it against the exact 85-predictor LightGBM roster, and then publishes a compact reader-facing inventory.

In [33]:
# 4.6.2.5.1A — Freeze the Tree ablation feature contract
# ---------------------------------------------------------------------

TREE_ABLATION_SOURCE_ORDER = [
    "Recent sequence history",
    "Comparable Daypart history",
    "Current mobility state",
]

TREE_ABLATION_SOURCE_QUESTIONS = {
    "Recent sequence history": "How did the series just get here?",
    "Comparable Daypart history": "What does comparable recent history look like?",
    "Current mobility state": "Where is the series now?",
}

TREE_ABLATION_SOURCE_DESCRIPTIONS = {
    "Recent sequence history": (
        "Immediate sequence lags plus the latest step-to-step change."
    ),
    "Comparable Daypart history": (
        "Same-Daypart lags and change, rolling and exponentially weighted "
        "levels, recent variability, and historical observation support."
    ),
    "Current mobility state": (
        "The latest observed target value available when the forecast is issued."
    ),
}

EXPECTED_TREE_ABLATION_FEATURE_COUNTS = {
    "Recent sequence history": 5,
    "Comparable Daypart history": 11,
    "Current mobility state": 1,
}


# ---------------------------------------------------------------------
# Recover the eight Tree-owned champion jobs
# ---------------------------------------------------------------------
# Champion ownership is already frozen in 4.6.1. The ablation tests only the
# jobs where LightGBM actually belongs to the final forecasting system.

tree_champion_jobs_df = (
    champion_registry_df.loc[
        champion_registry_df["champion_family"].eq("tree"),
        ["metric", "horizon"],
    ]
    .drop_duplicates()
    .sort_values(["metric", "horizon"])
    .reset_index(drop=True)
)

if len(tree_champion_jobs_df) != 8:
    raise RuntimeError(
        f"Expected 8 Tree champion jobs; found {len(tree_champion_jobs_df)}."
    )

tree_champion_metrics = sorted(
    tree_champion_jobs_df["metric"].unique()
)


# ---------------------------------------------------------------------
# Recover the exact 85-predictor LightGBM input roster
# ---------------------------------------------------------------------
# The predictor registry is the executable feature contract. Checking the
# taxonomy against this roster prevents a feature-family label from describing
# an input that the selected models never actually received.

tree_predictor_registry_df = pd.read_parquet(
    require_file(
        TREE_PREDICTOR_REGISTRY_PATH,
        "LightGBM predictor registry",
    ),
    columns=[
        "metric",
        "predictor_order",
        "predictor_name",
        "is_categorical",
    ],
)

require_columns(
    tree_predictor_registry_df,
    [
        "metric",
        "predictor_order",
        "predictor_name",
        "is_categorical",
    ],
    "LightGBM predictor registry",
)

predictor_counts = (
    tree_predictor_registry_df
    .groupby("metric", observed=True)["predictor_name"]
    .nunique()
)

if not predictor_counts.eq(85).all():
    raise RuntimeError(
        "Every LightGBM target must contain exactly 85 frozen predictors."
    )


# ---------------------------------------------------------------------
# Attach the semantic feature family already established upstream
# ---------------------------------------------------------------------

tree_predictor_family_df = (
    tree_predictor_registry_df
    .rename(columns={"predictor_name": "feature"})
    .merge(
        feature_family_lookup_df,
        on="feature",
        how="left",
        validate="many_to_one",
    )
)

if tree_predictor_family_df["feature_family"].isna().any():
    missing_features = sorted(
        tree_predictor_family_df.loc[
            tree_predictor_family_df["feature_family"].isna(),
            "feature",
        ].unique()
    )
    raise RuntimeError(
        "The frozen LightGBM predictor roster contains features without "
        f"semantic family labels: {missing_features}"
    )


# ---------------------------------------------------------------------
# Confirm that every ablation has the same meaning across Tree targets
# ---------------------------------------------------------------------
# Taxi trips, FHVHV trips, and Subway may share an ablation label only when
# that label removes the same named inputs from every affected model.

source_feature_lists = {}

for source in TREE_ABLATION_SOURCE_ORDER:
    metric_feature_lists = {}

    for metric in tree_champion_metrics:
        metric_source_df = (
            tree_predictor_family_df.loc[
                tree_predictor_family_df["metric"].eq(metric)
                & tree_predictor_family_df["feature_family"].eq(source)
            ]
            .sort_values("predictor_order")
        )

        metric_feature_lists[metric] = tuple(
            metric_source_df["feature"]
        )

    unique_memberships = set(metric_feature_lists.values())

    if len(unique_memberships) != 1:
        raise RuntimeError(
            f"{source} does not remove the same predictors from every "
            f"Tree-owned target: {metric_feature_lists}"
        )

    source_feature_lists[source] = list(
        unique_memberships.pop()
    )

    expected_count = EXPECTED_TREE_ABLATION_FEATURE_COUNTS[source]
    observed_count = len(source_feature_lists[source])

    if observed_count != expected_count:
        raise RuntimeError(
            f"{source}: expected {expected_count} predictors; "
            f"found {observed_count}."
        )

# Current mobility state is intentionally a one-feature information source.
# Guard that meaning explicitly so the test cannot drift into a broader group.
if source_feature_lists["Current mobility state"] != ["origin_value"]:
    raise RuntimeError(
        "Current mobility state must remove origin_value and nothing else."
    )

all_ablation_features = [
    feature
    for source in TREE_ABLATION_SOURCE_ORDER
    for feature in source_feature_lists[source]
]

if len(all_ablation_features) != len(set(all_ablation_features)):
    raise RuntimeError(
        "Tree ablation information sources overlap. "
        "Each removed predictor must belong to exactly one experiment."
    )


# ---------------------------------------------------------------------
# Preserve a durable one-row-per-feature experiment contract
# ---------------------------------------------------------------------
# The machine-readable contract remains normalized even though the notebook
# presents only three reader-facing experiment rows.

feature_contract_rows = []

for source_order, source in enumerate(
    TREE_ABLATION_SOURCE_ORDER,
    start=1,
):
    ablation_id = (
        "drop_"
        + source.lower().replace(" ", "_")
    )

    for feature_order, feature in enumerate(
        source_feature_lists[source],
        start=1,
    ):
        feature_contract_rows.append(
            {
                "ablation_contract_id": TREE_ABLATION_CONTRACT_ID,
                "ablation_id": ablation_id,
                "source_order": source_order,
                "information_source": source,
                "forecasting_question":
                    TREE_ABLATION_SOURCE_QUESTIONS[source],
                "source_description":
                    TREE_ABLATION_SOURCE_DESCRIPTIONS[source],
                "feature_order": feature_order,
                "feature_name": feature,
            }
        )

tree_ablation_feature_contract_df = pd.DataFrame(
    feature_contract_rows
)

if len(tree_ablation_feature_contract_df) != 17:
    raise RuntimeError(
        "The formal Tree ablation contract must contain exactly "
        "17 source-feature rows."
    )

if WRITE_INTERMEDIATES:
    atomic_write_parquet(
        tree_ablation_feature_contract_df,
        TREE_ABLATION_FEATURE_CONTRACT_PATH,
    )


# ---------------------------------------------------------------------
# Reader-facing rendering
# ---------------------------------------------------------------------
# One row per experiment is easier to understand than a 17-row raw registry,
# while the full feature list remains visible for reproducibility.

tree_ablation_inventory_df = (
    tree_ablation_feature_contract_df
    .sort_values(["source_order", "feature_order"])
    .groupby(
        [
            "source_order",
            "information_source",
            "forecasting_question",
            "source_description",
        ],
        observed=True,
        as_index=False,
    )
    .agg(
        **{
            "Feature count": ("feature_name", "nunique"),
            "Features removed": (
                "feature_name",
                lambda values: "\n".join(
                    f"• {value}" for value in values
                ),
            ),
        }
    )
    .sort_values("source_order")
    .rename(
        columns={
            "information_source": "Information source",
            "forecasting_question": "Forecasting question",
            "source_description": "What the model loses",
        }
    )
    .drop(columns="source_order")
)

tree_ablation_inventory_style = (
    tree_ablation_inventory_df.style
    .hide(axis="index")
    .set_properties(
        subset=[
            "Information source",
            "Forecasting question",
            "What the model loses",
        ],
        **{
            "text-align": "left",
            "vertical-align": "top",
            "white-space": "normal",
        },
    )
    .set_properties(
        subset=["Features removed"],
        **{
            "text-align": "left",
            "vertical-align": "top",
            "white-space": "pre-wrap",
            "font-family": "monospace",
        },
    )
    .set_properties(
        subset=["Feature count"],
        **{
            "text-align": "center",
            "vertical-align": "top",
        },
    )
)

display(tree_ablation_inventory_style)


Information source,Forecasting question,What the model loses,Feature count,Features removed
Recent sequence history,How did the series just get here?,Immediate sequence lags plus the latest step-to-step change.,5,• seq_delta_1 • seq_lag_1 • seq_lag_2 • seq_lag_3 • seq_lag_4
Comparable Daypart history,What does comparable recent history look like?,"Same-Daypart lags and change, rolling and exponentially weighted levels, recent variability, and historical observation support.",11,• same_daypart_delta_1d • same_daypart_ewm_mean_7d • same_daypart_lag_1d • same_daypart_lag_2d • same_daypart_lag_7d • same_daypart_roll_mean_28d • same_daypart_roll_mean_7d • same_daypart_roll_obs_share_28d • same_daypart_roll_obs_share_7d • same_daypart_roll_std_28d • same_daypart_roll_std_7d
Current mobility state,Where is the series now?,The latest observed target value available when the forecast is issued.,1,• origin_value


**Findings.** The formal ablation contract contains **three non-overlapping information sources and 17 predictors**. Their membership is identical across every target represented by the eight Tree champions.

**Recent sequence history** removes 5 predictors describing the immediately preceding trajectory. **Comparable Daypart history** removes 11 predictors describing comparable historical behavior and the amount of usable history supporting those summaries. **Current mobility state** removes only `origin_value`, the latest observed value available when the forecast is issued.

The Comparable Daypart experiment is therefore broader than a simple seasonality test. It asks whether the Tree model still performs well after losing its engineered reference for what normally happened at comparable recent times.

**4.6.2.5.1B — Verify that the ablation changes only the information source being removed.**

Before fitting anything, we validate the experimental workload against the selected LightGBM handoff.

This is a contract check rather than another analytical decision. Every ablated model must inherit the same model family, Full-History track, raw target representation, 400-tree specification, validation-start-only cadence, unrestricted legal training history, and final output rule used by the intact Tree candidate.

Crossing the eight Tree champion jobs with the three approved information sources produces **24 ablated fits**. The intact comparison requires no additional fit because its row-level ordinary-validation predictions are already saved upstream.

In [34]:
# 4.6.2.5.1B — Freeze and validate the 24-fit ablation workload
# ---------------------------------------------------------------------

tree_selected_manifest_df = pd.read_parquet(
    require_file(
        TREE_SELECTED_MANIFEST_PATH,
        "selected LightGBM manifest",
    )
)

require_columns(
    tree_selected_manifest_df,
    ["path"],
    "selected LightGBM manifest",
)

# The manifest is the producer-owned inventory. Require every artifact used by
# the ablation to appear there rather than trusting local path assumptions.
manifest_paths = {
    Path(str(path))
    for path in tree_selected_manifest_df["path"].dropna()
}

required_tree_paths = [
    TREE_SELECTED_MODEL_REGISTRY_PATH,
    TREE_SELECTED_VALIDATION_PREDICTIONS_PATH,
    TREE_PREDICTOR_REGISTRY_PATH,
]

missing_manifest_paths = [
    str(path)
    for path in required_tree_paths
    if path not in manifest_paths
]

if missing_manifest_paths:
    raise RuntimeError(
        "Selected LightGBM manifest does not contain required ablation "
        f"inputs: {missing_manifest_paths}"
    )


# ---------------------------------------------------------------------
# Recover the exact selected LightGBM model definition
# ---------------------------------------------------------------------

tree_selected_model_registry_df = pd.read_parquet(
    require_file(
        TREE_SELECTED_MODEL_REGISTRY_PATH,
        "selected LightGBM model registry",
    )
)

tree_model_required_columns = [
    "modeling_track",
    "metric",
    "horizon",
    "model_family",
    "capacity_profile",
    "target_representation",
    "refit_cadence",
    "training_row_cap",
    "validation_start_origin_id",
    "train_rows",
    "validation_rows",
    "mae",
    "final_prediction_rule",
    "raw_prediction_preserved",
    "final_holdout_scored",
    "param_n_estimators",
]

require_columns(
    tree_selected_model_registry_df,
    tree_model_required_columns,
    "selected LightGBM model registry",
)

# Only the eight jobs that actually became final Tree champions participate.
tree_ablation_model_contract_df = tree_champion_jobs_df.merge(
    tree_selected_model_registry_df,
    on=["metric", "horizon"],
    how="left",
    validate="one_to_one",
)

if len(tree_ablation_model_contract_df) != 8:
    raise RuntimeError(
        "The selected LightGBM registry did not resolve all 8 Tree champions."
    )

if tree_ablation_model_contract_df["model_family"].isna().any():
    raise RuntimeError(
        "At least one Tree champion is absent from the selected "
        "LightGBM model registry."
    )


# ---------------------------------------------------------------------
# Guard methodology inherited from the selected 4.4.1 candidate
# ---------------------------------------------------------------------
# These assertions prevent a feature-removal test from accidentally becoming
# a model-definition, training-track, or target-representation experiment.

expected_contract_values = {
    "modeling_track": "full_history_regime_aware",
    "model_family": "LightGBM",
    "capacity_profile": "more_expressive",
    "target_representation": "raw",
    "refit_cadence": "validation_start_only",
    "final_prediction_rule": "max(0, raw_prediction)",
}

for column, expected_value in expected_contract_values.items():
    if not tree_ablation_model_contract_df[column].eq(
        expected_value
    ).all():
        observed_values = sorted(
            tree_ablation_model_contract_df[column]
            .dropna()
            .astype(str)
            .unique()
        )
        raise RuntimeError(
            f"Tree ablation requires {column}={expected_value}; "
            f"found {observed_values}."
        )

if not tree_ablation_model_contract_df[
    "param_n_estimators"
].eq(400).all():
    raise RuntimeError(
        "The frozen Tree candidate must use 400 boosting rounds."
    )

if not tree_ablation_model_contract_df[
    "training_row_cap"
].isna().all():
    raise RuntimeError(
        "Formal Tree ablation must preserve the complete legal training "
        "history; a training-row cap was unexpectedly found."
    )

if not tree_ablation_model_contract_df[
    "raw_prediction_preserved"
].all():
    raise RuntimeError(
        "The selected Tree contract must preserve raw predictions."
    )

if tree_ablation_model_contract_df[
    "final_holdout_scored"
].any():
    raise RuntimeError(
        "The selected Tree development contract unexpectedly marks the "
        "final holdout as scored."
    )


# ---------------------------------------------------------------------
# Recover the complete selected LightGBM parameter dictionary
# ---------------------------------------------------------------------
# 4.4.1 exports every selected hyperparameter as param_* columns. Reading them
# from the registry keeps 4.6.2 from independently rebuilding the model spec.

tree_param_columns = sorted(
    column
    for column in tree_selected_model_registry_df.columns
    if column.startswith("param_")
)

if not tree_param_columns:
    raise RuntimeError(
        "Selected LightGBM registry contains no frozen param_* columns."
    )

parameter_variation = (
    tree_ablation_model_contract_df[tree_param_columns]
    .nunique(dropna=False)
)

if parameter_variation.gt(1).any():
    varying_parameters = parameter_variation.loc[
        parameter_variation.gt(1)
    ].index.tolist()

    raise RuntimeError(
        "Tree champions do not share one frozen LightGBM definition. "
        f"Varying parameters: {varying_parameters}"
    )

tree_parameter_row = (
    tree_ablation_model_contract_df[tree_param_columns]
    .iloc[0]
)

TREE_ABLATION_LGBM_PARAMS = {}

for column, value in tree_parameter_row.items():
    if pd.isna(value):
        continue

    if isinstance(value, np.generic):
        value = value.item()

    TREE_ABLATION_LGBM_PARAMS[
        column.removeprefix("param_")
    ] = value


# ---------------------------------------------------------------------
# Cross 8 champion jobs with 3 approved information-source removals
# ---------------------------------------------------------------------

ablation_source_workload_df = (
    tree_ablation_feature_contract_df
    .groupby(
        ["source_order", "ablation_id", "information_source"],
        observed=True,
        as_index=False,
    )
    .agg(features_removed=("feature_name", "nunique"))
)

tree_ablation_workload_df = (
    tree_ablation_model_contract_df[
        [
            "metric",
            "horizon",
            "modeling_track",
            "capacity_profile",
            "target_representation",
            "refit_cadence",
            "validation_start_origin_id",
            "train_rows",
            "validation_rows",
        ]
    ]
    .merge(ablation_source_workload_df, how="cross")
    .sort_values(["metric", "horizon", "source_order"])
    .reset_index(drop=True)
)

tree_ablation_workload_df["base_predictors"] = 85
tree_ablation_workload_df["predictors_retained"] = (
    tree_ablation_workload_df["base_predictors"]
    - tree_ablation_workload_df["features_removed"]
)
tree_ablation_workload_df["ablation_contract_id"] = (
    TREE_ABLATION_CONTRACT_ID
)

tree_ablation_workload_df["logical_unit_id"] = (
    tree_ablation_workload_df["metric"].astype(str)
    + "__h"
    + tree_ablation_workload_df["horizon"].astype(int).astype(str)
    + "__"
    + tree_ablation_workload_df["ablation_id"]
)

if len(tree_ablation_workload_df) != 24:
    raise RuntimeError(
        f"Expected 24 Tree ablation units; "
        f"found {len(tree_ablation_workload_df)}."
    )

if tree_ablation_workload_df["logical_unit_id"].duplicated().any():
    raise RuntimeError(
        "Tree ablation workload contains duplicate logical units."
    )

expected_retained_counts = {
    "Recent sequence history": 80,
    "Comparable Daypart history": 74,
    "Current mobility state": 84,
}

for source, expected_count in expected_retained_counts.items():
    observed_counts = (
        tree_ablation_workload_df.loc[
            tree_ablation_workload_df["information_source"].eq(source),
            "predictors_retained",
        ]
        .unique()
    )

    if (
        len(observed_counts) != 1
        or int(observed_counts[0]) != expected_count
    ):
        raise RuntimeError(
            f"{source}: expected {expected_count} retained predictors; "
            f"found {observed_counts.tolist()}."
        )

if WRITE_INTERMEDIATES:
    atomic_write_parquet(
        tree_ablation_workload_df,
        TREE_ABLATION_WORKLOAD_PATH,
    )


# ---------------------------------------------------------------------
# Reader-facing workload summary
# ---------------------------------------------------------------------

tree_ablation_workload_display_df = (
    tree_ablation_workload_df
    .groupby(
        ["source_order", "information_source"],
        observed=True,
        as_index=False,
    )
    .agg(
        **{
            "Features removed": ("features_removed", "first"),
            "Predictors retained": ("predictors_retained", "first"),
            "Tree champion jobs": ("logical_unit_id", "nunique"),
        }
    )
    .sort_values("source_order")
    .rename(
        columns={"information_source": "Information source"}
    )
    .drop(columns="source_order")
)

tree_ablation_workload_display_df["Planned ablated fits"] = (
    tree_ablation_workload_display_df["Tree champion jobs"]
)

display(
    tree_ablation_workload_display_df.style
    .hide(axis="index")
    .set_properties(**{"text-align": "center"})
    .set_properties(
        subset=["Information source"],
        **{
            "text-align": "left",
            "font-weight": "600",
        },
    )
)

print(
    "Tree ablation contract validated\n"
    f"  Tree champion jobs:    {len(tree_champion_jobs_df)}\n"
    f"  Information sources:   {len(TREE_ABLATION_SOURCE_ORDER)}\n"
    f"  Planned ablated fits:  {len(tree_ablation_workload_df)}\n"
    "  Intact baseline fits: 0 — saved validation forecasts are reused\n"
    "  Final holdout used:   No",
    flush=True,
)


Information source,Features removed,Predictors retained,Tree champion jobs,Planned ablated fits
Recent sequence history,5,80,8,8
Comparable Daypart history,11,74,8,8
Current mobility state,1,84,8,8


Tree ablation contract validated
  Tree champion jobs:    8
  Information sources:   3
  Planned ablated fits:  24
  Intact baseline fits: 0 — saved validation forecasts are reused
  Final holdout used:   No


**Findings.** The formal experiment contains **24 ablated LightGBM fits: 8 Tree champion jobs × 3 information-source removals**.

The preflight confirms that every experiment preserves the selected Full-History, raw-target, 400-tree, validation-start-only LightGBM contract and its complete legally observable training history. Removing Recent sequence history leaves 80 predictors, removing Comparable Daypart history leaves 74, and removing Current mobility state leaves 84.

The intact models are not retrained. Their saved ordinary-validation forecasts provide the reference result on the same forecast opportunities, so the experiment isolates the information source being withheld rather than mixing feature removal with a second intact-model training run.

### 4.6.2.5.2 Retrain the Tree Champions Without One Information Source

Each ablated model is reconstructed from the same inputs and legality rules used by the selected LightGBM ordinary-validation run.

For one Target × Horizon job, the workflow is:

1. load only that job's Full-History initial-training and rolling-validation matrix rows;
2. recover the exact 85-predictor order and categorical treatment from the frozen predictor registry;
3. align validation to the already-saved intact LightGBM forecasts;
4. fit using only outcomes observable at the start of validation;
5. remove one information source from both training and scoring inputs;
6. apply the same final `max(0, raw prediction)` rule;
7. compare the ablated forecast with the intact forecast on identical rows.

The saved intact raw MAE and training/validation row counts are reconstructed from these inputs before any ablation result is accepted. If those checks do not reproduce the 4.4.1 handoff, execution stops.

**4.6.2.5.2A — Build the exact training and scoring helpers for the ablation.**

The helpers below reproduce the selected Tree branch's temporal legality, categorical treatment, prediction rule, and validation denominator.

This block defines methodology only; it does not fit a model. The key safeguard is that each ablated model is compared against the intact model on the exact same saved ordinary-validation rows.


In [35]:
# 4.6.2.5.2A — Build exact Tree-ablation training/scoring helpers
# ---------------------------------------------------------------------

TREE_ABLATION_ROW_KEYS = [
    "forecast_sequence_id",
    "taxi_zone_id",
    "origin_observation_sequence_id",
    "target_observation_sequence_id",
    "horizon",
]


def tree_ablation_checkpoint_path(
    logical_unit_id: str,
) -> Path:
    """Return the durable one-row result path for one ablation unit."""
    return (
        TREE_ABLATION_PARTS_DIR
        / f"{logical_unit_id}.parquet"
    )


def prepare_tree_ablation_frames(
    X_train: pd.DataFrame,
    X_valid: pd.DataFrame,
    categorical_names: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Reproduce training-only categorical preparation for LightGBM.

    Categories are learned from training only. Validation-only values map to
    __UNSEEN__ so future category knowledge cannot leak into model preparation.
    """
    train = X_train.copy()
    valid = X_valid.copy()

    for column in categorical_names:
        train_values = (
            train[column]
            .astype("string")
            .fillna("__MISSING__")
        )
        valid_values = (
            valid[column]
            .astype("string")
            .fillna("__MISSING__")
        )

        categories = (
            train_values
            .drop_duplicates()
            .tolist()
        )

        if "__UNSEEN__" not in categories:
            categories.append("__UNSEEN__")

        valid_values = valid_values.where(
            valid_values.isin(categories),
            "__UNSEEN__",
        )

        train[column] = pd.Categorical(
            train_values,
            categories=categories,
        )
        valid[column] = pd.Categorical(
            valid_values,
            categories=categories,
        )

    return train, valid


def mean_absolute_error_462(
    actual: np.ndarray,
    prediction: np.ndarray,
) -> float:
    """Return native-unit MAE on one common forecast population."""
    return float(
        np.mean(np.abs(prediction - actual))
    )


def load_tree_ablation_job(
    metric: str,
    horizon: int,
) -> dict:
    """
    Recover one champion's exact legal training and validation surfaces.

    Saved intact validation predictions define the denominator. Matrix rows
    supply the 85 predictors and the legal pre-validation training history.
    """
    horizon = int(horizon)

    model_match = tree_ablation_model_contract_df.loc[
        tree_ablation_model_contract_df["metric"].eq(metric)
        & tree_ablation_model_contract_df["horizon"].eq(horizon)
    ]

    if len(model_match) != 1:
        raise RuntimeError(
            f"{metric} · h={horizon}: expected one selected model row; "
            f"found {len(model_match)}."
        )

    model_row = model_match.iloc[0]
    track = str(model_row["modeling_track"])

    predictor_rows = (
        tree_predictor_registry_df.loc[
            tree_predictor_registry_df["metric"].eq(metric)
        ]
        .sort_values("predictor_order")
    )

    predictor_names = predictor_rows["predictor_name"].tolist()
    categorical_names = predictor_rows.loc[
        predictor_rows["is_categorical"],
        "predictor_name",
    ].tolist()

    if len(predictor_names) != 85:
        raise RuntimeError(
            f"{metric}: expected 85 LightGBM predictors; "
            f"found {len(predictor_names)}."
        )

    matrix_path = require_file(
        FORECASTING_MATRIX_PATHS[track],
        f"{track} forecasting matrix",
    )

    matrix_core_columns = [
        "modeling_track",
        "stage",
        "forecast_sequence_id",
        "taxi_zone_id",
        "metric",
        "origin_observation_sequence_id",
        "target_observation_sequence_id",
        "horizon",
        "target_observed_value",
        "target_label_available",
    ]

    matrix_columns = list(
        dict.fromkeys(
            matrix_core_columns + predictor_names
        )
    )

    # Schema inspection is cheap and prevents an expensive fit from starting
    # against a forecasting matrix that no longer matches the frozen contract.
    available_matrix_columns = set(
        pq.ParquetFile(matrix_path)
        .schema_arrow
        .names
    )
    missing_matrix_columns = sorted(
        set(matrix_columns) - available_matrix_columns
    )

    if missing_matrix_columns:
        raise RuntimeError(
            f"{metric} · h={horizon}: forecasting matrix is missing "
            f"{missing_matrix_columns}."
        )

    matrix_df = pd.read_parquet(
        matrix_path,
        columns=matrix_columns,
        filters=[
            ("metric", "==", metric),
            ("horizon", "==", horizon),
            (
                "stage",
                "in",
                ["initial_training", "rolling_validation"],
            ),
        ],
    )

    if not matrix_df["modeling_track"].eq(track).all():
        raise RuntimeError(
            f"{metric} · h={horizon}: matrix returned rows from "
            "another modeling track."
        )

    for column in [
        "origin_observation_sequence_id",
        "target_observation_sequence_id",
    ]:
        matrix_df[column] = pd.to_numeric(
            matrix_df[column],
            errors="raise",
        ).astype("int64")

    # A row is legal training evidence only when the target was explicitly
    # available and actually observed. Temporal legality is added below.
    matrix_df["_label_available"] = (
        matrix_df["target_label_available"]
        .fillna(False)
        .astype(bool)
        & matrix_df["target_observed_value"].notna()
    )


    # -----------------------------------------------------------------
    # Recover the exact intact-model validation denominator
    # -----------------------------------------------------------------

    intact_columns = [
        "modeling_track",
        "stage",
        "metric",
        "horizon",
        "forecast_sequence_id",
        "taxi_zone_id",
        "origin_observation_sequence_id",
        "target_observation_sequence_id",
        "actual",
        "prediction_raw",
        "prediction_final",
        "benchmark_prediction",
        "model_prediction_available",
        "benchmark_prediction_available",
    ]

    intact_validation_df = pd.read_parquet(
        require_file(
            TREE_SELECTED_VALIDATION_PREDICTIONS_PATH,
            "selected LightGBM validation predictions",
        ),
        columns=intact_columns,
        filters=[
            ("modeling_track", "==", track),
            ("metric", "==", metric),
            ("horizon", "==", horizon),
            ("stage", "==", "rolling_validation"),
        ],
    )

    if intact_validation_df.empty:
        raise RuntimeError(
            f"{metric} · h={horizon}: saved intact validation "
            "surface is empty."
        )

    if not intact_validation_df[
        "model_prediction_available"
    ].all():
        raise RuntimeError(
            f"{metric} · h={horizon}: intact validation contains "
            "a missing LightGBM prediction."
        )

    if not intact_validation_df[
        "benchmark_prediction_available"
    ].all():
        raise RuntimeError(
            f"{metric} · h={horizon}: intact validation contains "
            "a missing benchmark prediction."
        )

    numeric_validation_columns = [
        "actual",
        "prediction_raw",
        "prediction_final",
        "benchmark_prediction",
    ]

    if not np.isfinite(
        intact_validation_df[
            numeric_validation_columns
        ].to_numpy(dtype=float)
    ).all():
        raise RuntimeError(
            f"{metric} · h={horizon}: intact validation contains "
            "non-finite prediction evidence."
        )

    if intact_validation_df.duplicated(
        TREE_ABLATION_ROW_KEYS
    ).any():
        raise RuntimeError(
            f"{metric} · h={horizon}: saved validation keys "
            "are not unique."
        )


    # -----------------------------------------------------------------
    # Attach predictor rows to the exact saved validation population
    # -----------------------------------------------------------------

    matrix_validation_df = matrix_df.loc[
        matrix_df["stage"].eq("rolling_validation")
        & matrix_df["_label_available"]
    ].copy()

    if matrix_validation_df.duplicated(
        TREE_ABLATION_ROW_KEYS
    ).any():
        raise RuntimeError(
            f"{metric} · h={horizon}: matrix validation keys "
            "are not unique."
        )

    validation_df = matrix_validation_df.merge(
        intact_validation_df[
            TREE_ABLATION_ROW_KEYS
            + [
                "actual",
                "prediction_raw",
                "prediction_final",
                "benchmark_prediction",
            ]
        ],
        on=TREE_ABLATION_ROW_KEYS,
        how="inner",
        validate="one_to_one",
    )

    if len(validation_df) != len(intact_validation_df):
        raise RuntimeError(
            f"{metric} · h={horizon}: predictor alignment changed "
            "the saved validation denominator."
        )

    if not np.allclose(
        validation_df["target_observed_value"].to_numpy(
            dtype=float
        ),
        validation_df["actual"].to_numpy(dtype=float),
        rtol=1e-10,
        atol=1e-10,
    ):
        raise RuntimeError(
            f"{metric} · h={horizon}: matrix outcomes disagree "
            "with the saved intact validation artifact."
        )


    # -----------------------------------------------------------------
    # Reproduce validation-start-only training legality
    # -----------------------------------------------------------------
    # A training target is legal only when it is known by the first validation
    # origin. This reproduces the selected one-fit-at-validation-start policy.

    validation_start_origin_id = int(
        validation_df[
            "origin_observation_sequence_id"
        ].min()
    )

    expected_validation_start = int(
        model_row["validation_start_origin_id"]
    )

    if validation_start_origin_id != expected_validation_start:
        raise RuntimeError(
            f"{metric} · h={horizon}: validation start changed. "
            f"Expected {expected_validation_start}; "
            f"found {validation_start_origin_id}."
        )

    training_mask = (
        matrix_df["_label_available"]
        & matrix_df[
            "target_observation_sequence_id"
        ].le(validation_start_origin_id)
    )

    train_df = matrix_df.loc[training_mask].copy()

    if len(train_df) != int(model_row["train_rows"]):
        raise RuntimeError(
            f"{metric} · h={horizon}: legal training rows changed. "
            f"Expected {int(model_row['train_rows']):,}; "
            f"found {len(train_df):,}."
        )

    if len(validation_df) != int(model_row["validation_rows"]):
        raise RuntimeError(
            f"{metric} · h={horizon}: validation rows changed. "
            f"Expected {int(model_row['validation_rows']):,}; "
            f"found {len(validation_df):,}."
        )


    # -----------------------------------------------------------------
    # Reproduce the saved intact score before allowing ablation
    # -----------------------------------------------------------------

    actual = validation_df["actual"].to_numpy(dtype=float)
    intact_raw = validation_df[
        "prediction_raw"
    ].to_numpy(dtype=float)
    intact_final = validation_df[
        "prediction_final"
    ].to_numpy(dtype=float)
    benchmark = validation_df[
        "benchmark_prediction"
    ].to_numpy(dtype=float)

    expected_final = np.maximum(intact_raw, 0.0)

    if not np.allclose(
        intact_final,
        expected_final,
        rtol=0.0,
        atol=0.0,
    ):
        raise RuntimeError(
            f"{metric} · h={horizon}: saved final predictions "
            "do not match max(0, raw prediction)."
        )

    intact_raw_mae = mean_absolute_error_462(
        actual,
        intact_raw,
    )

    if not np.isclose(
        intact_raw_mae,
        float(model_row["mae"]),
        rtol=1e-10,
        atol=1e-10,
    ):
        raise RuntimeError(
            f"{metric} · h={horizon}: reconstructed raw MAE "
            "does not reproduce the selected Tree registry."
        )

    return {
        "metric": metric,
        "horizon": horizon,
        "modeling_track": track,
        "predictor_names": predictor_names,
        "categorical_names": categorical_names,
        "train_df": train_df,
        "validation_df": validation_df,
        "intact_raw_mae": intact_raw_mae,
        "intact_final_mae": mean_absolute_error_462(
            actual,
            intact_final,
        ),
        "benchmark_mae": mean_absolute_error_462(
            actual,
            benchmark,
        ),
    }


def fit_tree_ablation_unit(
    payload: dict,
    workload_row,
) -> dict:
    """
    Retrain one LightGBM after removing one complete information source.

    Remaining predictors are allowed to adapt, so the measured MAE change
    reflects predictive information they could not replace after retraining.
    """
    source = workload_row.information_source

    removed_features = (
        tree_ablation_feature_contract_df.loc[
            tree_ablation_feature_contract_df[
                "information_source"
            ].eq(source)
        ]
        .sort_values("feature_order")["feature_name"]
        .tolist()
    )

    missing_removed_features = sorted(
        set(removed_features)
        - set(payload["predictor_names"])
    )

    if missing_removed_features:
        raise RuntimeError(
            f"{payload['metric']} · h={payload['horizon']} · "
            f"{source}: missing removal features "
            f"{missing_removed_features}."
        )

    retained_predictors = [
        feature
        for feature in payload["predictor_names"]
        if feature not in removed_features
    ]
    retained_categorical = [
        feature
        for feature in payload["categorical_names"]
        if feature in retained_predictors
    ]

    if (
        len(retained_predictors)
        != int(workload_row.predictors_retained)
    ):
        raise RuntimeError(
            f"{payload['metric']} · h={payload['horizon']} · "
            f"{source}: retained-predictor count differs from "
            "the frozen workload."
        )

    X_train, X_valid = prepare_tree_ablation_frames(
        payload["train_df"][retained_predictors],
        payload["validation_df"][retained_predictors],
        retained_categorical,
    )

    y_train = pd.to_numeric(
        payload["train_df"]["target_observed_value"],
        errors="raise",
    ).astype(float)

    actual = payload["validation_df"][
        "actual"
    ].to_numpy(dtype=float)
    benchmark = payload["validation_df"][
        "benchmark_prediction"
    ].to_numpy(dtype=float)

    # Hyperparameters come directly from the selected-model registry.
    # Removing predictors is therefore the only model-definition change.
    model = LGBMRegressor(
        **TREE_ABLATION_LGBM_PARAMS
    )

    fit_start = perf_counter()

    model.fit(
        X_train,
        y_train,
        categorical_feature=retained_categorical,
    )

    fit_seconds = perf_counter() - fit_start

    prediction_raw = np.asarray(
        model.predict(X_valid),
        dtype=float,
    )

    if not np.isfinite(prediction_raw).all():
        raise RuntimeError(
            f"{payload['metric']} · h={payload['horizon']} · "
            f"{source}: ablation produced non-finite predictions."
        )

    prediction_final = np.maximum(
        prediction_raw,
        0.0,
    )

    ablated_raw_mae = mean_absolute_error_462(
        actual,
        prediction_raw,
    )
    ablated_final_mae = mean_absolute_error_462(
        actual,
        prediction_final,
    )

    intact_raw_mae = float(payload["intact_raw_mae"])
    intact_final_mae = float(payload["intact_final_mae"])
    benchmark_mae = float(payload["benchmark_mae"])

    raw_mae_delta = (
        ablated_raw_mae - intact_raw_mae
    )
    final_mae_delta = (
        ablated_final_mae - intact_final_mae
    )

    raw_mae_degradation_pct = (
        100.0 * raw_mae_delta / intact_raw_mae
        if intact_raw_mae > 0
        else np.nan
    )
    final_mae_degradation_pct = (
        100.0 * final_mae_delta / intact_final_mae
        if intact_final_mae > 0
        else np.nan
    )

    intact_benchmark_skill_pct = (
        100.0
        * (benchmark_mae - intact_final_mae)
        / benchmark_mae
        if benchmark_mae > 0
        else np.nan
    )
    ablated_benchmark_skill_pct = (
        100.0
        * (benchmark_mae - ablated_final_mae)
        / benchmark_mae
        if benchmark_mae > 0
        else np.nan
    )

    return {
        "status": "success",
        "ablation_contract_id":
            TREE_ABLATION_CONTRACT_ID,
        "logical_unit_id":
            workload_row.logical_unit_id,
        "metric":
            payload["metric"],
        "horizon":
            int(payload["horizon"]),
        "modeling_track":
            payload["modeling_track"],
        "ablation_id":
            workload_row.ablation_id,
        "information_source":
            source,
        "features_removed":
            len(removed_features),
        "predictors_retained":
            len(retained_predictors),
        "train_rows":
            len(payload["train_df"]),
        "validation_rows":
            len(payload["validation_df"]),
        "intact_raw_mae":
            intact_raw_mae,
        "ablated_raw_mae":
            ablated_raw_mae,
        "raw_mae_delta":
            raw_mae_delta,
        "raw_mae_degradation_pct":
            raw_mae_degradation_pct,
        "intact_final_mae":
            intact_final_mae,
        "ablated_final_mae":
            ablated_final_mae,
        "final_mae_delta":
            final_mae_delta,
        "final_mae_degradation_pct":
            final_mae_degradation_pct,
        "benchmark_mae":
            benchmark_mae,
        "intact_benchmark_skill_pct":
            intact_benchmark_skill_pct,
        "ablated_benchmark_skill_pct":
            ablated_benchmark_skill_pct,
        "benchmark_skill_change_pp":
            (
                ablated_benchmark_skill_pct
                - intact_benchmark_skill_pct
            ),
        "negative_prediction_count":
            int((prediction_raw < 0).sum()),
        "fit_seconds":
            float(fit_seconds),
    }


**4.6.2.5.2B — Run or recover the 24 ablated LightGBM fits.**

The restart unit is one **Target × Horizon × Information Source** experiment.

Each successful unit writes a one-row checkpoint immediately after fitting. On rerun, a unit is reused only if its checkpoint is readable, belongs to the current ablation contract, matches the expected logical identity, and reports `status = "success"`.

Because all three ablations for one Target × Horizon job use the same underlying matrix, that job's data remain in memory while its three source-removal experiments are fit. The payload is then released before the next job is loaded.

Completed compatible work is recovered automatically. Missing or failed work executes only when `RUN_TREE_ABLATION = True`.


In [36]:
# 4.6.2.5.2B — Run or recover the formal Tree ablation
# ---------------------------------------------------------------------

TREE_ABLATION_PARTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


def tree_ablation_checkpoint_is_success(
    logical_unit_id: str,
) -> bool:
    """Recognize one valid successful checkpoint for the current contract."""
    path = tree_ablation_checkpoint_path(
        logical_unit_id
    )

    if not path.exists():
        return False

    try:
        checkpoint_df = pd.read_parquet(
            path,
            columns=[
                "status",
                "ablation_contract_id",
                "logical_unit_id",
            ],
        )
    except Exception:
        return False

    if len(checkpoint_df) != 1:
        return False

    row = checkpoint_df.iloc[0]

    return (
        row["status"] == "success"
        and row["ablation_contract_id"]
            == TREE_ABLATION_CONTRACT_ID
        and row["logical_unit_id"]
            == logical_unit_id
    )


def tree_ablation_final_is_valid() -> bool:
    """Recognize a complete 24-unit final artifact for this contract."""
    if not TREE_ABLATION_FINAL_PATH.exists():
        return False

    try:
        final_df = pd.read_parquet(
            TREE_ABLATION_FINAL_PATH,
            columns=[
                "status",
                "ablation_contract_id",
                "logical_unit_id",
            ],
        )
    except Exception:
        return False

    expected_units = set(
        tree_ablation_workload_df["logical_unit_id"]
    )
    observed_units = set(
        final_df["logical_unit_id"]
    )

    return (
        len(final_df) == len(expected_units)
        and observed_units == expected_units
        and final_df["status"].eq("success").all()
        and final_df["ablation_contract_id"]
            .eq(TREE_ABLATION_CONTRACT_ID)
            .all()
    )


# ---------------------------------------------------------------------
# Optional result reset
# ---------------------------------------------------------------------
# Reset removes only fitted-result checkpoints and the consolidated result.
# The feature/workload contracts remain intact because they define the current
# experiment rather than representing completed model work.

if RESET_TREE_ABLATION_CHECKPOINTS:
    if TREE_ABLATION_PARTS_DIR.exists():
        shutil.rmtree(
            TREE_ABLATION_PARTS_DIR
        )

    TREE_ABLATION_FINAL_PATH.unlink(
        missing_ok=True
    )

    TREE_ABLATION_PARTS_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )


# ---------------------------------------------------------------------
# Fast path: reuse a complete consolidated artifact
# ---------------------------------------------------------------------

if tree_ablation_final_is_valid():
    tree_ablation_results_df = pd.read_parquet(
        TREE_ABLATION_FINAL_PATH
    )

    print(
        "Tree ablation recovered from the completed final artifact: "
        f"{len(tree_ablation_results_df)}/24 successful units.",
        flush=True,
    )

else:
    completed_units = {
        logical_unit_id
        for logical_unit_id
        in tree_ablation_workload_df["logical_unit_id"]
        if tree_ablation_checkpoint_is_success(
            logical_unit_id
        )
    }

    print(
        "Tree ablation restart state: "
        f"{len(completed_units)}/24 successful units recovered.",
        flush=True,
    )

    workload_to_run_df = (
        tree_ablation_workload_df.loc[
            ~tree_ablation_workload_df[
                "logical_unit_id"
            ].isin(completed_units)
        ]
        .sort_values(
            ["metric", "horizon", "source_order"]
        )
        .reset_index(drop=True)
    )


    # -------------------------------------------------------------
    # Execute only unfinished logical units
    # -------------------------------------------------------------

    if (
        RUN_TREE_ABLATION
        and not workload_to_run_df.empty
    ):
        active_job = None
        payload = None

        progress = tqdm(
            workload_to_run_df.itertuples(
                index=False
            ),
            total=len(workload_to_run_df),
            desc="Tree ablation",
        )

        for workload_row in progress:
            job_key = (
                workload_row.metric,
                int(workload_row.horizon),
            )

            progress.set_postfix_str(
                f"{TARGET_SHORT_LABELS[workload_row.metric]} "
                f"h={int(workload_row.horizon)} · "
                f"{workload_row.information_source}"
            )

            try:
                # The same large matrix payload supports all three source
                # removals for one job, avoiding three redundant Parquet reads.
                if job_key != active_job:
                    if payload is not None:
                        del payload
                        gc.collect()

                    payload = load_tree_ablation_job(
                        workload_row.metric,
                        int(workload_row.horizon),
                    )
                    active_job = job_key

                result = fit_tree_ablation_unit(
                    payload,
                    workload_row,
                )

            except Exception as exc:
                # A failed unit remains unfinished. Its failure is durable and
                # visible, and a later RUN=True rerun will retry that unit.
                result = {
                    "status": "failed",
                    "ablation_contract_id":
                        TREE_ABLATION_CONTRACT_ID,
                    "logical_unit_id":
                        workload_row.logical_unit_id,
                    "metric":
                        workload_row.metric,
                    "horizon":
                        int(workload_row.horizon),
                    "modeling_track":
                        workload_row.modeling_track,
                    "ablation_id":
                        workload_row.ablation_id,
                    "information_source":
                        workload_row.information_source,
                    "error_type":
                        type(exc).__name__,
                    "error_message":
                        str(exc),
                }

            atomic_write_parquet(
                pd.DataFrame([result]),
                tree_ablation_checkpoint_path(
                    workload_row.logical_unit_id
                ),
            )

        if payload is not None:
            del payload
            gc.collect()


    # -------------------------------------------------------------
    # Reconstruct state from the 24 expected checkpoint locations
    # -------------------------------------------------------------

    checkpoint_frames = []

    for logical_unit_id in (
        tree_ablation_workload_df[
            "logical_unit_id"
        ]
    ):
        path = tree_ablation_checkpoint_path(
            logical_unit_id
        )

        if path.exists():
            checkpoint_frames.append(
                pd.read_parquet(path)
            )

    tree_ablation_results_df = (
        pd.concat(
            checkpoint_frames,
            ignore_index=True,
            sort=False,
        )
        if checkpoint_frames
        else pd.DataFrame()
    )

    if tree_ablation_results_df.empty:
        successful_units = set()
        failed_units = 0
    else:
        successful_units = set(
            tree_ablation_results_df.loc[
                tree_ablation_results_df[
                    "status"
                ].eq("success"),
                "logical_unit_id",
            ]
        )
        failed_units = int(
            tree_ablation_results_df[
                "status"
            ].eq("failed").sum()
        )

    not_executed_units = (
        24
        - len(successful_units)
        - failed_units
    )

    print(
        "Tree ablation result state: "
        f"{len(successful_units)}/24 successful · "
        f"{failed_units} failed · "
        f"{not_executed_units} not yet executed.",
        flush=True,
    )


    # -------------------------------------------------------------
    # Publish only after every expected logical unit has succeeded
    # -------------------------------------------------------------

    expected_units = set(
        tree_ablation_workload_df[
            "logical_unit_id"
        ]
    )

    if successful_units == expected_units:
        tree_ablation_results_df = (
            tree_ablation_results_df.loc[
                tree_ablation_results_df[
                    "status"
                ].eq("success")
            ]
            .sort_values(
                [
                    "metric",
                    "horizon",
                    "information_source",
                ]
            )
            .reset_index(drop=True)
        )

        if WRITE_FINAL_OUTPUTS:
            atomic_write_parquet(
                tree_ablation_results_df,
                TREE_ABLATION_FINAL_PATH,
            )

        print(
            "Tree ablation complete: "
            "24/24 current-contract units succeeded.",
            flush=True,
        )

    elif not RUN_TREE_ABLATION:
        print(
            "Ablation fitting is disabled. "
            "Set RUN_TREE_ABLATION = True to execute unfinished units.",
            flush=True,
        )


Tree ablation recovered from the completed final artifact: 24/24 successful units.


**Findings.** All **24 planned Tree ablations completed successfully**, with no failed units. The full experiment finished in approximately **5 minutes 24 seconds**, or about **13.5 seconds per Target × Horizon × Information Source experiment**.

The completed artifact covers all eight Tree champion jobs under all three approved information-source removals. Because each logical experiment is checkpointed independently, the completed results can be reused without repeating successful model fits.

### 4.6.2.5.3 Compare What the Tree Models Use with What They Actually Need

The ablation result is easiest to interpret as **MAE degradation relative to the intact model**:

$$
100 \times
\frac{\text{Ablated MAE} - \text{Intact MAE}}
{\text{Intact MAE}}
$$

A positive value means removing the information source made the forecast worse. A value near zero means the remaining predictors were able to compensate. A negative value means the retrained model actually improved after that source was removed.

This percentage is calculated from each job's complete ordinary-validation MAE, not from tiny row-level denominators. It can therefore be compared across Taxi trips, FHVHV trips, and Subway without treating their native units as interchangeable.

We first examine the ablation result directly, then compare it with the SHAP reliance of the intact models.

**4.6.2.5.3A — Measure how much accuracy is lost when each information source is removed.**

The summary below answers the most direct ablation question: across the eight Tree champions, how consistently does forecast accuracy deteriorate when each source disappears?

The heatmap retains the individual Target × Horizon results. Positive cells indicate that the information source contributed predictive value that the remaining model could not fully replace.


In [37]:
# ---------------------------------------------------------------------
# 4.6.2.5.3A — Summarize formal Tree ablation damage
# ---------------------------------------------------------------------

expected_ablation_units = set(tree_ablation_workload_df["logical_unit_id"])
completed_ablation_units = set(tree_ablation_results_df["logical_unit_id"])

tree_ablation_complete = (
    len(tree_ablation_results_df) == 24
    and tree_ablation_results_df["status"].eq("success").all()
    and completed_ablation_units == expected_ablation_units
)

if not tree_ablation_complete:
    print(
        "Tree ablation interpretation deferred: "
        "24/24 successful units are required.",
        flush=True,
    )

else:
    # SHAP describes what the intact model used. Joining it to the ablation
    # results lets us later compare fitted attribution with information that
    # actually becomes costly to remove and retrain around.
    tree_ablation_shap_df = (
        tree_family_share_df.loc[
            tree_family_share_df["feature_family"].isin(
                TREE_ABLATION_SOURCE_ORDER
            ),
            ["metric", "horizon", "feature_family", "shap_share"],
        ]
        .rename(columns={"feature_family": "information_source"})
    )

    tree_ablation_analysis_df = tree_ablation_results_df.merge(
        tree_ablation_shap_df,
        on=["metric", "horizon", "information_source"],
        how="left",
        validate="one_to_one",
    )

    if tree_ablation_analysis_df["shap_share"].isna().any():
        raise RuntimeError(
            "At least one formal ablation is missing its intact-model "
            "SHAP family share."
        )

    tree_ablation_analysis_df["shap_share_pct"] = (
        100.0 * tree_ablation_analysis_df["shap_share"]
    )
    tree_ablation_analysis_df = add_job_labels(tree_ablation_analysis_df)


    # -----------------------------------------------------------------
    # Compact source-level summary
    # -----------------------------------------------------------------

    tree_ablation_group_summary_df = (
        tree_ablation_analysis_df
        .groupby("information_source", observed=True, as_index=False)
        .agg(
            jobs=("logical_unit_id", "nunique"),
            mean_shap_share_pct=("shap_share_pct", "mean"),
            median_mae_degradation_pct=("final_mae_degradation_pct", "median"),
            minimum_mae_degradation_pct=("final_mae_degradation_pct", "min"),
            maximum_mae_degradation_pct=("final_mae_degradation_pct", "max"),
            jobs_hurt=(
                "final_mae_degradation_pct",
                lambda values: int(values.gt(0).sum()),
            ),
        )
    )

    source_order_lookup = {
        source: order
        for order, source in enumerate(TREE_ABLATION_SOURCE_ORDER, start=1)
    }

    tree_ablation_group_summary_df["source_order"] = (
        tree_ablation_group_summary_df["information_source"]
        .map(source_order_lookup)
    )

    tree_ablation_group_summary_df = (
        tree_ablation_group_summary_df
        .sort_values("source_order")
        .drop(columns="source_order")
        .rename(
            columns={
                "information_source": "Information source",
                "jobs": "Jobs tested",
                "mean_shap_share_pct": "Mean SHAP share (%)",
                "median_mae_degradation_pct": "Median MAE degradation (%)",
                "minimum_mae_degradation_pct": "Minimum degradation (%)",
                "maximum_mae_degradation_pct": "Maximum degradation (%)",
                "jobs_hurt": "Jobs worsened after removal",
            }
        )
    )

    display(tree_ablation_group_summary_df.round(3))


    # -----------------------------------------------------------------
    # Job-level signed-damage heatmap
    # -----------------------------------------------------------------
    # Keep the heatmap's z-values numeric for the color scale, but construct
    # every hover field as text first. This guarantees <=3 displayed decimals
    # without reducing the precision of the saved analytical results.

    tree_job_order = [
        job for job in JOB_LABEL_ORDER
        if job in set(tree_ablation_analysis_df["Job"])
    ]

    damage_matrix_df = (
        tree_ablation_analysis_df
        .pivot(
            index="information_source",
            columns="Job",
            values="final_mae_degradation_pct",
        )
        .reindex(index=TREE_ABLATION_SOURCE_ORDER, columns=tree_job_order)
    )

    intact_mae_matrix_df = (
        tree_ablation_analysis_df
        .pivot(index="information_source", columns="Job", values="intact_final_mae")
        .reindex_like(damage_matrix_df)
    )

    ablated_mae_matrix_df = (
        tree_ablation_analysis_df
        .pivot(index="information_source", columns="Job", values="ablated_final_mae")
        .reindex_like(damage_matrix_df)
    )

    shap_matrix_df = (
        tree_ablation_analysis_df
        .pivot(index="information_source", columns="Job", values="shap_share_pct")
        .reindex_like(damage_matrix_df)
    )

    if RUN_VISUALS:
        hover_data = np.dstack([
            intact_mae_matrix_df.map(
                lambda value: format_hover_number(value)
            ).to_numpy(),
            ablated_mae_matrix_df.map(
                lambda value: format_hover_number(value)
            ).to_numpy(),
            shap_matrix_df.map(
                lambda value: format_hover_number(value, suffix="%")
            ).to_numpy(),
            damage_matrix_df.map(
                lambda value: format_hover_number(value, signed=True, suffix="%")
            ).to_numpy(),
        ])

        # Cell labels stay intentionally lighter than the tooltip; their job is
        # to show direction and approximate magnitude at a glance.
        heatmap_text = damage_matrix_df.map(
            lambda value: (
                "" if pd.isna(value)
                else format_hover_number(value, decimals=1, signed=True, suffix="%")
            )
        ).to_numpy()

        tree_ablation_heatmap_fig = go.Figure(
            go.Heatmap(
                z=damage_matrix_df.to_numpy(),
                x=list(damage_matrix_df.columns),
                y=list(damage_matrix_df.index),
                customdata=hover_data,
                text=heatmap_text,
                texttemplate="%{text}",
                colorscale=BRAND_DIVERGING_SEQUENCE,
                zmid=0,
                colorbar={"title": "MAE<br>degradation (%)"},
                hovertemplate=(
                    "<b>%{y}</b><br>"
                    "%{x}<br>"
                    "MAE degradation: %{customdata[3]}<br>"
                    "Intact MAE: %{customdata[0]}<br>"
                    "Ablated MAE: %{customdata[1]}<br>"
                    "Intact SHAP share: %{customdata[2]}"
                    "<extra></extra>"
                ),
            )
        )

        tree_ablation_heatmap_fig = apply_branding(tree_ablation_heatmap_fig)
        tree_ablation_heatmap_fig.update_layout(
            title=(
                "What happens when the Tree champions lose an information source?"
                "<br><sup>Positive values mean ordinary-validation MAE increased "
                "after retraining without that source.</sup>"
            ),
            height=430,
            margin={"l": 190, "r": 60, "t": 95, "b": 80},
        )
        tree_ablation_heatmap_fig.update_xaxes(title_text="")
        tree_ablation_heatmap_fig.update_yaxes(title_text="")
        tree_ablation_heatmap_fig.show()

,Information source,Jobs tested,Mean SHAP share (%),Median MAE degradation (%),Minimum degradation (%),Maximum degradation (%),Jobs worsened after removal
2,Recent sequence history,8,35.586,11.491,-1.510,31.224,6
0,Comparable Daypart history,8,17.476,4.387,0.463,16.864,8
1,Current mobility state,8,16.685,1.110,0.409,1.969,8


**Findings.** Removing **Recent sequence history** causes the largest typical accuracy loss, increasing MAE by a median **11.491%** across the eight Tree champions. The effect is strongly horizon-dependent: removing recent sequence information increases MAE by **27.618%** and **31.224%** for Taxi trips at h=1 and h=2, **7.601%** and **15.382%** for FHVHV trips, and **17.068%** for Subway h=1.

That dependence largely disappears at h=5. Recent-sequence removal changes MAE by only about **+0.1% for Taxi**, **−0.1% for FHVHV**, and **−1.5% for Subway**, showing that the longer-horizon models can replace immediate sequence information with other historical signals.

**Comparable Daypart history** shows the opposite pattern. Removing it worsens **all eight jobs**, with a median MAE increase of **4.387%**, and it becomes especially important at h=5: MAE rises **7.995% for Taxi trips**, **7.496% for FHVHV trips**, and **16.864% for Subway**. Subway h=1 also depends substantially on this information, with a **13.4%** degradation.

**Current mobility state** is consistently useful but much easier to replace. Removing `origin_value` worsens all eight jobs, but only modestly: the median degradation is **1.110%** and the largest is **1.969%**.

The ablation therefore reveals a clear shift in forecasting dependence: **short horizons need the immediate trajectory, while longer horizons depend more on recurring comparable-Daypart history.**

<img src="image-20260914-215309.png" width="" align="" />

**4.6.2.5.3B — Compare information the intact model uses with information it cannot easily replace.**

SHAP and ablation answer different questions.

A high SHAP share means the intact model frequently used that information when constructing its predictions. A large ablation penalty means the **remaining predictors could not recover the same accuracy after that information was removed and the model was retrained**.

Putting the two measures together helps distinguish four patterns:

- high SHAP and high ablation damage → heavily used and difficult to replace;
- high SHAP but little ablation damage → heavily used, but substitutes exist;
- lower SHAP but meaningful ablation damage → modest fitted attribution may understate the source's structural value;
- little SHAP and little damage → comparatively weak evidence that the source is essential.

No arbitrary cutoff is imposed. The purpose is to see whether attribution and predictive dependence tell a coherent story.

In [38]:
# ---------------------------------------------------------------------
# 4.6.2.5.3B — Compare intact SHAP reliance with ablation damage
# ---------------------------------------------------------------------

if not tree_ablation_complete:
    print(
        "SHAP-versus-ablation comparison deferred until "
        "the 24-unit experiment is complete.",
        flush=True,
    )

else:
    # For each Tree job, identify the removal that damages accuracy most.
    # This shows whether the same information source is essential everywhere
    # or whether predictive dependence changes with forecast horizon.
    largest_damage_idx = (
        tree_ablation_analysis_df
        .groupby(["metric", "horizon"], observed=True)["final_mae_degradation_pct"]
        .idxmax()
    )

    tree_largest_damage_df = (
        tree_ablation_analysis_df.loc[
            largest_damage_idx,
            ["Job", "information_source", "shap_share_pct", "final_mae_degradation_pct"],
        ]
        .rename(
            columns={
                "information_source": "Largest-damage removal",
                "shap_share_pct": "Intact SHAP share (%)",
                "final_mae_degradation_pct": "MAE degradation (%)",
            }
        )
    )

    tree_largest_damage_df["Job"] = pd.Categorical(
        tree_largest_damage_df["Job"],
        categories=JOB_LABEL_ORDER,
        ordered=True,
    )
    tree_largest_damage_df = tree_largest_damage_df.sort_values("Job").reset_index(
        drop=True
    )

    display(tree_largest_damage_df.round(3))


    if RUN_VISUALS:
        plot_df = tree_ablation_analysis_df.copy()

        # Pre-format all reader-facing numbers. Axes remain numeric, but the
        # tooltip can never expose the underlying 6–9 digit analytical values.
        plot_df["shap_hover"] = plot_df["shap_share_pct"].map(
            lambda value: format_hover_number(value, suffix="%")
        )
        plot_df["damage_hover"] = plot_df["final_mae_degradation_pct"].map(
            lambda value: format_hover_number(value, signed=True, suffix="%")
        )
        plot_df["intact_mae_hover"] = plot_df["intact_final_mae"].map(
            format_hover_number
        )
        plot_df["ablated_mae_hover"] = plot_df["ablated_final_mae"].map(
            format_hover_number
        )

        tree_used_needed_fig = px.scatter(
            plot_df,
            x="shap_share_pct",
            y="final_mae_degradation_pct",
            color="information_source",
            color_discrete_sequence=BRAND_COLOR_SEQUENCE,
            custom_data=[
                "Job",
                "shap_hover",
                "damage_hover",
                "intact_mae_hover",
                "ablated_mae_hover",
            ],
            labels={
                "shap_share_pct": "Intact-model SHAP share (%)",
                "final_mae_degradation_pct": "MAE degradation after removal (%)",
                "information_source": "Information source",
            },
        )

        tree_used_needed_fig.update_traces(
            hovertemplate=(
                "<b>%{customdata[0]}</b><br>"
                "Intact SHAP share: %{customdata[1]}<br>"
                "MAE degradation: %{customdata[2]}<br>"
                "Intact MAE: %{customdata[3]}<br>"
                "Ablated MAE: %{customdata[4]}"
                "<extra></extra>"
            )
        )

        tree_used_needed_fig.add_hline(y=0, line_dash="dot")
        tree_used_needed_fig = apply_branding(tree_used_needed_fig)
        tree_used_needed_fig.update_layout(
            title=(
                "What the Tree champions use versus what they cannot replace"
                "<br><sup>SHAP describes intact-model use; ablation measures "
                "accuracy lost after retraining without that information.</sup>"
            ),
            height=560,
            margin={"l": 80, "r": 40, "t": 95, "b": 70},
        )
        tree_used_needed_fig.show()

,Job,Largest-damage removal,Intact SHAP share (%),MAE degradation (%)
0,Taxi trips · h=1,Recent sequence history,59.840,27.618
1,Taxi trips · h=2,Recent sequence history,59.526,31.224
2,Taxi trips · h=5,Comparable Daypart history,29.267,7.995
3,FHVHV trips · h=1,Recent sequence history,52.451,7.601
4,FHVHV trips · h=2,Recent sequence history,55.467,15.382
5,FHVHV trips · h=5,Comparable Daypart history,37.118,7.496
6,Subway · h=1,Recent sequence history,53.503,17.068
7,Subway · h=5,Comparable Daypart history,48.540,16.864


**Findings.** SHAP attribution and retrained ablation tell a strikingly consistent horizon story.

For **every Tree h=1 or h=2 champion**, removing Recent sequence history causes the largest accuracy loss. These are also the jobs where the intact models assign roughly half or more of their SHAP importance to recent sequence information. Taxi trips h=1 and h=2 provide the clearest examples: Recent sequence history carries about **60% of intact-model SHAP importance**, and removing it raises MAE by **27.618%** and **31.224%** respectively.

For **all three h=5 Tree champions**, the largest ablation penalty instead comes from removing Comparable Daypart history: **7.995% for Taxi trips, 7.496% for FHVHV trips, and 16.864% for Subway**. This agrees with the growing prominence of comparable-period history in the intact h=5 models.

The most informative disagreement concerns **Current mobility state**. `origin_value` receives very large intact-model SHAP importance at h=5—**52.796% for Taxi trips** and **35.528% for FHVHV trips**—yet removing it increases MAE by only about **1.7%–1.9%**. The model therefore prefers to use the latest observed level when it is available, but other historical predictors can recover most of that information after retraining.

This is exactly why the two interpretation methods are complementary: **SHAP shows what the intact model chooses to use, while ablation shows which information cannot easily be replaced.**

### Section Summary

The formal ablation confirms that the Tree champions are genuinely dependent on historical mobility information, but **which history matters changes with forecast horizon**.

At h=1 and h=2, immediate sequence history is the least replaceable source of information. Removing it substantially degrades Taxi, FHVHV, and Subway forecasts, with the largest losses exceeding 25% for Taxi trips. At h=5, that dependence fades and **Comparable Daypart history becomes the most consequential information source** across all three Tree-owned targets.

Current mobility state provides an important contrast between feature attribution and predictive necessity. Although `origin_value` can receive very high SHAP importance in the intact h=5 models, removing it produces only modest accuracy losses because other historical features can substitute for much of the same information.

The combined SHAP and ablation evidence therefore supports a clearer interpretation than either method alone: **the Tree models use the latest mobility level when convenient, but their harder-to-replace forecasting signal comes from short-term trajectory at near horizons and recurring comparable-period structure farther ahead.**

These conclusions come entirely from ordinary validation. The final January–March 2026 holdout remains untouched by the ablation experiment.

## 4.6.2.6 Select Specific Forecast Failures for Record-Level Analysis

The aggregate diagnostics have already identified several different ways forecasts can become difficult. The grading requirement now asks for something more concrete: **specific failed forecast records, the circumstances surrounding them, and plausible improvements.**

The three formal case studies should not simply be the three largest numerical errors. That could select several versions of the same underlying problem.

Instead, this section first builds a small candidate slate from diagnostically distinct mechanisms already established in the holdout analysis:

- a sparse or zero-demand local failure;
- an error during one of the system-wide shared-shock weeks;
- a speed forecast under extreme recent volatility;
- the separately identified Subway h=2 structural weakness.

The candidate table uses the within-job error index to make failures across trips, riders, and speed comparable, while retaining the native-unit values needed to understand each record.


**4.6.2.6.1 — Build a small slate of distinct failed forecast records.**

Each category contributes up to three severe-error candidates. Overlapping mechanisms are deliberately excluded where possible so that a shared-shock example is not simultaneously just another zero-demand case, and a high-volatility speed example is not simply a row from one of the already identified shock weeks.

Within each category, candidates are ranked by **error relative to that forecasting job's normal MAE**. We retain different Target × Horizon jobs before taking the top examples, which prevents one unusually difficult job from filling the entire slate.

This is candidate generation, not automatic case selection. The final three records will be frozen only after reviewing whether they tell genuinely different failure stories.

In [39]:
# 4.6.2.6.1 — Build the record-level failure candidate slate
# ---------------------------------------------------------------------

FAILURE_CASE_CANDIDATES_PER_CATEGORY = 3


def select_failure_case_candidates_462(
    frame: pd.DataFrame,
    *,
    mask: pd.Series,
    category: str,
    category_code: str,
) -> pd.DataFrame:
    """
    Return severe examples while preserving Target × Horizon diversity.

    Keeping only the strongest row from each job before taking the final
    candidates prevents one difficult job from filling an entire category.
    """
    candidate_df = frame.loc[
        mask
        & frame["severe_error"]
        & frame["row_error_index"].notna()
    ].copy()

    if candidate_df.empty:
        raise RuntimeError(
            f"No severe forecast rows are available for '{category}'."
        )

    candidate_df = (
        candidate_df
        .sort_values(
            [
                "row_error_index",
                "absolute_error",
            ],
            ascending=[False, False],
        )
        .drop_duplicates(
            ["metric", "horizon"]
        )
        .head(
            FAILURE_CASE_CANDIDATES_PER_CATEGORY
        )
        .copy()
    )

    candidate_df["failure_category"] = category
    candidate_df["candidate_id"] = [
        f"{category_code}{index}"
        for index
        in range(
            1,
            len(candidate_df) + 1,
        )
    ]

    return candidate_df

# ---------------------------------------------------------------------
# Reader-facing case studies require interpretable physical geography
# ---------------------------------------------------------------------
# Taxi Zone IDs 264 and 265 are canonical nonphysical "Unknown" locations
# retained upstream for lossless joins. They remain valid analytical rows, but
# they are poor case-study examples because the reader cannot interpret where
# the failure occurred.

physical_case_zone_mask = (
    ~failure_burden_df["taxi_zone_id"].isin([264, 265])
    & failure_burden_df["zone"].notna()
    & ~failure_burden_df["zone"].eq("Unknown")
)

# ---------------------------------------------------------------------
# Define four candidate mechanisms already established by Section 3
# ---------------------------------------------------------------------

subway_h2_mask = (
    failure_burden_df["metric"].eq(
        "subway_ridership"
    )
    & failure_burden_df["horizon"].eq(2)
)

# Keep this case focused on persistent local sparsity by excluding the broad
# system-shock weeks identified separately.
zero_demand_mask = (
    physical_case_zone_mask
    & failure_burden_df["is_trip_count"]
    & failure_burden_df["low_zero_demand"]
    & ~failure_burden_df["shared_shock"]
)


# Shared-shock examples exclude the other named mechanisms wherever possible
# so the reader sees a broad temporary disruption rather than a duplicate case.
shared_shock_case_mask = (
    physical_case_zone_mask
    & failure_burden_df["shared_shock"]
    & ~failure_burden_df["low_zero_demand"]
    & ~failure_burden_df["unstable_conditions"]
    & ~subway_h2_mask
)


# Extreme-volatility cases stay outside shared-shock weeks so unusual recent
# speed instability remains the primary observed context.
speed_volatility_mask = (
    physical_case_zone_mask
    & failure_burden_df["unstable_conditions"]
    & ~failure_burden_df["shared_shock"]
)

# Subway h=2 is a whole-job weakness rather than a calibrated row mechanism.
# Requiring a severe champion miss where the weekly benchmark wins gives the
# candidate a clear comparative failure interpretation.
subway_h2_case_mask = (
    physical_case_zone_mask
    & subway_h2_mask
    & failure_burden_df["benchmark_loss"]
    & ~failure_burden_df["shared_shock"]
)


failure_candidate_frames = [
    select_failure_case_candidates_462(
        failure_burden_df,
        mask=zero_demand_mask,
        category=
            "Sparse / zero-demand local failure",
        category_code="Z",
    ),
    select_failure_case_candidates_462(
        failure_burden_df,
        mask=shared_shock_case_mask,
        category="Shared-shock failure",
        category_code="S",
    ),
    select_failure_case_candidates_462(
        failure_burden_df,
        mask=speed_volatility_mask,
        category=
            "Extreme speed-volatility failure",
        category_code="V",
    ),
    select_failure_case_candidates_462(
        failure_burden_df,
        mask=subway_h2_case_mask,
        category=
            "Subway h=2 structural weakness",
        category_code="H",
    ),
]

failure_case_candidates_df = pd.concat(
    failure_candidate_frames,
    ignore_index=True,
)

failure_case_candidates_df = add_job_labels(
    failure_case_candidates_df
)

failure_case_candidates_df[
    "benchmark_better"
] = (
    failure_case_candidates_df[
        "benchmark_loss"
    ]
)

failure_case_candidates_display_df = (
    failure_case_candidates_df[
        [
            "candidate_id",
            "failure_category",
            "Job",
            "taxi_zone_id",
            "zone",
            "borough",
            "target_date",
            "target_temporal_bucket",
            "actual",
            "prediction_final",
            "benchmark_prediction",
            "absolute_error",
            "row_error_index",
            "benchmark_better",
            "same_daypart_roll_obs_share_28d",
            "same_daypart_roll_mean_28d",
            "same_daypart_roll_std_28d",
            "failure_combination",
        ]
    ]
    .rename(
        columns={
            "candidate_id": "Candidate",
            "failure_category":
                "Failure category",
            "taxi_zone_id": "Taxi Zone",
            "zone": "Zone",
            "borough": "Borough",
            "target_date": "Target date",
            "target_temporal_bucket":
                "Temporal bucket",
            "actual": "Actual",
            "prediction_final":
                "Champion forecast",
            "benchmark_prediction":
                "Weekly benchmark",
            "absolute_error":
                "Absolute error",
            "row_error_index":
                "Error index",
            "benchmark_better":
                "Benchmark better",
            "same_daypart_roll_obs_share_28d":
                "Prior 28d observed share",
            "same_daypart_roll_mean_28d":
                "Prior same-Daypart mean",
            "same_daypart_roll_std_28d":
                "Prior same-Daypart volatility",
            "failure_combination":
                "Diagnostic combination",
        }
    )
)

display(
    failure_case_candidates_display_df.style
    .format(precision=3, thousands=",")
    .hide(axis="index")
    .set_properties(
        subset=[
            "Candidate",
            "Failure category",
            "Job",
            "Zone",
            "Borough",
            "Temporal bucket",
            "Diagnostic combination",
        ],
        **{"text-align": "left", "vertical-align": "top"},
    )
)

Candidate,Failure category,Job,Taxi Zone,Zone,Borough,Target date,Temporal bucket,Actual,Champion forecast,Weekly benchmark,Absolute error,Error index,Benchmark better,Prior 28d observed share,Prior same-Daypart mean,Prior same-Daypart volatility,Diagnostic combination
Z1,Sparse / zero-demand local failure,Taxi trips · h=5,34,Brooklyn Navy Yard,Brooklyn,2026-03-21 00:00:00,weekend_overnight,198.000,5.867,0.000,192.133,"1,326.283",False,1.000,14.929,32.408,Low / zero demand
Z2,Sparse / zero-demand local failure,Taxi trips · h=1,195,Red Hook,Brooklyn,2026-03-09 00:00:00,weekday_am_peak,12.000,96.536,7.000,84.536,604.027,True,1.000,2.357,1.967,Low / zero demand
Z3,Sparse / zero-demand local failure,Taxi trips · h=2,157,Maspeth,Queens,2026-03-14 00:00:00,weekend_overnight,114.000,56.634,60.000,57.366,380.057,True,1.000,6.714,4.189,Low / zero demand
S1,Shared-shock failure,Taxi trips · h=5,236,Upper East Side North,Manhattan,2026-02-23 00:00:00,weekday_midday,158.000,"1,988.073","1,308.000","1,830.073","12,632.912",True,1.000,"2,017.929",502.991,Shared-shock weeks
S2,Shared-shock failure,Subway · h=5,162,Midtown East,Manhattan,2026-02-23 00:00:00,weekday_pm_peak,"11,318.000","56,103.940","28,073.000","44,785.940","9,843.983",True,1.000,"56,954.464","28,964.449",Shared-shock weeks
S3,Shared-shock failure,Taxi trips · h=2,236,Upper East Side North,Manhattan,2026-01-25 00:00:00,weekend_midday,339.000,"1,806.239","1,493.000","1,467.239","9,720.593",True,1.000,74.179,86.101,Shared-shock weeks
V1,Extreme speed-volatility failure,Taxi speed · h=5,81,Eastchester,Bronx,2026-03-11 00:00:00,weekday_overnight,73.806,18.902,23.730,54.904,"2,525.819",True,0.821,18.575,7.831,High speed volatility
V2,Extreme speed-volatility failure,FHVHV speed · h=1,111,Green-Wood Cemetery,Brooklyn,2026-03-08 00:00:00,weekend_am_peak,38.540,15.350,10.821,23.190,"2,431.100",False,0.857,16.002,5.775,High speed volatility
V3,Extreme speed-volatility failure,FHVHV speed · h=5,111,Green-Wood Cemetery,Brooklyn,2026-03-08 00:00:00,weekend_am_peak,38.540,14.559,10.821,23.981,"2,302.948",False,0.964,13.858,6.181,High speed volatility
H1,Subway h=2 structural weakness,Subway · h=2,162,Midtown East,Manhattan,2026-03-16 00:00:00,weekday_pm_peak,"75,712.000","41,888.797","68,358.000","33,823.203","6,593.344",True,1.000,"24,120.571","14,350.271",No identified mechanism


**Findings.** The candidate slate contains severe forecast misses from four distinct diagnostic contexts, all tied to named physical Taxi Zones that can support meaningful record-level interpretation.

The **sparse / zero-demand** examples show that very low-volume locations can be difficult even when recent history is fully observed. In **Red Hook, Taxi trips h=1 (Z2)**, actual activity was only **12 trips**, while the champion forecast **96.536** and the weekly benchmark forecast **7**. Its prior 28-day observation share was **100%**, so the miss reflects forecasting behavior in a genuinely quiet local market rather than missing recent history.

The **shared-shock** examples capture abrupt departures from otherwise informative historical levels. In **Upper East Side North, Taxi trips h=5 (S1)** on February 23, actual activity fell to **158 trips**, while the champion forecast **1,988.073**, close to the recent same-Daypart mean of **2,017.929**. The weekly benchmark also missed the drop, but less severely at **1,308 trips**. This pattern is consistent with a temporary disruption that historical repetition alone did not anticipate.

The **extreme speed-volatility** examples illustrate a different forecasting challenge. In **Eastchester, Taxi speed h=5 (V1)**, actual speed reached **73.806 mph**, while the champion forecast **18.902 mph** and the weekly benchmark **23.730 mph**. The prior same-Daypart mean was **18.575 mph**, indicating an outcome far outside the location's recent comparable pattern. Green-Wood Cemetery supplies additional FHVHV-speed examples at h=1 and h=5.

The **Subway h=2** candidate represents a broader model/horizon weakness rather than an identified row-level warning condition. At **Midtown East (H1)**, actual ridership reached **75,712**, while the Transformer champion forecast **41,888.797** and the weekly benchmark forecast **68,358**. No sparse-demand, shared-shock, or extreme-volatility condition identifies this row, making it a useful example of a severe error arising inside the generally weaker Subway h=2 forecasting job.

> ✅ Decision — Use Z2, S1, and H1 as the three formal forecast\-failure case studies

The detailed record\-level analysis will use three severe forecast failures representing different forecasting challenges:

\- Z2 — Red Hook, Taxi trips h=1 a sparse\-demand local overprediction despite complete recent observation support;
\- S1 — Upper East Side North, Taxi trips h=5: a shared\-shock failure in which actual activity abruptly departed from an otherwise informative historical pattern; and
\- H1 — Midtown East, Subway h=2: a severe miss with no identified row\-level warning, representing the broader structural weakness of the Subway h=2 champion\.

V1 — Eastchester, Taxi speed h=5 provides a strong and interpretable example of extreme speed volatility, but that mechanism is already supported by the broader volatility analysis\. H1 adds a more distinct failure mode by demonstrating that severe error can also occur without one of the calibrated row\-level warning conditions\.

These three records are frozen because they are individually severe, diagnostically different, and point toward different potential model improvements\.

**4.6.2.6.2 — Freeze the selected records and recover comparable local context.**

A single large forecast error is easier to understand when we can see what happened around it. For each selected record, this block therefore recovers nearby observations from the same **Target × Horizon × Taxi Zone × temporal bucket**.

Keeping the temporal bucket fixed matters because a Weekday AM Peak and Weekend Overnight can represent very different mobility conditions even in the same place.

Each case timeline compares four quantities:

- the observed outcome;
- the frozen champion forecast;
- the previous-week same-Daypart benchmark; and
- the rolling 28-day same-Daypart mean available before the forecast.

The rolling mean is context rather than another model forecast. It shows whether the failed observation was consistent with, or sharply different from, the recent comparable pattern.

In [40]:
# ---------------------------------------------------------------------
# 4.6.2.6.2 — Freeze the selected records and shared context helper
# ---------------------------------------------------------------------

SELECTED_FAILURE_CASE_IDS = ["Z2", "S1", "H1"]

selected_failure_cases_df = failure_case_candidates_df.loc[
    failure_case_candidates_df["candidate_id"].isin(SELECTED_FAILURE_CASE_IDS)
].copy()

# These three rows anchor the formal case-study narrative, so any missing,
# duplicated, or reclassified record should stop the notebook rather than
# silently changing the interpretation.
if set(selected_failure_cases_df["candidate_id"]) != set(SELECTED_FAILURE_CASE_IDS):
    raise RuntimeError(
        "The selected failure cases must resolve exactly to Z2, S1, and H1."
    )

if len(selected_failure_cases_df) != 3:
    raise RuntimeError(
        "Each selected failure-case ID must resolve to exactly one row."
    )

if selected_failure_cases_df["failure_category"].nunique() != 3:
    raise RuntimeError(
        "The formal case studies must represent three distinct mechanisms."
    )

if not selected_failure_cases_df["severe_error"].all():
    raise RuntimeError(
        "Every formal case study must remain a severe-error record."
    )


def build_failure_case_context_462(case_row, days_before=28, days_after=14):
    """
    Recover like-for-like nearby forecasts for one selected record.

    Target, horizon, Taxi Zone, and temporal bucket stay fixed so the chart
    isolates change over time rather than mixing different mobility contexts.
    """
    case_date = pd.Timestamp(case_row["target_date"]).normalize()
    window_start = max(
        FINAL_HOLDOUT_START_DATE,
        case_date - pd.Timedelta(days=days_before),
    )
    window_end = min(
        FINAL_HOLDOUT_END_DATE,
        case_date + pd.Timedelta(days=days_after),
    )

    context_df = failure_burden_df.loc[
        failure_burden_df["metric"].eq(case_row["metric"])
        & failure_burden_df["horizon"].eq(int(case_row["horizon"]))
        & failure_burden_df["taxi_zone_id"].eq(case_row["taxi_zone_id"])
        & failure_burden_df["target_temporal_bucket"].eq(
            case_row["target_temporal_bucket"]
        )
        & failure_burden_df["target_date"].between(window_start, window_end)
    ].copy()

    context_df = context_df.sort_values("target_date").reset_index(drop=True)

    if context_df.empty:
        raise RuntimeError(
            f"{case_row['candidate_id']}: no comparable local context found."
        )

    if not context_df["target_date"].eq(case_date).any():
        raise RuntimeError(
            f"{case_row['candidate_id']}: selected failure date is missing "
            "from its local context window."
        )

    return context_df


def show_failure_case_462(candidate_id):
    """Display one selected failure and its comparable local timeline."""
    case_match = selected_failure_cases_df.loc[
        selected_failure_cases_df["candidate_id"].eq(candidate_id)
    ]

    if len(case_match) != 1:
        raise RuntimeError(
            f"{candidate_id}: expected exactly one selected record."
        )

    case_row = case_match.iloc[0]
    context_df = build_failure_case_context_462(case_row)

    temporal_label = TEMPORAL_BUCKET_LABELS.get(
        case_row["target_temporal_bucket"],
        str(case_row["target_temporal_bucket"]),
    )

    detail_df = pd.DataFrame([{
        "Candidate": candidate_id,
        "Failure category": case_row["failure_category"],
        "Job": case_row["Job"],
        "Zone": case_row["zone"],
        "Borough": case_row["borough"],
        "Target date": pd.Timestamp(case_row["target_date"]).strftime("%Y-%m-%d"),
        "Temporal bucket": temporal_label,
        "Actual": case_row["actual"],
        "Champion forecast": case_row["prediction_final"],
        "Weekly benchmark": case_row["benchmark_prediction"],
        "Absolute error": case_row["absolute_error"],
        "Error index": case_row["row_error_index"],
        "Prior 28d observed share (%)":
            100 * case_row["same_daypart_roll_obs_share_28d"],
        "Prior same-Daypart mean": case_row["same_daypart_roll_mean_28d"],
        "Prior same-Daypart volatility": case_row["same_daypart_roll_std_28d"],
        "Diagnostic combination": case_row["failure_combination"],
    }])

    display(
        detail_df.style
        .format(precision=3, thousands=",")
        .hide(axis="index")
        .set_properties(
            subset=[
                "Candidate", "Failure category", "Job", "Zone", "Borough",
                "Target date", "Temporal bucket", "Diagnostic combination",
            ],
            **{"text-align": "left", "vertical-align": "top"},
        )
    )

    if RUN_VISUALS:
        plot_df = context_df.copy()

        # Reader-facing hover values are formatted before Plotly sees them.
        # The analytical columns remain full precision for plotting and math.
        plot_df["actual_hover"] = plot_df["actual"].map(
            lambda x: f"{x:,.3f}"
        )
        plot_df["champion_hover"] = plot_df["prediction_final"].map(
            lambda x: f"{x:,.3f}"
        )
        plot_df["benchmark_hover"] = plot_df["benchmark_prediction"].map(
            lambda x: f"{x:,.3f}"
        )
        plot_df["history_hover"] = plot_df["same_daypart_roll_mean_28d"].map(
            lambda x: "Not available" if pd.isna(x) else f"{x:,.3f}"
        )

        hover_strings = np.column_stack([
            plot_df["actual_hover"],
            plot_df["champion_hover"],
            plot_df["benchmark_hover"],
            plot_df["history_hover"],
        ])

        fig = go.Figure()

        fig.add_trace(
            go.Scatter(
                x=plot_df["target_date"],
                y=plot_df["actual"],
                mode="lines+markers",
                name="Actual",
                line={"color": BRAND_COLORS["dark_teal"], "width": 3},
                marker={"size": 7},
                customdata=hover_strings,
                hovertemplate=(
                    "<b>%{x|%Y-%m-%d}</b><br>"
                    "Actual: %{customdata[0]}<br>"
                    "Champion forecast: %{customdata[1]}<br>"
                    "Weekly benchmark: %{customdata[2]}<br>"
                    "Prior 28d same-Daypart mean: %{customdata[3]}"
                    "<extra></extra>"
                ),
            )
        )

        fig.add_trace(
            go.Scatter(
                x=plot_df["target_date"],
                y=plot_df["prediction_final"],
                mode="lines+markers",
                name="Champion forecast",
                line={"color": BRAND_COLORS["terracotta"], "width": 2.5},
                marker={"size": 6},
                hoverinfo="skip",
            )
        )

        fig.add_trace(
            go.Scatter(
                x=plot_df["target_date"],
                y=plot_df["benchmark_prediction"],
                mode="lines",
                name="Previous week · same Daypart",
                line={
                    "color": BRAND_COLOR_SEQUENCE[2],
                    "width": 2,
                    "dash": "dash",
                },
                hoverinfo="skip",
            )
        )

        fig.add_trace(
            go.Scatter(
                x=plot_df["target_date"],
                y=plot_df["same_daypart_roll_mean_28d"],
                mode="lines",
                name="Prior 28d same-Daypart mean",
                line={
                    "color": BRAND_COLORS["seafoam"],
                    "width": 2,
                    "dash": "dot",
                },
                hoverinfo="skip",
            )
        )

        case_date = pd.Timestamp(case_row["target_date"])

        fig.add_vline(
            x=case_date,
            line_width=2,
            line_dash="dot",
            line_color=BRAND_COLORS["terracotta"],
        )
        fig.add_annotation(
            x=case_date,
            y=1.0,
            yref="paper",
            text=f"{candidate_id} selected failure",
            showarrow=False,
            yshift=12,
            font={"color": BRAND_COLORS["terracotta"]},
        )

        fig = apply_branding(fig)
        fig.update_layout(
            title=(
                f"{candidate_id} · {case_row['Job']} · {case_row['zone']}"
                f"<br><sup>{temporal_label} · comparable local context</sup>"
            ),
            xaxis_title="Target date",
            yaxis_title=TARGET_LABELS[case_row["metric"]],
            height=500,
            hovermode="x unified",
            margin={"l": 70, "r": 30, "t": 95, "b": 60},
        )
        fig.show()

    return context_df


selected_failure_roster_display_df = selected_failure_cases_df[
    [
        "candidate_id", "failure_category", "Job", "zone", "borough",
        "target_date", "target_temporal_bucket", "actual",
        "prediction_final", "benchmark_prediction", "row_error_index",
    ]
].rename(
    columns={
        "candidate_id": "Candidate",
        "failure_category": "Failure category",
        "zone": "Zone",
        "borough": "Borough",
        "target_date": "Target date",
        "target_temporal_bucket": "Temporal bucket",
        "actual": "Actual",
        "prediction_final": "Champion forecast",
        "benchmark_prediction": "Weekly benchmark",
        "row_error_index": "Error index",
    }
)

display(
    selected_failure_roster_display_df.style
    .format(precision=3, thousands=",")
    .hide(axis="index")
)

print(
    "PASS — three distinct severe forecast failures are frozen for "
    "record-level analysis.",
    flush=True,
)

Candidate,Failure category,Job,Zone,Borough,Target date,Temporal bucket,Actual,Champion forecast,Weekly benchmark,Error index
Z2,Sparse / zero-demand local failure,Taxi trips · h=1,Red Hook,Brooklyn,2026-03-09 00:00:00,weekday_am_peak,12.000,96.536,7.000,604.027
S1,Shared-shock failure,Taxi trips · h=5,Upper East Side North,Manhattan,2026-02-23 00:00:00,weekday_midday,158.000,"1,988.073","1,308.000","12,632.912"
H1,Subway h=2 structural weakness,Subway · h=2,Midtown East,Manhattan,2026-03-16 00:00:00,weekday_pm_peak,"75,712.000","41,888.797","68,358.000","6,593.344"


PASS — three distinct severe forecast failures are frozen for record-level analysis.


**Findings.** The formal case-study set contains three severe forecast misses representing different forecasting challenges: a sparse-demand local failure, a broad temporary shock, and a Subway h=2 miss without an identified row-level warning condition.

Each record is paired with like-for-like nearby observations from the same Target × Horizon × Taxi Zone × temporal bucket. This makes the detailed timelines easier to interpret because changes over time are not mixed with differences between locations or dayparts.

### 4.6.2.6.3 Case Z2 — Sparse Local Demand Without Missing History

The first case asks whether a large forecast error came from **missing historical information** or from the difficulty of forecasting a genuinely small local signal.

Z2 separates those explanations cleanly. Red Hook had complete recent observation support, so the error cannot be explained simply by absent history.

The timeline below shows whether the champion repeatedly forecasts above this quiet local series or whether the selected date was itself unusually low.

In [41]:
# ---------------------------------------------------------------------
# 4.6.2.6.3 — Inspect Z2: sparse local demand
# ---------------------------------------------------------------------

z2_failure_context_df = show_failure_case_462("Z2")

Candidate,Failure category,Job,Zone,Borough,Target date,Temporal bucket,Actual,Champion forecast,Weekly benchmark,Absolute error,Error index,Prior 28d observed share (%),Prior same-Daypart mean,Prior same-Daypart volatility,Diagnostic combination
Z2,Sparse / zero-demand local failure,Taxi trips · h=1,Red Hook,Brooklyn,2026-03-09,Weekday · AM peak,12.000,96.536,7.000,84.536,604.027,100.000,2.357,1.967,Low / zero demand


**Findings.** Red Hook's selected miss is not a missing-history problem. Recent observation support is complete, and the nearby Taxi-trip counts are consistently small, usually in the single digits or low teens.

The h=1 Tree champion usually stays near that quiet local scale, but it occasionally produces much larger positive forecasts. On March 9, actual demand was **12 trips** while the champion forecast **96.536**; the previous-week benchmark was much closer at **7 trips**. Similar forecast spikes also appear elsewhere in the local timeline while observed demand remains very small.

This pattern suggests that **one model trained across many Taxi Zones can sometimes overpredict a genuinely sparse local market**, even when the recent history itself is complete. A future improvement could therefore focus on better local calibration or a model designed specifically for very small trip counts, while continuing to treat genuine low activity as valid observed data rather than filtering it out.

### 4.6.2.6.4 Case S1 — When Normally Useful History Stops Being Enough

The second case examines a different forecasting problem: **what happens when recent history looks reasonable until the observed system suddenly behaves differently?**

S1 occurs during a week in which many forecasting jobs became unusually difficult at the same time. The useful question is therefore whether the champion followed the recent mobility pattern while the actual outcome broke sharply away from it.

If both the champion and the simple weekly benchmark follow the historical level but miss the same sudden change, the limitation is not simply a poor choice of lag. The model may be missing information about conditions that were not visible in ordinary mobility history.

In [42]:
# ---------------------------------------------------------------------
# 4.6.2.6.4 — Inspect S1: temporary shared shock
# ---------------------------------------------------------------------

s1_failure_context_df = show_failure_case_462("S1")

Candidate,Failure category,Job,Zone,Borough,Target date,Temporal bucket,Actual,Champion forecast,Weekly benchmark,Absolute error,Error index,Prior 28d observed share (%),Prior same-Daypart mean,Prior same-Daypart volatility,Diagnostic combination
S1,Shared-shock failure,Taxi trips · h=5,Upper East Side North,Manhattan,2026-02-23,Weekday · Midday,158.000,"1,988.073","1,308.000","1,830.073","12,632.912",100.000,"2,017.929",502.991,Shared-shock weeks


**Findings.** Upper East Side North shows a clear temporary-break pattern. Before the selected failure, observed Taxi demand is generally around **2,000–2,500 trips**, and both the champion forecast and the weekly benchmark usually operate in that same range.

On February 23, actual demand abruptly fell to **158 trips**, while the champion forecast remained at **1,988.073** and the prior 28-day same-Daypart mean was **2,017.929**. The weekly benchmark also missed the drop at **1,308 trips**, although by less than the champion.

The important feature of this case is that the forecast followed a historical pattern that had been useful immediately beforehand, while the observed system suddenly moved somewhere very different. This is more consistent with a **temporary shared shock** than with ordinary forecast drift. A future model could benefit from additional context available before forecast time—such as service, event, or weather conditions—but this record alone does not identify the specific real-world cause.

### 4.6.2.6.5 Case H1 — A Severe Miss Inside the Subway h=2 Weakness

The third case removes the obvious row-level warnings.

H1 is not characterized by sparse demand, a shared-shock week, or the calibrated extreme-volatility condition. Instead, it provides a concrete example of the broader weakness already observed for **Subway h=2**.

The weekly benchmark is especially useful here. If a simple previous-week value stays close to the observed ridership while the Transformer moves much farther away, the error points toward a horizon/model representation problem rather than an observation that was difficult for every forecasting approach.

In [43]:
# ---------------------------------------------------------------------
# 4.6.2.6.5 — Inspect H1: Subway h=2 structural weakness
# ---------------------------------------------------------------------

h1_failure_context_df = show_failure_case_462("H1")

Candidate,Failure category,Job,Zone,Borough,Target date,Temporal bucket,Actual,Champion forecast,Weekly benchmark,Absolute error,Error index,Prior 28d observed share (%),Prior same-Daypart mean,Prior same-Daypart volatility,Diagnostic combination
H1,Subway h=2 structural weakness,Subway · h=2,Midtown East,Manhattan,2026-03-16,Weekday · PM peak,"75,712.000","41,888.797","68,358.000","33,823.203","6,593.344",100.000,"24,120.571","14,350.271",No identified mechanism


**Findings.** Midtown East provides a different kind of failure because the selected record has no identified sparse-demand, shared-shock, or extreme-volatility warning.

Observed Subway ridership around the selected date is generally high, with nearby weekday PM-peak values often between roughly **60,000 and 87,000 riders**. On March 16, actual ridership was **75,712**, while the Transformer champion forecast only **41,888.797**. The simple weekly benchmark was much closer at **68,358**.

The miss is therefore not simply an unpredictable one-day collapse or spike. It occurs inside an otherwise active ridership pattern where the weekly benchmark preserves the local level substantially better than the h=2 Transformer. This supports the broader evidence that **the Subway h=2 forecast is less reliable than the neighboring Subway horizons**, while this single record should be treated as an illustration rather than the cause of that weakness. Future work should investigate a forecasting design tailored more specifically to the two-step-ahead Subway problem rather than retroactively changing the frozen champion.

### Section Summary

The three case studies reveal **three different ways a strong forecasting system can still fail**.

Red Hook shows that a model trained across many Taxi Zones can occasionally overpredict a genuinely sparse local market even when recent history is complete. Upper East Side North shows the opposite problem: historical patterns can be reasonable and informative until a broad temporary disruption causes actual mobility to move somewhere the recent record does not anticipate. Midtown East shows that severe errors can also occur without an obvious row-level warning condition, reflecting a broader weakness in the Subway h=2 forecast.

These cases point toward different future improvements—better local calibration for sparse demand, richer external context for temporary shocks, and a forecasting design tailored more specifically to Subway h=2—rather than one universal correction or one holdout-derived exclusion rule.

## 4.6.2.7 Test Sensitivity and Summarize Practical Tradeoffs

The failure analysis explains **where** the frozen system struggles. Two final questions test how defensible that system is as a modeling choice.

First, would an important Tree-model design choice have materially changed the results? The Tree pipeline evaluated both **Full-History Regime-Aware** and **Post-CP Primary** training histories. Because that comparison already exists upstream, this notebook can reuse it rather than retraining anything.

Second, what practical costs come with the final mix of Tree, Neural, and Transformer champions? The families solve different forecasting problems, so one pooled accuracy ranking would be misleading. The useful comparison is instead operational: what each family contributes, how much modeling machinery it requires, and how directly its behavior can be explained.

Neither analysis reopens champion selection.

### 4.6.2.7.1 Sensitivity to the Amount of Training History

The selected Tree model uses **Full-History Regime-Aware** training. That gives LightGBM access to a longer historical record while explicitly representing whether an observation occurred before or after congestion pricing.

A useful sensitivity question is:

> **Would the eight Tree champions perform materially worse if the model were restricted to Post-CP history only?**

The upstream LightGBM experiment already evaluated both training histories on the same validation jobs. This block reuses that evidence and narrows it to the eight Target × Horizon jobs that became Tree champions.

Two scale-aware comparisons are used:

- **NMAE reduction:** how much lower normalized error is under Full-History;
- **benchmark-skill advantage:** how many additional percentage points of improvement over the weekly baseline Full-History achieves.

Positive values favor the selected Full-History design.

In [44]:
# ---------------------------------------------------------------------
# 4.6.2.7.1 — Reuse matched LightGBM training-history sensitivity
# ---------------------------------------------------------------------

tree_history_audit_df = pd.read_parquet(
    require_file(
        TREE_FULL_AUDIT_MODEL_REGISTRY_PATH,
        "complete LightGBM two-track model registry",
    ),
    columns=TREE_FULL_AUDIT_MODEL_REGISTRY_REQUIRED_COLUMNS,
)

require_columns(
    tree_history_audit_df,
    TREE_FULL_AUDIT_MODEL_REGISTRY_REQUIRED_COLUMNS,
    "complete LightGBM two-track model registry",
)

expected_tracks = {"full_history_regime_aware", "post_cp_primary"}

# Sensitivity is meaningful only when the artifact contains one successful
# result for each track × Target × Horizon combination.
if set(tree_history_audit_df["modeling_track"]) != expected_tracks:
    raise RuntimeError(
        "The LightGBM sensitivity registry must contain exactly the "
        "Full-History and Post-CP tracks."
    )

if len(tree_history_audit_df) != 2 * EXPECTED_JOB_COUNT:
    raise RuntimeError(
        "The LightGBM sensitivity registry must contain 30 matched jobs."
    )

if not tree_history_audit_df["status"].eq("success").all():
    raise RuntimeError(
        "At least one frozen LightGBM sensitivity job is not successful."
    )


def split_tree_history_track_462(frame, track, prefix):
    """Rename one training-history track for a clean matched comparison."""
    return frame.loc[
        frame["modeling_track"].eq(track),
        [
            "metric", "horizon", "mae", "nmae",
            "benchmark_mae", "skill_vs_benchmark_pct",
        ],
    ].rename(
        columns={
            "mae": f"{prefix}_mae",
            "nmae": f"{prefix}_nmae",
            "benchmark_mae": f"{prefix}_benchmark_mae",
            "skill_vs_benchmark_pct": f"{prefix}_skill_pct",
        }
    )


full_history_metrics_df = split_tree_history_track_462(
    tree_history_audit_df,
    "full_history_regime_aware",
    "full",
)
post_cp_metrics_df = split_tree_history_track_462(
    tree_history_audit_df,
    "post_cp_primary",
    "post",
)

tree_history_all_jobs_df = full_history_metrics_df.merge(
    post_cp_metrics_df,
    on=["metric", "horizon"],
    how="inner",
    validate="one_to_one",
)

if len(tree_history_all_jobs_df) != EXPECTED_JOB_COUNT:
    raise RuntimeError(
        "The two LightGBM histories do not align on all 15 jobs."
    )

# Reproduce the frozen upstream direction before narrowing to Tree champions.
# This is a contract check, not a new model-selection step.
all_jobs_favor_full_history = (
    tree_history_all_jobs_df["full_nmae"].lt(
        tree_history_all_jobs_df["post_nmae"]
    )
    & tree_history_all_jobs_df["full_skill_pct"].gt(
        tree_history_all_jobs_df["post_skill_pct"]
    )
)

if not all_jobs_favor_full_history.all():
    raise RuntimeError(
        "The LightGBM registry no longer reproduces the expected "
        "Full-History sensitivity result."
    )

tree_champion_jobs_df = (
    champion_registry_df.loc[
        champion_registry_df["champion_family"].eq("tree"),
        ["metric", "horizon"],
    ]
    .drop_duplicates()
)

if len(tree_champion_jobs_df) != 8:
    raise RuntimeError(
        "Expected exactly eight frozen Tree champion jobs."
    )

tree_history_sensitivity_df = tree_history_all_jobs_df.merge(
    tree_champion_jobs_df,
    on=["metric", "horizon"],
    how="inner",
    validate="one_to_one",
)

tree_history_sensitivity_df["nmae_reduction_pct"] = (
    100
    * (
        tree_history_sensitivity_df["post_nmae"]
        - tree_history_sensitivity_df["full_nmae"]
    )
    / tree_history_sensitivity_df["post_nmae"]
)

tree_history_sensitivity_df["skill_advantage_pp"] = (
    tree_history_sensitivity_df["full_skill_pct"]
    - tree_history_sensitivity_df["post_skill_pct"]
)

tree_history_sensitivity_df["full_history_better_both"] = (
    tree_history_sensitivity_df["nmae_reduction_pct"].gt(0)
    & tree_history_sensitivity_df["skill_advantage_pp"].gt(0)
)

tree_history_sensitivity_df = add_job_labels(
    tree_history_sensitivity_df
)

tree_history_sensitivity_df["Job"] = pd.Categorical(
    tree_history_sensitivity_df["Job"],
    categories=JOB_LABEL_ORDER,
    ordered=True,
)

tree_history_sensitivity_df = (
    tree_history_sensitivity_df
    .sort_values("Job")
    .reset_index(drop=True)
)

tree_history_display_df = tree_history_sensitivity_df[
    [
        "Job", "full_nmae", "post_nmae", "nmae_reduction_pct",
        "full_skill_pct", "post_skill_pct", "skill_advantage_pp",
    ]
].rename(
    columns={
        "full_nmae": "Full-History NMAE",
        "post_nmae": "Post-CP NMAE",
        "nmae_reduction_pct": "NMAE reduction (%)",
        "full_skill_pct": "Full-History skill (%)",
        "post_skill_pct": "Post-CP skill (%)",
        "skill_advantage_pp": "Skill advantage (pp)",
    }
)

display(
    tree_history_display_df.style
    .format(precision=3, thousands=",")
    .hide(axis="index")
)

print(
    "Training-history sensitivity\n"
    f"  All LightGBM jobs favoring Full-History: "
    f"{int(all_jobs_favor_full_history.sum())}/{EXPECTED_JOB_COUNT}\n"
    f"  Tree champions favoring Full-History: "
    f"{int(tree_history_sensitivity_df['full_history_better_both'].sum())}/8",
    flush=True,
)

if RUN_VISUALS:
    sensitivity_plot_df = tree_history_sensitivity_df.copy()

    # Preformatted hover strings prevent raw floating-point precision from
    # leaking into the reader-facing tooltip.
    sensitivity_plot_df["nmae_hover"] = (
        sensitivity_plot_df["nmae_reduction_pct"]
        .map(lambda x: f"{x:+.3f}%")
    )
    sensitivity_plot_df["skill_hover"] = (
        sensitivity_plot_df["skill_advantage_pp"]
        .map(lambda x: f"{x:+.3f} pp")
    )
    sensitivity_plot_df["full_nmae_hover"] = (
        sensitivity_plot_df["full_nmae"]
        .map(lambda x: f"{x:.3f}")
    )
    sensitivity_plot_df["post_nmae_hover"] = (
        sensitivity_plot_df["post_nmae"]
        .map(lambda x: f"{x:.3f}")
    )

    fig = px.scatter(
        sensitivity_plot_df,
        x="nmae_reduction_pct",
        y="skill_advantage_pp",
        text="Job",
        custom_data=[
            "Job",
            "nmae_hover",
            "skill_hover",
            "full_nmae_hover",
            "post_nmae_hover",
        ],
    )

    fig.update_traces(
        textposition="top center",
        marker={
            "size": 11,
            "color": BRAND_COLORS["terracotta"],
            "line": {
                "color": BRAND_COLORS["dark_teal"],
                "width": 1,
            },
        },
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "NMAE reduction: %{customdata[1]}<br>"
            "Skill advantage: %{customdata[2]}<br>"
            "Full-History NMAE: %{customdata[3]}<br>"
            "Post-CP NMAE: %{customdata[4]}"
            "<extra></extra>"
        ),
    )

    fig.add_vline(x=0, line_dash="dot")
    fig.add_hline(y=0, line_dash="dot")

    fig = apply_branding(fig)
    fig.update_layout(
        title=(
            "Do the Tree champions benefit from longer training history?"
            "<br><sup>Positive values on both axes favor "
            "Full-History Regime-Aware training.</sup>"
        ),
        xaxis_title="NMAE reduction from Full-History (%)",
        yaxis_title="Benchmark-skill advantage (percentage points)",
        height=520,
        margin={"l": 80, "r": 40, "t": 95, "b": 70},
    )
    fig.show()

Job,Full-History NMAE,Post-CP NMAE,NMAE reduction (%),Full-History skill (%),Post-CP skill (%),Skill advantage (pp)
Taxi trips · h=1,0.135,0.162,16.590,30.242,16.360,13.882
Taxi trips · h=2,0.145,0.179,19.142,25.290,7.594,17.695
Taxi trips · h=5,0.136,0.178,23.661,29.880,8.138,21.742
FHVHV trips · h=1,0.092,0.107,14.183,31.370,20.013,11.356
FHVHV trips · h=2,0.101,0.114,11.075,24.439,15.013,9.426
FHVHV trips · h=5,0.097,0.112,13.655,27.989,16.586,11.403
Subway · h=1,0.088,0.109,19.522,21.637,2.494,19.143
Subway · h=5,0.093,0.104,10.735,16.855,6.728,10.127


Training-history sensitivity
  All LightGBM jobs favoring Full-History: 15/15
  Tree champions favoring Full-History: 8/8


**Findings.** All **8 Tree champion jobs** perform better when the model is trained on the longer Full-History record rather than only Post-CP data. The same pattern holds across all **15 LightGBM jobs** evaluated upstream.

Across the eight Tree champions, using the longer history reduces normalized forecast error by a median **15.387%** and improves performance relative to the weekly benchmark by a median **12.643 percentage points**.

The largest benefit appears for **Taxi trips h=5**, where the longer history reduces normalized error by **23.661%** and improves performance relative to the weekly benchmark by **21.742 percentage points**. Even the least sensitive Tree champion still improves: **Subway h=5** shows a **10.735%** reduction in normalized error and a **10.127-point** improvement over the weekly benchmark.

The benefit is therefore not limited to one target or horizon. The longer regime-aware training history provides a consistent accuracy advantage across the Tree jobs that entered the final champion system, supporting the selected training-history design without reopening champion selection.

### 4.6.2.7.2 Identify the Practical Tradeoffs in the Final Champion System

The final forecasting system does not have one universally best architecture. Different model families win different forecasting jobs, and those gains come with different costs in training complexity, data requirements, interpretability, and local reliability.

Rather than ranking the families on one pooled accuracy number, this section asks a more practical question:

> **What do we gain from each modeling choice, and what do we give up in return?**

> 🧩 Synthesis — The final champion system makes four deliberate tradeoffs

🎯 Specialization vs operational simplicity
Different model families win different jobs: 8 Tree, 3 Neural, and 4 Transformer\. That specialization lets each family play to its strengths, but it also means maintaining three forecasting pipelines and 50 final\-holdout training calls instead of one universal architecture\.

🗂️ More historical data vs simpler data requirements
Every Tree champion performs better with Full\-History Regime\-Aware training\. Across the eight Tree champions, normalized forecast error falls by a median 15\.387%, while performance relative to the weekly benchmark improves by a median 12\.643 percentage points\. The gain is meaningful, but it depends on maintaining a longer historical record and its regime information\.

🔍 Temporal flexibility vs interpretability
Neural and Transformer champions use richer temporal representations and specialize successfully on speed targets\. Tree champions, however, support much more direct feature\-level explanation through SHAP and retrained ablation Greater model complexity does not guarantee greater reliability in every job: Subway h=2 remains less reliable than the neighboring Subway horizons\.

🌐 Global learning vs local calibration
The Tree model benefits from learning across many Taxi Zones and wins 8 of the 15 forecasting jobs, but that pooling can occasionally miscalibrate genuinely sparse local markets\. Red Hook illustrates this clearly: the model had complete recent history but still produced a large overprediction in a very low\-volume zone\.

Bottom line: the final system favors job\-specific predictive strength over architectural simplicity\. We accept additional operational complexity, longer data dependencies, and uneven interpretability because the specialized models forecast their assigned jobs more effectively—while carrying those limitations forward explicitly for downstream use\.

In [45]:
# ---------------------------------------------------------------------
# 4.6.2.7.2 — Verify the practical tradeoff synthesis
# ---------------------------------------------------------------------

# The reader-facing tradeoff analysis belongs in Markdown, not a prose-heavy
# dataframe. This small audit keeps the headline counts tied to the frozen
# champion registry before we use them in the synthesis callout.
tradeoff_family_counts = (
    champion_registry_df["champion_family"]
    .value_counts()
    .to_dict()
)

expected_family_counts = {
    "tree": 8,
    "neural": 3,
    "transformer": 4,
}

if tradeoff_family_counts != expected_family_counts:
    raise RuntimeError(
        "The frozen champion-family counts no longer match the expected "
        "8 Tree / 3 Neural / 4 Transformer contract."
    )

# These are the frozen final-holdout training-call counts for the champion
# jobs, carried forward from the family-specific forecasting workflows.
tradeoff_training_calls = {
    "tree": 8,
    "neural": 18,
    "transformer": 24,
}
total_training_calls = sum(tradeoff_training_calls.values())

if total_training_calls != 50:
    raise RuntimeError(
        "The expected final-holdout training-call total must remain 50."
    )

print(
    "PASS — tradeoff synthesis contract verified\n"
    f"  Champion jobs: 8 Tree / 3 Neural / 4 Transformer\n"
    f"  Final-holdout training calls: {total_training_calls}",
    flush=True,
)

PASS — tradeoff synthesis contract verified
  Champion jobs: 8 Tree / 3 Neural / 4 Transformer
  Final-holdout training calls: 50


### Section Summary

The sensitivity and tradeoff analyses show that the final forecasting design is not simply the most accurate collection of models; it is a set of deliberate compromises.

For the Tree champions, using the longer Full-History record produces a consistent accuracy benefit, supporting the additional historical-data requirement. Across model families, specialization improves Target × Horizon performance but requires three forecasting architectures with different training and interpretation workflows. Tree models provide the clearest feature-level explanation, while Neural and Transformer models offer richer temporal representations for the speed targets they specialize in. Global Tree learning also provides broad coverage at the cost of occasional local calibration failures in very sparse markets.

The final champion system therefore favors **job-specific predictive strength over architectural simplicity**, while carrying its known reliability and interpretability limitations forward explicitly for downstream use.

> ✅ Decision — Advance the frozen champion system unchanged to 4\.7\.1 packaging

The 15\-job champion system is sufficiently validated to move forward without changing the frozen champion mapping or removing difficult forecast conditions from the analytical surface\.

The evaluation shows strong overall performance alongside several important limitations: sparse local demand can produce occasional large overpredictions, broad temporary shocks can break otherwise useful historical patterns, extreme speed volatility remains difficult, and Subway h=2 is less reliable than the neighboring Subway horizons\. These are documented reliability cautions, not reasons to retroactively tune or exclude forecasts using final\-holdout error\.

4\.7\.1 will therefore preserve the complete frozen forecast surface and package these limitations explicitly as provenance, reliability, and diagnostic metadata\. Any downstream forecast\-support or data\-sufficiency eligibility will be defined separately from information available at forecast origin, with genuine observed zero activity kept distinct from missing or insufficient history\.

4\.7\.1 should also prepare reader\-facing fields that can support a future Showcase view of notable forecast successes and failures, including champion\-versus\-benchmark error, location, time, Target × Horizon, champion family, and relevant diagnostic context\. This Showcase layer will illustrate forecast behavior without changing the underlying forecasting contract\.

## 4.6.2.8 Finalize the Evaluation Handoff

The forecasting system is frozen. The remaining job is to leave **4.7.1** a clean, reproducible handoff rather than asking it to reconstruct conclusions from charts and notebook prose.

This final section therefore does three things:

- packages the compact evaluation results that matter downstream;
- preserves a narrow record-level surface for reliability and future **Forecast Wins & Misses** storytelling; and
- verifies that the exported evidence still represents the same 15-job champion system evaluated throughout this notebook.

No model is retrained, reselected, or filtered here.

**4.6.2.8.1 — Package the evaluation evidence needed downstream.**

Most of the detailed analysis can stay in this notebook. The handoff only needs the pieces that another notebook or Showcase page would reasonably reuse.

The row-level export is intentionally narrow. It keeps the observed outcome, champion and benchmark forecasts, error measures, place and time, and the diagnostic labels established in this notebook. That is enough for 4.7.1 to identify notable forecast successes and failures without copying the full diagnostic feature surface.

One downstream rule is also made machine-readable: **holdout-derived difficulty signals remain context, not forecast-rejection rules**. Any actual forecast-support eligibility will be defined separately in 4.7.1 from information available at forecast origin.

In [52]:
# ---------------------------------------------------------------------
# 4.6.2.8.1 — Package the compact evaluation handoff
# ---------------------------------------------------------------------

# Keep the downstream reliability rule short and machine-readable. These
# conditions help explain forecast risk, but none may reject a forecast merely
# because the condition was associated with larger final-holdout errors.
reliability_guidance_df = pd.DataFrame([
    {
        "guidance_id": "sparse_local_demand",
        "scope": "Demand forecasts",
        "downstream_guidance":
            "Keep sparse-demand context visible; genuine low activity remains valid data.",
        "may_exclude_forecast": False,
    },
    {
        "guidance_id": "shared_shock",
        "scope": "All forecast targets",
        "downstream_guidance":
            "Keep shared-shock context visible; do not reject the forecast retrospectively.",
        "may_exclude_forecast": False,
    },
    {
        "guidance_id": "extreme_speed_volatility",
        "scope": "Speed forecasts",
        "downstream_guidance":
            "Keep volatility context visible; the holdout-derived threshold is diagnostic only.",
        "may_exclude_forecast": False,
    },
    {
        "guidance_id": "subway_h2",
        "scope": "Subway ridership · h=2",
        "downstream_guidance":
            "Retain the frozen champion while carrying its relative reliability caution.",
        "may_exclude_forecast": False,
    },
    {
        "guidance_id": "genuine_zero",
        "scope": "Taxi and FHVHV trip counts",
        "downstream_guidance":
            "Treat an observed zero as valid activity, not as missing history.",
        "may_exclude_forecast": False,
    },
])

expected_guidance_ids = {
    "sparse_local_demand",
    "shared_shock",
    "extreme_speed_volatility",
    "subway_h2",
    "genuine_zero",
}

if set(reliability_guidance_df["guidance_id"]) != expected_guidance_ids:
    raise RuntimeError(
        "The downstream reliability-guidance contract is incomplete."
    )

if reliability_guidance_df["may_exclude_forecast"].any():
    raise RuntimeError(
        "A holdout-derived diagnostic was incorrectly converted into an exclusion rule."
    )


# ---------------------------------------------------------------------
# Preserve the three formal failure examples and their local timelines
# ---------------------------------------------------------------------
# The selected rows explain WHAT failed. Their comparable local context helps
# a downstream reader understand whether the miss was isolated or part of a
# surrounding pattern.

case_context_frames = []

for candidate_id, context_df in {
    "Z2": z2_failure_context_df,
    "S1": s1_failure_context_df,
    "H1": h1_failure_context_df,
}.items():
    case_row = selected_failure_cases_df.loc[
        selected_failure_cases_df["candidate_id"].eq(candidate_id)
    ].iloc[0]

    case_context_df = context_df.copy()
    case_context_df.insert(0, "candidate_id", candidate_id)
    case_context_df.insert(1, "failure_category", case_row["failure_category"])
    case_context_frames.append(case_context_df)

failure_case_context_df = pd.concat(
    case_context_frames,
    ignore_index=True,
    sort=False,
)

if set(failure_case_context_df["candidate_id"]) != {"Z2", "S1", "H1"}:
    raise RuntimeError(
        "Failure-case context must contain Z2, S1, and H1."
    )


# ---------------------------------------------------------------------
# Build a narrow row-level surface for 4.7.1 and the future Showcase
# ---------------------------------------------------------------------
# Preserve the same physical forecast identity already validated when the
# diagnostic surface was built. The origin and target observation IDs matter:
# forecast_sequence_id alone is not sufficient to identify one forecast row.
#
# Absolute champion advantage has an intuitive direction:
# positive means the champion was closer to the observed outcome than the
# previous-week benchmark; negative means the benchmark was closer.

record_diagnostic_columns = [
    "modeling_track",
    "champion_family",
    "metric",
    "horizon",
    "forecast_sequence_id",
    "taxi_zone_id",
    "origin_observation_sequence_id",
    "target_observation_sequence_id",
    "zone",
    "borough",
    "target_date",
    "target_temporal_bucket",
    "actual",
    "prediction_final",
    "benchmark_prediction",
    "absolute_error",
    "benchmark_absolute_error",
    "row_error_index",
    "benchmark_loss",
    "severe_error",
    "failure_combination",
    "low_zero_demand",
    "unstable_conditions",
    "shared_shock",
]

require_columns(
    failure_burden_df,
    record_diagnostic_columns,
    "record-level failure diagnostic surface",
)

record_diagnostics_df = failure_burden_df[
    record_diagnostic_columns
].copy()

record_diagnostics_df["champion_advantage_abs"] = (
    record_diagnostics_df["benchmark_absolute_error"]
    - record_diagnostics_df["absolute_error"]
)
record_diagnostics_df["champion_better_than_benchmark"] = (
    record_diagnostics_df["champion_advantage_abs"].gt(0)
)

# Reuse the exact physical forecast key already validated in 4.6.2.2.
# This prevents the compact export from imposing a weaker grain than its
# authoritative row-level source.
record_key = [
    "modeling_track",
    "metric",
    "horizon",
    "forecast_sequence_id",
    "taxi_zone_id",
    "origin_observation_sequence_id",
    "target_observation_sequence_id",
]

duplicate_record_rows = record_diagnostics_df.duplicated(record_key).sum()

if duplicate_record_rows:
    raise RuntimeError(
        "Record-level diagnostic export is not unique on the frozen "
        f"physical forecast key: {duplicate_record_rows:,} duplicate rows."
    )

# Packaging must preserve the complete primary evaluation population rather
# than silently dropping difficult or duplicated-looking observations.
if len(record_diagnostics_df) != expected_primary_rows:
    raise RuntimeError(
        "Record-level diagnostic export no longer matches the primary "
        f"evaluation denominator: {len(record_diagnostics_df):,} vs "
        f"{expected_primary_rows:,}."
    )


# ---------------------------------------------------------------------
# Write only the contracts that downstream work should consume directly
# ---------------------------------------------------------------------

final_handoff_frames = {
    "job_reliability_summary": (
        job_fragility_df,
        JOB_RELIABILITY_SUMMARY_PATH,
    ),
    "failure_mechanism_summary": (
        failure_mechanism_summary_df,
        FAILURE_MECHANISM_SUMMARY_PATH,
    ),
    "failure_case_studies": (
        selected_failure_cases_df,
        FAILURE_CASE_STUDIES_PATH,
    ),
    "failure_case_context": (
        failure_case_context_df,
        FAILURE_CASE_CONTEXT_PATH,
    ),
    "model_interpretation_coverage": (
        interpretation_coverage_df,
        INTERPRETATION_COVERAGE_PATH,
    ),
    "tree_training_history_sensitivity": (
        tree_history_sensitivity_df,
        TREE_HISTORY_SENSITIVITY_PATH,
    ),
    "reliability_guidance": (
        reliability_guidance_df,
        RELIABILITY_GUIDANCE_PATH,
    ),
    "record_diagnostics": (
        record_diagnostics_df,
        RECORD_DIAGNOSTICS_PATH,
    ),
}

if WRITE_FINAL_OUTPUTS:
    for frame, path in final_handoff_frames.values():
        atomic_write_parquet(frame, path)

print(
    "PASS — downstream evaluation contracts are packaged.\n"
    f"  Compact contracts: {len(final_handoff_frames)}\n"
    f"  Record-level diagnostic rows: {len(record_diagnostics_df):,}\n"
    "  Holdout-derived exclusion rules: 0",
    flush=True,
)

PASS — downstream evaluation contracts are packaged.
  Compact contracts: 8
  Record-level diagnostic rows: 1,421,436
  Holdout-derived exclusion rules: 0


**Findings.** The handoff keeps the detailed evaluation compact: seven small analytical contracts plus one narrow record-level diagnostic surface covering the full primary holdout.

That row-level surface also carries the ingredients needed for a future **Forecast Wins & Misses** view—observed value, champion and benchmark predictions, absolute errors, champion advantage, location, time, Target × Horizon, model family, and diagnostic context—without changing the frozen forecasting contract.

**4.6.2.8.2 — Run one final semantic QA before publishing the handoff.**

The closeout should catch contract failures, not repeat the analysis. These checks therefore focus only on facts that must remain true when 4.7.1 consumes the exports: the 15 frozen jobs, family ownership, holdout separation, row count, interpretation coverage, completed Tree ablation, selected failure cases, and training-history sensitivity result.

In [54]:
# ---------------------------------------------------------------------
# 4.6.2.8.2 — Final semantic QA
# ---------------------------------------------------------------------

final_qa_records = []


def record_final_qa(check_name, passed, detail):
    """Collect one final handoff check so failures can be reported together."""
    final_qa_records.append({
        "check": check_name,
        "passed": bool(passed),
        "detail": detail,
    })


# The frozen registry is the identity of the final forecasting system.
unique_champion_jobs = champion_registry_df[
    ["metric", "horizon"]
].drop_duplicates()

record_final_qa(
    "Frozen champion roster",
    len(champion_registry_df) == EXPECTED_JOB_COUNT
    and len(unique_champion_jobs) == EXPECTED_JOB_COUNT,
    f"{len(unique_champion_jobs)}/{EXPECTED_JOB_COUNT} unique jobs",
)

expected_family_counts = {"tree": 8, "transformer": 4, "neural": 3}
actual_family_counts = (
    champion_registry_df["champion_family"]
    .value_counts()
    .to_dict()
)

record_final_qa(
    "Champion-family ownership",
    actual_family_counts == expected_family_counts,
    str(actual_family_counts),
)

holdout_selection_used = (
    champion_registry_df["holdout_used_for_selection"]
    .fillna(False)
    .astype(bool)
    .any()
)

record_final_qa(
    "Final holdout excluded from model selection",
    not holdout_selection_used,
    f"holdout_used_for_selection={holdout_selection_used}",
)

record_final_qa(
    "Primary evaluation denominator",
    len(record_diagnostics_df) == expected_primary_rows,
    f"{len(record_diagnostics_df):,}/{expected_primary_rows:,} rows",
)

interpreted_jobs = int(
    interpretation_coverage_df["Champion jobs"].sum()
)

record_final_qa(
    "Champion interpretation coverage",
    interpreted_jobs == EXPECTED_JOB_COUNT,
    f"{interpreted_jobs}/{EXPECTED_JOB_COUNT} jobs",
)


# Formal ablation is part of the graded deep evaluation, so every planned
# Tree champion × information-source removal must survive the final handoff.
require_file(
    TREE_ABLATION_FINAL_PATH,
    "complete Tree ablation results",
)

final_tree_ablation_df = pd.read_parquet(
    TREE_ABLATION_FINAL_PATH,
    columns=["status", "logical_unit_id"],
)

ablation_complete = (
    len(final_tree_ablation_df) == 24
    and final_tree_ablation_df["status"].eq("success").all()
    and not final_tree_ablation_df["logical_unit_id"].duplicated().any()
)

record_final_qa(
    "Formal Tree ablation",
    ablation_complete,
    f"{len(final_tree_ablation_df)}/24 successful units",
)

selected_case_ids = set(
    selected_failure_cases_df["candidate_id"]
)

record_final_qa(
    "Formal failure case studies",
    len(selected_failure_cases_df) == 3
    and selected_case_ids == {"Z2", "S1", "H1"},
    ", ".join(sorted(selected_case_ids)),
)

tree_history_wins = int(
    tree_history_sensitivity_df["full_history_better_both"].sum()
)

record_final_qa(
    "Tree training-history sensitivity",
    len(tree_history_sensitivity_df) == 8
    and tree_history_wins == 8,
    f"{tree_history_wins}/8 Tree champions favor Full-History",
)

record_final_qa(
    "No holdout-derived rejection rule",
    not reliability_guidance_df["may_exclude_forecast"].any(),
    "0 diagnostic conditions may exclude forecasts",
)

final_qa_df = pd.DataFrame(final_qa_records)
failed_qa_df = final_qa_df.loc[~final_qa_df["passed"]]

if not failed_qa_df.empty:
    failed_checks = failed_qa_df["check"].tolist()
    raise RuntimeError(
        f"Final 4.6.2 QA failed: {failed_checks}"
    )

display(
    final_qa_df.rename(
        columns={
            "check": "QA check",
            "passed": "Passed",
            "detail": "Evidence",
        }
    ).style.hide(axis="index")
)

print(
    "PASS — final 4.6.2 semantic QA succeeded.\n"
    f"  Checks passed: {len(final_qa_df)}/{len(final_qa_df)}",
    flush=True,
)

QA check,Passed,Evidence
Frozen champion roster,True,15/15 unique jobs
Champion-family ownership,True,"{'tree': 8, 'transformer': 4, 'neural': 3}"
Final holdout excluded from model selection,True,holdout_used_for_selection=False
Primary evaluation denominator,True,"1,421,436/1,421,436 rows"
Champion interpretation coverage,True,15/15 jobs
Formal Tree ablation,True,24/24 successful units
Formal failure case studies,True,"H1, S1, Z2"
Tree training-history sensitivity,True,8/8 Tree champions favor Full-History
No holdout-derived rejection rule,True,0 diagnostic conditions may exclude forecasts


PASS — final 4.6.2 semantic QA succeeded.
  Checks passed: 9/9


**Findings.** Final QA confirms that the packaged handoff still represents the intended forecasting system: **15 frozen champion jobs**, complete interpretation coverage, **24/24 successful Tree ablations**, the three formal failure records, and all **8 Tree champions** favoring Full-History training in the sensitivity analysis.

The final holdout remains evaluation-only, and none of the diagnostic conditions has been converted into a retrospective forecast-rejection rule.

**4.6.2.8.3 — Publish one authoritative handoff manifest.**

The final manifest gives 4.7.1 one place to discover the evaluation outputs. Compact 4.6.2 contracts are listed directly, while the completed Tree ablation and frozen 4.6.1 prediction surface remain referenced at their authoritative paths rather than copied again.

In [56]:
# ---------------------------------------------------------------------
# 4.6.2.8.3 — Publish the final handoff manifest
# ---------------------------------------------------------------------

handoff_purposes = {
    "job_reliability_summary":
        "Target × Horizon reliability summary for downstream context.",
    "failure_mechanism_summary":
        "Prevalence, severity, and error burden of diagnostic mechanisms.",
    "failure_case_studies":
        "Three frozen record-level failure examples.",
    "failure_case_context":
        "Comparable local timelines surrounding the three formal failures.",
    "model_interpretation_coverage":
        "Champion-family interpretation methods and coverage.",
    "tree_training_history_sensitivity":
        "Full-History versus Post-CP sensitivity for the eight Tree champions.",
    "reliability_guidance":
        "Diagnostic cautions that must not become holdout-tuned exclusions.",
    "record_diagnostics":
        "Narrow row-level surface for reliability analysis and Forecast Wins & Misses.",
}

handoff_records = []

for artifact_id, (frame, path) in final_handoff_frames.items():
    require_file(path, f"4.6.2 handoff artifact '{artifact_id}'")

    handoff_records.append({
        "artifact_id": artifact_id,
        "path": str(path),
        "rows": len(frame),
        "columns": len(frame.columns),
        "artifact_role": "4.6.2_final_contract",
        "purpose": handoff_purposes[artifact_id],
    })


# Keep independently owned large artifacts in place. The manifest points to
# them so 4.7.1 can consume the authoritative source without another copy.
handoff_records.extend([
    {
        "artifact_id": "tree_feature_group_ablation",
        "path": str(TREE_ABLATION_FINAL_PATH),
        "rows": len(final_tree_ablation_df),
        "columns": len(final_tree_ablation_df.columns),
        "artifact_role": "4.6.2_evaluation_result",
        "purpose":
            "Formal retraining ablation of the three Tree information sources.",
    },
    {
        "artifact_id": "champion_holdout_predictions",
        "path": str(CHAMPION_HOLDOUT_PREDICTIONS_PATH),
        "rows": np.nan,
        "columns": np.nan,
        "artifact_role": "4.6.1_upstream_reference",
        "purpose":
            "Frozen row-level champion forecasts used for downstream packaging.",
    },
])

evaluation_handoff_manifest_df = pd.DataFrame(
    handoff_records
)

if evaluation_handoff_manifest_df["artifact_id"].duplicated().any():
    raise RuntimeError(
        "The final 4.6.2 handoff manifest contains duplicate artifact IDs."
    )

for row in evaluation_handoff_manifest_df.itertuples(index=False):
    require_file(
        Path(row.path),
        f"handoff artifact '{row.artifact_id}'",
    )

if WRITE_FINAL_OUTPUTS:
    atomic_write_parquet(
        evaluation_handoff_manifest_df,
        EVALUATION_HANDOFF_MANIFEST_PATH,
    )


# The manifest doubles as the final file inventory, so the notebook does not
# need a second table that repeats the same paths and purposes.
manifest_display_df = evaluation_handoff_manifest_df[
    [
        "artifact_id",
        "artifact_role",
        "rows",
        "columns",
        "purpose",
    ]
].rename(
    columns={
        "artifact_id": "Artifact",
        "artifact_role": "Role",
        "rows": "Rows",
        "columns": "Columns",
        "purpose": "Purpose",
    }
)

display(
    manifest_display_df.style
    .format(
        {"Rows": "{:,.0f}", "Columns": "{:,.0f}"},
        na_rep="—",
    )
    .hide(axis="index")
    .set_properties(
        subset=["Artifact", "Role", "Purpose"],
        **{"text-align": "left", "vertical-align": "top"},
    )
)

print(
    "PASS — 4.6.2 evaluation handoff published.\n"
    f"  Manifest artifacts: {len(evaluation_handoff_manifest_df)}\n"
    f"  Manifest: {EVALUATION_HANDOFF_MANIFEST_PATH}",
    flush=True,
)

Artifact,Role,Rows,Columns,Purpose
job_reliability_summary,4.6.2_final_contract,15,29,Target × Horizon reliability summary for downstream context.
failure_mechanism_summary,4.6.2_final_contract,3,11,"Prevalence, severity, and error burden of diagnostic mechanisms."
failure_case_studies,4.6.2_final_contract,3,87,Three frozen record-level failure examples.
failure_case_context,4.6.2_final_contract,92,83,Comparable local timelines surrounding the three formal failures.
model_interpretation_coverage,4.6.2_final_contract,3,5,Champion-family interpretation methods and coverage.
tree_training_history_sensitivity,4.6.2_final_contract,8,16,Full-History versus Post-CP sensitivity for the eight Tree champions.
reliability_guidance,4.6.2_final_contract,5,4,Diagnostic cautions that must not become holdout-tuned exclusions.
record_diagnostics,4.6.2_final_contract,"1,421,436",26,Narrow row-level surface for reliability analysis and Forecast Wins & Misses.
tree_feature_group_ablation,4.6.2_evaluation_result,24,2,Formal retraining ablation of the three Tree information sources.
champion_holdout_predictions,4.6.1_upstream_reference,—,—,Frozen row-level champion forecasts used for downstream packaging.


PASS — 4.6.2 evaluation handoff published.
  Manifest artifacts: 10
  Manifest: pipeline_data/4.6.2.final_tables/forecast_evaluation_handoff_manifest.parquet


**Findings.** The handoff manifest is the single inventory for downstream forecast evaluation evidence. It exposes the compact 4.6.2 contracts, the formal Tree ablation, and the frozen champion-prediction source without creating redundant copies of completed upstream artifacts.

4.7.1 can therefore focus on **packaging and forecast-support eligibility**, while the record-level diagnostics provide a direct foundation for future Showcase views such as **Forecast Wins & Misses**.

### Section Summary

The evaluation handoff is complete and internally consistent. Downstream work receives the reliability summaries, formal failure examples, model-interpretation evidence, Tree sensitivity and ablation results, and a narrow record-level diagnostic surface without reopening model selection or turning final-holdout difficulty into an exclusion rule.

# Notebook Close

**4.6.2 is complete.**

The frozen champion system advances unchanged to **4.7.1 Package Forecast Outputs for Counterfactual and Showcase Use**. That notebook can now define forecast-support eligibility from information available at forecast origin and package the evaluated forecasts for Chapter 5 and reader-facing uses such as **Forecast Wins & Misses**.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=5c43663c-345d-4c21-9332-dd4f928af3ba' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>